In [1]:
import fitz
import json
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

class PDFColumnExtractor:
    """
    Extracts structured data from two-column PDF layout with vertical divider line.
    """
    
    def __init__(self, pdf_path: str):
        """
        Initialize the PDF extractor.
        
        Args:
            pdf_path: Path to the PDF file
        """
        self.pdf_path = pdf_path
        self.doc = fitz.open(pdf_path)
        
    def detect_vertical_divider(self, page: fitz.Page, tolerance: int = 5) -> Optional[float]:
        """
        Detect the vertical line that divides the page into two columns.
        
        Args:
            page: PyMuPDF page object
            tolerance: Pixel tolerance for line detection
            
        Returns:
            X-coordinate of the dividing line, or None if not found
        """
        # Get page dimensions
        page_width = page.rect.width
        page_height = page.rect.height
        
        # Look for vertical lines in the middle region of the page
        middle_region_start = page_width * 0.4
        middle_region_end = page_width * 0.6
        
        # Extract drawings/paths from the page
        try:
            drawings = page.get_drawings()
        except:
            # If get_drawings() fails, use fallback
            return page_width / 2
        
        vertical_lines = []
        
        for drawing in drawings:
            # Check if this is a vertical line
            for item in drawing["items"]:
                try:
                    if item[0] == "l":  # line
                        # item format: ("l", point1, point2)
                        if len(item) >= 3:
                            point1 = item[1]
                            point2 = item[2]
                            
                            # Points are fitz.Point objects with x and y attributes
                            x1, y1 = point1.x, point1.y
                            x2, y2 = point2.x, point2.y
                            
                            # Check if it's a vertical line (x coordinates are similar)
                            if abs(x1 - x2) <= tolerance:
                                # Check if it's in the middle region
                                if middle_region_start <= x1 <= middle_region_end:
                                    # Check if it spans most of the page height
                                    line_height = abs(y2 - y1)
                                    if line_height > page_height * 0.5:  # Line covers >50% of page height
                                        vertical_lines.append(x1)
                    
                    elif item[0] == "re":  # rectangle
                        # Check if rectangle is thin and vertical (could be a line)
                        if len(item) >= 2:
                            rect = item[1]
                            if hasattr(rect, 'x0'):
                                x0, y0, x1, y1 = rect.x0, rect.y0, rect.x1, rect.y1
                                width = abs(x1 - x0)
                                height = abs(y1 - y0)
                                
                                # Thin vertical rectangle in middle region
                                if width <= tolerance and height > page_height * 0.5:
                                    center_x = (x0 + x1) / 2
                                    if middle_region_start <= center_x <= middle_region_end:
                                        vertical_lines.append(center_x)
                except (AttributeError, IndexError, TypeError):
                    # Skip items that don't match expected format
                    continue
        
        if vertical_lines:
            # Return the average x-coordinate
            return sum(vertical_lines) / len(vertical_lines)
        
        # Alternative method: analyze text block distribution
        # If no line found, try to detect column split from text positions
        blocks = page.get_text("dict")["blocks"]
        x_coords = []
        
        for block in blocks:
            if "lines" in block:
                x0, y0, x1, y1 = block["bbox"]
                x_coords.append(x0)
                x_coords.append(x1)
        
        if x_coords:
            # Find a gap in the middle region where no text exists
            x_coords.sort()
            
            # Look for the largest gap in the middle region
            max_gap = 0
            split_point = page_width / 2
            
            for i in range(len(x_coords) - 1):
                if middle_region_start <= x_coords[i] <= middle_region_end:
                    gap = x_coords[i + 1] - x_coords[i]
                    if gap > max_gap:
                        max_gap = gap
                        split_point = (x_coords[i] + x_coords[i + 1]) / 2
            
            if max_gap > 20:  # Significant gap found
                return split_point
        
        # Fallback: return middle of the page
        return page_width / 2
    
    def extract_text_blocks_by_column(self, page: fitz.Page, divider_x: float) -> Tuple[List[Dict], List[Dict]]:
        """
        Extract text blocks separated by column.
        
        Args:
            page: PyMuPDF page object
            divider_x: X-coordinate of the column divider
            
        Returns:
            Tuple of (left_column_blocks, right_column_blocks)
        """
        # Extract text blocks with positional information
        blocks = page.get_text("dict")["blocks"]
        
        left_blocks = []
        right_blocks = []
        
        for block in blocks:
            if "lines" not in block:  # Skip image blocks
                continue
                
            # Get block bounding box
            x0, y0, x1, y1 = block["bbox"]
            block_center_x = (x0 + x1) / 2
            
            # Extract text from block
            text_content = ""
            for line in block["lines"]:
                for span in line["spans"]:
                    text_content += span["text"] + " "
            
            text_content = text_content.strip()
            
            if not text_content:
                continue
            
            block_data = {
                "text": text_content,
                "bbox": [x0, y0, x1, y1],
                "font_size": block["lines"][0]["spans"][0]["size"] if block["lines"] else 0,
                "font_name": block["lines"][0]["spans"][0]["font"] if block["lines"] else "",
            }
            
            # Assign to left or right column
            if block_center_x < divider_x:
                left_blocks.append(block_data)
            else:
                right_blocks.append(block_data)
        
        # Sort blocks by vertical position (top to bottom)
        left_blocks.sort(key=lambda b: b["bbox"][1])
        right_blocks.sort(key=lambda b: b["bbox"][1])
        
        return left_blocks, right_blocks
    
    def detect_tables_in_blocks(self, blocks: List[Dict]) -> List[Dict]:
        """
        Detect table-like structures in text blocks.
        
        Args:
            blocks: List of text blocks
            
        Returns:
            List of identified tables with structure
        """
        tables = []
        i = 0
        
        while i < len(blocks):
            block = blocks[i]
            text = block["text"]
            
            # Simple heuristic: look for patterns that indicate tables
            # 1. Multiple tab characters or large spaces
            # 2. Consistent alignment
            # 3. Numeric values in specific positions
            
            if "\t" in text or "  " in text:  # Has multiple spaces/tabs
                # Try to extract table structure
                lines = text.split("\n")
                
                # Check if it looks like a table header
                potential_table = {
                    "type": "table",
                    "bbox": block["bbox"],
                    "rows": []
                }
                
                for line in lines:
                    # Split by multiple spaces or tabs
                    parts = [p.strip() for p in line.split("  ") if p.strip()]
                    if not parts:
                        parts = [p.strip() for p in line.split("\t") if p.strip()]
                    
                    if parts:
                        potential_table["rows"].append(parts)
                
                if len(potential_table["rows"]) >= 2:  # At least header + 1 row
                    tables.append(potential_table)
            
            i += 1
        
        return tables
    
    def extract_structured_content(self, blocks: List[Dict]) -> Dict:
        """
        Extract structured content including headings, paragraphs, and tables.
        
        Args:
            blocks: List of text blocks from one column
            
        Returns:
            Structured content dictionary
        """
        structured = {
            "headings": [],
            "paragraphs": [],
            "tables": [],
            "lists": []
        }
        
        for block in blocks:
            text = block["text"]
            font_size = block.get("font_size", 0)
            
            # Identify headings (larger font, shorter text)
            if font_size > 12 and len(text) < 100:
                structured["headings"].append({
                    "text": text,
                    "level": "h1" if font_size > 16 else "h2",
                    "bbox": block["bbox"]
                })
            
            # Identify tables
            elif "\t" in text or ("  " in text and any(char.isdigit() for char in text)):
                table = self.parse_table_from_text(text, block["bbox"])
                if table:
                    structured["tables"].append(table)
            
            # Identify lists (lines starting with bullets or numbers)
            elif text.startswith(("•", "-", "*")) or (text[0].isdigit() and ". " in text[:5]):
                structured["lists"].append({
                    "text": text,
                    "bbox": block["bbox"]
                })
            
            # Regular paragraphs
            else:
                structured["paragraphs"].append({
                    "text": text,
                    "bbox": block["bbox"]
                })
        
        return structured
    
    def parse_table_from_text(self, text: str, bbox: List[float]) -> Optional[Dict]:
        """
        Parse table structure from text with tabs/spaces.
        
        Args:
            text: Text containing table data
            bbox: Bounding box of the text block
            
        Returns:
            Structured table dictionary or None
        """
        lines = text.strip().split("\n")
        if len(lines) < 2:
            return None
        
        table_data = {
            "type": "table",
            "bbox": bbox,
            "headers": [],
            "rows": []
        }
        
        # Process each line
        for idx, line in enumerate(lines):
            # Try splitting by multiple spaces first
            parts = [p.strip() for p in line.split("  ") if p.strip()]
            
            # If that doesn't work, try tabs
            if len(parts) <= 1:
                parts = [p.strip() for p in line.split("\t") if p.strip()]
            
            if not parts:
                continue
            
            # First line is likely headers
            if idx == 0 and not any(p.replace(".", "").replace("-", "").isdigit() for p in parts):
                table_data["headers"] = parts
            else:
                table_data["rows"].append(parts)
        
        # Validate table has content
        if not table_data["rows"]:
            return None
        
        return table_data
    
    def extract_page_data(self, page_number: int) -> Dict:
        """
        Extract structured data from a specific page.
        
        Args:
            page_number: Page number (0-indexed)
            
        Returns:
            Structured data dictionary containing left and right column content
        """
        if page_number >= len(self.doc):
            raise ValueError(f"Page {page_number} does not exist. PDF has {len(self.doc)} pages.")
        
        page = self.doc[page_number]
        
        # Detect the vertical divider line
        divider_x = self.detect_vertical_divider(page)
        
        # Extract text blocks by column
        left_blocks, right_blocks = self.extract_text_blocks_by_column(page, divider_x)
        
        # Extract structured content from each column
        left_content = self.extract_structured_content(left_blocks)
        right_content = self.extract_structured_content(right_blocks)
        
        # Compile final result
        result = {
            "page_number": page_number,
            "page_dimensions": {
                "width": page.rect.width,
                "height": page.rect.height
            },
            "divider_x_coordinate": divider_x,
            "left_column": left_content,
            "right_column": right_content,
            "raw_left_blocks": left_blocks,  # Keep raw data for debugging
            "raw_right_blocks": right_blocks
        }
        
        return result
    
    def close(self):
        """Close the PDF document."""
        self.doc.close()


# Example usage function
def extract_pdf_page(pdf_path: str, page_number: int) -> Dict:
    """
    Main function to extract structured data from a PDF page.
    
    Args:
        pdf_path: Path to PDF file
        page_number: Page number to extract (0-indexed)
        
    Returns:
        Structured data dictionary
    """
    extractor = PDFColumnExtractor(pdf_path)
    
    try:
        data = extractor.extract_page_data(page_number)
        return data
    finally:
        extractor.close()


# Example usage:
if __name__ == "__main__":
    # Replace with your PDF path
    pdf_path = r"C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\data\\process\\24-25-DC-CatalogFINAL-5.17.24_compressed.pdf"
    page_num = 43  # First page (0-indexed)
    
    # Extract data
    result = extract_pdf_page(pdf_path, page_num)
    
    # Print structured output
    print(json.dumps(result, indent=2, default=str))
    
    # Access specific data
    print("\n--- LEFT COLUMN HEADINGS ---")
    for heading in result["left_column"]["headings"]:
        print(f"- {heading['text']}")
    
    print("\n--- LEFT COLUMN TABLES ---")
    for table in result["left_column"]["tables"]:
        print(f"Headers: {table.get('headers', [])}")
        print(f"Rows: {len(table['rows'])}")
        for row in table["rows"][:3]:  # Print first 3 rows
            print(f"  {row}")
    
    print("\n--- RIGHT COLUMN TABLES ---")
    for table in result["right_column"]["tables"]:
        print(f"Headers: {table.get('headers', [])}")
        print(f"Rows: {len(table['rows'])}")
        for row in table["rows"][:3]:
            print(f"  {row}")

{
  "page_number": 43,
  "page_dimensions": {
    "width": 612.0,
    "height": 792.0
  },
  "divider_x_coordinate": 306.0,
  "left_column": {
    "headings": [],
    "paragraphs": [
      {
        "text": "Din\u00e9 College  Courses & Title",
        "bbox": [
          41.0,
          37.58199691772461,
          106.85388946533203,
          60.85598373413086
        ]
      },
      {
        "text": "Northland Pioneer Col- lege Early Childhood De- velopment (ECD) Courses  & Titles",
        "bbox": [
          115.86000061035156,
          37.58199691772461,
          229.376953125,
          82.45602416992188
        ]
      },
      {
        "text": "Din\u00e9 College  EDU Courses & Titles (required for  admittance to  BA Program)",
        "bbox": [
          235.1999969482422,
          37.58199691772461,
          301.7099914550781,
          104.05599975585938
        ]
      },
      {
        "text": "EDU 297 Practi- cum in Education (1)",
        "bbox": [
          235

### Below code is used to extract image from pdf file. code is working fine.This code needs to run one time.

In [ ]:
# import fitz  # PyMuPDF
# from PIL import Image
# import io
# import base64
# from typing import Dict, Optional
# import json

# class PDFImageExtractor:
#     """
#     Extract high-resolution images from PDF pages for VLM processing.
#     """
    
#     def __init__(self, pdf_path: str):
#         """
#         Initialize the PDF image extractor.
        
#         Args:
#             pdf_path: Path to the PDF file
#         """
#         self.pdf_path = pdf_path
#         self.doc = fitz.open(pdf_path)
    
#     def extract_page_as_image(
#         self, 
#         page_number: int, 
#         dpi: int = 300,
#         output_format: str = "PNG"
#     ) -> Dict:
#         """
#         Extract a PDF page as a high-resolution image.
        
#         Args:
#             page_number: Page number to extract (0-indexed)
#             dpi: Resolution in dots per inch (higher = better quality)
#                  Recommended values:
#                  - 150 DPI: Good for text recognition, smaller file size
#                  - 300 DPI: Excellent quality, recommended for detailed documents
#                  - 600 DPI: Very high quality, larger file size
#             output_format: Image format ("PNG", "JPEG", "TIFF")
            
#         Returns:
#             Dictionary containing image data and metadata
#         """
#         if page_number >= len(self.doc):
#             raise ValueError(f"Page {page_number} does not exist. PDF has {len(self.doc)} pages.")
        
#         # Get the page
#         page = self.doc[page_number]
        
#         # Calculate zoom factor from DPI
#         # PyMuPDF default is 72 DPI, so zoom = desired_dpi / 72
#         zoom = dpi / 72
#         mat = fitz.Matrix(zoom, zoom)
        
#         # Render page to pixmap (image)
#         pix = page.get_pixmap(matrix=mat, alpha=False)
        
#         # Convert pixmap to PIL Image
#         img_data = pix.tobytes(output_format.lower())
#         image = Image.open(io.BytesIO(img_data))
        
#         # Get page dimensions
#         page_rect = page.rect
        
#         # Prepare result
#         result = {
#             "page_number": page_number,
#             "dpi": dpi,
#             "format": output_format,
#             "dimensions": {
#                 "width": image.width,
#                 "height": image.height,
#                 "pdf_width": page_rect.width,
#                 "pdf_height": page_rect.height
#             },
#             "image": image,
#             "size_bytes": len(img_data)
#         }
        
#         return result
    
#     def save_page_as_image(
#         self, 
#         page_number: int, 
#         output_path: str,
#         dpi: int = 300,
#         output_format: str = "PNG"
#     ) -> str:
#         """
#         Extract and save a PDF page as an image file.
        
#         Args:
#             page_number: Page number to extract (0-indexed)
#             output_path: Path where to save the image
#             dpi: Resolution in dots per inch
#             output_format: Image format ("PNG", "JPEG", "TIFF")
            
#         Returns:
#             Path to saved image file
#         """
#         result = self.extract_page_as_image(page_number, dpi, output_format)
#         image = result["image"]
        
#         # Save image
#         if output_format.upper() == "JPEG":
#             image.save(output_path, format="JPEG", quality=95, optimize=True)
#         else:
#             image.save(output_path, format=output_format)
        
#         print(f"✓ Saved page {page_number} as {output_path}")
#         print(f"  Resolution: {image.width}x{image.height} pixels ({dpi} DPI)")
#         print(f"  File size: {result['size_bytes'] / 1024 / 1024:.2f} MB")
        
#         return output_path
    
#     def extract_page_as_base64(
#         self, 
#         page_number: int, 
#         dpi: int = 300,
#         output_format: str = "PNG"
#     ) -> Dict:
#         """
#         Extract a PDF page as base64-encoded image (for API calls).
        
#         Args:
#             page_number: Page number to extract (0-indexed)
#             dpi: Resolution in dots per inch
#             output_format: Image format ("PNG", "JPEG")
            
#         Returns:
#             Dictionary with base64 image data and metadata
#         """
#         result = self.extract_page_as_image(page_number, dpi, output_format)
#         image = result["image"]
        
#         # Convert to base64
#         buffer = io.BytesIO()
#         if output_format.upper() == "JPEG":
#             image.save(buffer, format="JPEG", quality=95)
#         else:
#             image.save(buffer, format="PNG")
        
#         img_bytes = buffer.getvalue()
#         base64_string = base64.b64encode(img_bytes).decode('utf-8')
        
#         # Determine media type for API calls
#         media_type = f"image/{output_format.lower()}"
        
#         return {
#             "page_number": page_number,
#             "base64_data": base64_string,
#             "media_type": media_type,
#             "dpi": dpi,
#             "dimensions": result["dimensions"],
#             "size_bytes": len(img_bytes)
#         }
    
#     def extract_multiple_pages(
#         self, 
#         page_numbers: list, 
#         output_dir: str,
#         dpi: int = 300,
#         output_format: str = "PNG"
#     ) -> list:
#         """
#         Extract multiple pages as images.
        
#         Args:
#             page_numbers: List of page numbers to extract
#             output_dir: Directory where to save images
#             dpi: Resolution in dots per inch
#             output_format: Image format
            
#         Returns:
#             List of saved file paths
#         """
#         import os
#         os.makedirs(output_dir, exist_ok=True)
        
#         saved_files = []
#         for page_num in page_numbers:
#             output_path = os.path.join(
#                 output_dir, 
#                 f"page_{page_num+1:04d}.{output_format.lower()}"
#             )
#             self.save_page_as_image(page_num, output_path, dpi, output_format)
#             saved_files.append(output_path)
        
#         return saved_files
    
#     def extract_all_pages(
#         self, 
#         output_dir: str,
#         dpi: int = 300,
#         output_format: str = "PNG"
#     ) -> list:
#         """
#         Extract all pages from PDF as images.
        
#         Args:
#             output_dir: Directory where to save images
#             dpi: Resolution in dots per inch
#             output_format: Image format
            
#         Returns:
#             List of saved file paths
#         """
#         page_count = len(self.doc)
#         print(f"Extracting all {page_count} pages from PDF...")
        
#         page_numbers = list(range(page_count))
#         return self.extract_multiple_pages(page_numbers, output_dir, dpi, output_format)
    
#     def get_pdf_info(self) -> Dict:
#         """
#         Get basic information about the PDF.
        
#         Returns:
#             Dictionary with PDF metadata
#         """
#         return {
#             "total_pages": len(self.doc),
#             "metadata": self.doc.metadata,
#             "is_encrypted": self.doc.is_encrypted,
#             "page_sizes": [
#                 {
#                     "page": i,
#                     "width": page.rect.width,
#                     "height": page.rect.height
#                 }
#                 for i, page in enumerate(self.doc)
#             ]
#         }
    
#     def close(self):
#         """Close the PDF document."""
#         self.doc.close()

# # Example usage and demonstrations

# if __name__ == "__main__":
#     # Replace with your PDF path
#     pdf_path = r"C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\data\\process\\24-25-DC-CatalogFINAL-5.17.24_compressed.pdf"
    
#     # Example 1: Extract single page and save as image
#     print("=" * 60)
#     print("Example 1: Extract and save page as image")
#     print("=" * 60)
    
#     extractor = PDFImageExtractor(pdf_path)
    
#     # Save page 0 as high-res PNG
#     # extractor.save_page_as_image(
#     #     page_number=0,
#     #     output_path="page_0.png",
#     #     dpi=120,  # High quality
#     #     output_format="PNG"
#     # )

#     extractor.extract_all_pages(
#         output_dir="C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\chtn_test_docs\\imgs",
#         dpi=120,  # High quality
#         output_format="PNG"
#     )
    
#     print("\n" + "=" * 60)
#     print("Example 2: Get PDF information")
#     print("=" * 60)
    
#     info = extractor.get_pdf_info()
#     print(f"Total pages: {info['total_pages']}")
#     print(f"First page size: {info['page_sizes'][0]}")
    
#     print("\n" + "=" * 60)
#     print("Example 3: Extract page as base64 (for API calls)")
#     print("=" * 60)

In [11]:
# First few page are getting extracted correctly.

In [ ]:
# import os
# import json
# import base64
# from pathlib import Path
# from typing import List, Dict, Optional
# from groq import Groq
# from dotenv import load_dotenv

# # Load environment variables
# load_dotenv()

# class PDFImageExtractor:
#     def __init__(self, api_key: Optional[str] = None):
#         """
#         Initialize the PDF Image Extractor
        
#         Args:
#             api_key: Groq API key (if not provided, loads from env)
#         """
#         self.client = Groq(api_key=api_key or os.getenv('GROQ_API_KEY'))
#         self.model = "meta-llama/llama-4-maverick-17b-128e-instruct"
    
#     def image_to_base64(self, image_path: str) -> str:
#         """
#         Convert image file to base64 data URL
        
#         Args:
#             image_path: Path to the image file
            
#         Returns:
#             Base64 encoded data URL
#         """
#         with open(image_path, 'rb') as image_file:
#             encoded = base64.b64encode(image_file.read()).decode('utf-8')
#             return f"data:image/png;base64,{encoded}"
    
#     def create_extraction_prompt(self, page_number: int) -> str:
#         """
#         Create a structured prompt for data extraction
        
#         Args:
#             page_number: Current page number
            
#         Returns:
#             Formatted prompt string
#         """
#         prompt = f"""Analyze this PDF page (Page {page_number}) and extract its content.

#     STEP 1: IDENTIFY PAGE TYPE
#     Look at the page and determine its type:
#     - "table_of_contents": Has items with page numbers (e.g., "Item Name  17")
#     - "organizational_chart": Has people's names with titles and credentials (no page numbers)
#     - "general": Other content

#     STEP 2: EXTRACT BASED ON TYPE

#     FOR TABLE OF CONTENTS (items have page numbers):
#     Extract ALL text items with their details. BE CONCISE - use minimal text for each item.

#     Return JSON (NO COMMENTS):
#     {{
#     "page_number": {page_number},
#     "page_type": "table_of_contents",
#     "all_items": [
#         {{"text": "Item name", "page": "16", "is_header": true, "horizontal_position": "left", "vertical_position": 1}},
#         {{"text": "Another item", "page": "17", "is_header": false, "horizontal_position": "left", "vertical_position": 2}}
#     ]
#     }}

#     FOR ORGANIZATIONAL CHART (people with credentials, no page numbers):
#     {{
#     "page_number": {page_number},
#     "page_type": "organizational_chart",
#     "sections": [
#         {{
#         "section_title": "College Administration",
#         "content_type": "hierarchical",
#         "content": [
#             {{
#             "name": "Dr. Charles Monty Roessel",
#             "title": "College President",
#             "credentials": [
#                 "Ed. D., Arizona State University, Educational Administration and Supervision",
#                 "M.A., Prescott College Journalism"
#             ]
#             }}
#         ]
#         }}
#     ]
#     }}

#     CRITICAL RULES:
#     - Extract ALL items - don't skip anything
#     - NO COMMENTS in JSON (no # comments)
#     - Keep text concise
#     - For TOC: Extract everything from BOTH left and right columns
#     - For org charts: Group people under section headers with their credentials

#     Return valid JSON only."""
#         return prompt

#     def group_toc_items_by_headers(self, all_items: list) -> list:
#         """
#         Group extracted items by headers based on reading order
#         In a two-column TOC: read left column top to bottom, then right column top to bottom
        
#         Args:
#             all_items: List of all extracted items with positioning
            
#         Returns:
#             List of sections with properly grouped content
#         """
#         if not all_items:
#             return []
        
#         # Separate left and right column items
#         left_items = [item for item in all_items if item.get('horizontal_position') == 'left']
#         right_items = [item for item in all_items if item.get('horizontal_position') == 'right']
        
#         # Sort each by vertical position
#         left_items.sort(key=lambda x: x.get('vertical_position', 999))
#         right_items.sort(key=lambda x: x.get('vertical_position', 999))
        
#         # Combine: left column first, then right column
#         reading_order = left_items + right_items
        
#         sections = []
#         current_section = None
        
#         for item in reading_order:
#             if item.get('is_header', False):
#                 # Save previous section
#                 if current_section:
#                     sections.append(current_section)
                
#                 # Start new section
#                 current_section = {
#                     'section_title': item.get('text', ''),
#                     'position': 'spanning',
#                     'content_type': 'toc_list',
#                     'content': []
#                 }
#             else:
#                 # Add item to current section
#                 if current_section:
#                     current_section['content'].append({
#                         'item': item.get('text', ''),
#                         'page': item.get('page', '')
#                     })
#                 else:
#                     # No header yet encountered, create default section
#                     current_section = {
#                         'section_title': 'Items',
#                         'position': 'spanning',
#                         'content_type': 'toc_list',
#                         'content': [{
#                             'item': item.get('text', ''),
#                             'page': item.get('page', '')
#                         }]
#                     }
        
#         # Add final section
#         if current_section:
#             sections.append(current_section)
        
#         return sections

#     def post_process_toc_sections(self, extracted_data: Dict) -> Dict:
#         """
#         Post-process extracted data to properly group items based on page type
        
#         Args:
#             extracted_data: Raw extracted data from LLM
            
#         Returns:
#             Corrected data with properly grouped sections
#         """
#         if not extracted_data:
#             return extracted_data
        
#         page_type = extracted_data.get('page_type', 'general')
        
#         # Handle TOC pages with all_items format
#         if page_type == 'table_of_contents' and 'all_items' in extracted_data:
#             all_items = extracted_data.get('all_items', [])
#             sections = self.group_toc_items_by_headers(all_items)
#             extracted_data['sections'] = sections
#             # Remove all_items from final output
#             del extracted_data['all_items']
#             return extracted_data
        
#         # Handle organizational chart pages - already in correct format
#         if page_type == 'organizational_chart':
#             # Already properly structured by LLM
#             return extracted_data
        
#         # Original post-processing logic for old format TOC pages
#         if not extracted_data.get('sections'):
#             return extracted_data
        
#         is_toc = (page_type == 'table_of_contents' or 
#                 extracted_data.get('metadata', {}).get('is_toc', False))
        
#         if not is_toc:
#             return extracted_data
        
#         sections = extracted_data['sections']
#         merged_sections = []
#         i = 0
        
#         while i < len(sections):
#             current_section = sections[i].copy()
            
#             if current_section.get('content_type') == 'toc_list':
#                 current_content = current_section.get('content', [])
                
#                 # Look ahead to merge continuation sections
#                 j = i + 1
#                 while j < len(sections):
#                     next_section = sections[j]
                    
#                     if (next_section.get('position') == 'right' or
#                         next_section.get('content_type') == 'toc_list'):
                        
#                         next_content = next_section.get('content', [])
                        
#                         if (len(next_content) <= 5 and 
#                             next_section.get('position') == 'right'):
#                             if isinstance(current_content, list) and isinstance(next_content, list):
#                                 current_content.extend(next_content)
#                             j += 1
#                         else:
#                             break
#                     else:
#                         break
                
#                 current_section['content'] = current_content
#                 current_section['position'] = 'spanning'
#                 merged_sections.append(current_section)
#                 i = j
#             else:
#                 merged_sections.append(current_section)
#                 i += 1
        
#         extracted_data['sections'] = merged_sections
#         return extracted_data

#     def extract_from_image(self, image_path: str, page_number: int) -> Dict:
#         """
#         Extract structured data from a single image
        
#         Args:
#             image_path: Path to the PNG image
#             page_number: Page number for reference
            
#         Returns:
#             Dictionary containing extracted data
#         """
#         try:
#             # Convert image to base64
#             image_data_url = self.image_to_base64(image_path)
            
#             # Create prompt
#             prompt = self.create_extraction_prompt(page_number)
            
#             # Call Groq API
#             completion = self.client.chat.completions.create(
#                 model=self.model,
#                 messages=[
#                     {
#                         "role": "user",
#                         "content": [
#                             {
#                                 "type": "text",
#                                 "text": prompt
#                             },
#                             {
#                                 "type": "image_url",
#                                 "image_url": {
#                                     "url": image_data_url
#                                 }
#                             }
#                         ]
#                     }
#                 ],
#                 temperature=0.3,
#                 max_completion_tokens=8192,  # Increased for large TOC pages
#                 top_p=0.95,
#                 stream=False,
#                 stop=None
#             )
            
#             # Extract response
#             response_text = completion.choices[0].message.content
            
#             # Try to parse JSON response
#             try:
#                 # Look for JSON in code blocks
#                 if "```json" in response_text:
#                     json_start = response_text.find("```json") + 7
#                     json_end = response_text.find("```", json_start)
#                     response_text = response_text[json_start:json_end].strip()
#                 elif "```" in response_text:
#                     json_start = response_text.find("```") + 3
#                     json_end = response_text.find("```", json_start)
#                     response_text = response_text[json_start:json_end].strip()
                
#                 # Remove Python-style comments (# ...) that might break JSON
#                 lines = response_text.split('\n')
#                 cleaned_lines = []
#                 for line in lines:
#                     # Remove inline comments
#                     if '#' in line:
#                         # Only remove if # is not inside a string
#                         in_string = False
#                         cleaned_line = []
#                         for i, char in enumerate(line):
#                             if char == '"' and (i == 0 or line[i-1] != '\\'):
#                                 in_string = not in_string
#                             if char == '#' and not in_string:
#                                 break
#                             cleaned_line.append(char)
#                         line = ''.join(cleaned_line).rstrip()
#                     if line.strip():  # Keep non-empty lines
#                         cleaned_lines.append(line)
#                 response_text = '\n'.join(cleaned_lines)
                
#                 # Handle incomplete JSON - try to fix it
#                 if not response_text.endswith('}'):
#                     # Check if we're in the middle of an array
#                     if '"all_items": [' in response_text and not response_text.rstrip().endswith(']'):
#                         # Find the last complete item in all_items
#                         last_complete = response_text.rfind('},')
#                         if last_complete > 0:
#                             # Truncate to last complete item and close properly
#                             response_text = response_text[:last_complete + 1] + '\n    ]\n}'
#                     elif response_text.rstrip().endswith(','):
#                         # Remove trailing comma and close
#                         response_text = response_text.rstrip().rstrip(',') + '\n}'
#                     else:
#                         # Try to close at the last complete object
#                         last_complete = response_text.rfind('},')
#                         if last_complete > 0:
#                             response_text = response_text[:last_complete + 1] + '\n    ]\n}'
#                         else:
#                             # Last resort: just try to close what we have
#                             response_text = response_text.rstrip() + '\n}'
                
#                 extracted_data = json.loads(response_text)
                
#                 # Post-process sections to fix column splitting issues
#                 extracted_data = self.post_process_toc_sections(extracted_data)
#             except json.JSONDecodeError as e:
#                 # Fallback: return raw text if JSON parsing fails
#                 print(f"⚠️  JSON parsing failed for page {page_number}: {str(e)}")
#                 print(f"Response preview (first 500 chars): {response_text[:500]}")
#                 extracted_data = {
#                     "page_number": page_number,
#                     "page_type": "general",
#                     "raw_text": response_text,
#                     "extraction_status": "json_parse_failed",
#                     "parse_error": str(e)
#                 }
            
#             return extracted_data
            
#         except Exception as e:
#             return {
#                 "page_number": page_number,
#                 "error": str(e),
#                 "extraction_status": "failed"
#             }
        
#     def extract_page_range(
#         self, 
#         image_dir: str, 
#         start_page: int = 1, 
#         end_page: Optional[int] = None,
#         output_file: Optional[str] = None
#     ) -> List[Dict]:
#         """
#         Extract data from a range of pages
        
#         Args:
#             image_dir: Directory containing page images
#             start_page: Starting page number (1-indexed)
#             end_page: Ending page number (inclusive). If None, processes all pages
#             output_file: Optional JSON file to save results
            
#         Returns:
#             List of extracted data dictionaries
#         """
#         image_dir = Path(image_dir)
#         all_extractions = []
        
#         # Find all page images
#         page_files = sorted(image_dir.glob("page_*.png"))
        
#         if not page_files:
#             print(f"No page images found in {image_dir}")
#             return []
        
#         # Determine page range
#         total_pages = len(page_files)
#         end_page = end_page or total_pages
        
#         # Validate range
#         if start_page < 1 or start_page > total_pages:
#             print(f"Invalid start_page: {start_page}. Must be between 1 and {total_pages}")
#             return []
        
#         if end_page > total_pages:
#             print(f"Warning: end_page {end_page} exceeds total pages {total_pages}. Using {total_pages}")
#             end_page = total_pages
        
#         print(f"Processing pages {start_page} to {end_page} of {total_pages} total pages...")
        
#         # Process each page in range
#         for i in range(start_page - 1, end_page):
#             page_file = page_files[i]
#             page_num = i + 1
            
#             print(f"\nProcessing {page_file.name} (Page {page_num}/{end_page})...")
            
#             extracted_data = self.extract_from_image(str(page_file), page_num)
#             all_extractions.append(extracted_data)
            
#             print(f"✓ Completed page {page_num}")
        
#         # Save to file if specified
#         if output_file:
#             with open(output_file, 'w', encoding='utf-8') as f:
#                 json.dump(all_extractions, f, indent=2, ensure_ascii=False)
#             print(f"\n✓ Results saved to {output_file}")
        
#         return all_extractions
    
#     def prepare_for_embeddings(self, extracted_data: List[Dict]) -> List[Dict]:
#         """
#         Prepare extracted data for embedding generation
        
#         Args:
#             extracted_data: List of extracted page data
            
#         Returns:
#             List of text chunks with metadata for embeddings
#         """
#         embedding_chunks = []
        
#         for page_data in extracted_data:
#             page_num = page_data.get('page_number', 0)
            
#             # Skip failed extractions
#             if page_data.get('extraction_status') == 'failed':
#                 continue
            
#             # Handle raw text fallback
#             if 'raw_text' in page_data:
#                 embedding_chunks.append({
#                     'text': page_data['raw_text'],
#                     'metadata': {
#                         'page': page_num,
#                         'type': 'raw'
#                     }
#                 })
#                 continue
            
#             # Process structured sections
#             page_title = page_data.get('title', '')
#             sections = page_data.get('sections', [])
            
#             for section in sections:
#                 section_title = section.get('section_title', '')
#                 content = section.get('content', '')
#                 position = section.get('position', '')
#                 content_type = section.get('content_type', 'text')
                
#                 # Handle different content formats
#                 if isinstance(content, list):
#                     # Check if it's a TOC-style list with item/page structure
#                     formatted_items = []
#                     for item in content:
#                         if isinstance(item, dict) and 'item' in item:
#                             page_num = item.get('page', '')
#                             formatted_items.append(f"{item['item']} - Page {page_num}" if page_num else item['item'])
#                         else:
#                             formatted_items.append(str(item))
#                     content = "\n".join(formatted_items)
#                 elif isinstance(content, dict):
#                     content = json.dumps(content, indent=2)
#                 elif not isinstance(content, str):
#                     content = str(content)
                
#                 # Create contextual text
#                 text_parts = []
#                 if page_title:
#                     text_parts.append(f"Page: {page_title}")
#                 if section_title:
#                     text_parts.append(f"Section: {section_title}")
#                 text_parts.append(content)
                
#                 full_text = "\n".join(text_parts)
                
#                 embedding_chunks.append({
#                     'text': full_text,
#                     'metadata': {
#                         'page': page_num,
#                         'section': section_title,
#                         'position': position,
#                         'content_type': content_type
#                     }
#                 })
        
#         return embedding_chunks


# # Example usage
# if __name__ == "__main__":
#     # Initialize extractor
#     extractor = PDFImageExtractor()
    
#     # Example 1: Extract specific page range
#     results = extractor.extract_page_range(
#         image_dir="C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\chtn_test_docs\\imgs",
#         start_page=10,
#         end_page=12,
#         output_file="C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\chtn_test_docs\\extracted_data.json"
#     )
    
#     # Example 2: Prepare for embeddings
#     embedding_data = extractor.prepare_for_embeddings(results)
    
#     # Save embedding-ready data
#     with open('embedding_chunks.json', 'w', encoding='utf-8') as f:
#         json.dump(embedding_data, f, indent=2, ensure_ascii=False)
    
#     print(f"\n✓ Created {len(embedding_data)} embedding chunks")
    
#     # Display summary
#     print("\n=== Extraction Summary ===")
#     for result in results:
#         page_num = result.get('page_number', 'Unknown')
#         page_type = result.get('page_type', 'Unknown')
#         status = result.get('extraction_status', 'success')
#         print(f"Page {page_num}: {page_type} - {status}")

Processing pages 10 to 12 of 101 total pages...

Processing page_0010.png (Page 10/12)...
✓ Completed page 10

Processing page_0011.png (Page 11/12)...
✓ Completed page 11

Processing page_0012.png (Page 12/12)...
✓ Completed page 12

✓ Results saved to C:\chtn\gen_ai\hitesh\zenza_chatbot\project_xa001\chtn_test_docs\extracted_data.json

✓ Created 19 embedding chunks

=== Extraction Summary ===
Page 10: general - success
Page 11: general - success
Page 12: general - success


#### Part 2 : Below code is working fine for page no 8 of PDF. but when page 10 comes then it is not working as per the expectation

In [ ]:
# import os
# import json
# import base64
# from pathlib import Path
# from typing import List, Dict, Optional
# from groq import Groq
# from dotenv import load_dotenv

# # Load environment variables
# load_dotenv()

# class ImprovedPDFImageExtractor:
#     def __init__(self, api_key: Optional[str] = None):
#         """
#         Initialize the PDF Image Extractor
        
#         Args:
#             api_key: Groq API key (if not provided, loads from env)
#         """
#         self.client = Groq(api_key=api_key or os.getenv('GROQ_API_KEY'))
#         self.model = "meta-llama/llama-4-maverick-17b-128e-instruct"
#         self.previous_page_context = None  # Store last section of previous page
    
#     def image_to_base64(self, image_path: str) -> str:
#         """Convert image file to base64 data URL"""
#         with open(image_path, 'rb') as image_file:
#             encoded = base64.b64encode(image_file.read()).decode('utf-8')
#             return f"data:image/png;base64,{encoded}"
    
#     def create_extraction_prompt(self, page_number: int, previous_context: Optional[str] = None) -> str:
#         """
#         Create a structured prompt for data extraction with context awareness
        
#         Args:
#             page_number: Current page number
#             previous_context: Last section title from previous page (if any)
            
#         Returns:
#             Formatted prompt string
#         """
#         context_info = ""
#         if previous_context:
#             context_info = f"""
# PREVIOUS PAGE CONTEXT:
# The previous page ended with content under the section: "{previous_context}"
# If this page starts mid-paragraph or continues that section, mark it with "continues_from_previous": true
# """

#         prompt = f"""Analyze this PDF page (Page {page_number}) and extract ALL content following these rules:

# {context_info}

# CRITICAL READING ORDER FOR TWO-COLUMN PAGES:
# 1. Read LEFT column from TOP to BOTTOM completely
# 2. Then read RIGHT column from TOP to BOTTOM completely
# 3. DO NOT jump between columns - complete one column before moving to next

# EXTRACTION RULES:

# 1. IDENTIFY PAGE TYPE:
#    - "table_of_contents": Items with page numbers
#    - "organizational_chart": People with titles/credentials
#    - "general": Regular text content (paragraphs, lists, etc.)

# 2. FOR GENERAL PAGES (most common):
#    Extract text in READING ORDER (left column completely, then right column).
   
#    Return JSON:
#    {{
#      "page_number": {page_number},
#      "page_type": "general",
#      "sections": [
#        {{
#          "section_title": "Section Header Name",
#          "continues_from_previous": false,
#          "continues_on_next": true,
#          "content": "Complete paragraph text from both columns in reading order...",
#          "column_flow": "left_to_right"
#        }}
#      ]
#    }}

# 3. FOR TABLE OF CONTENTS:
#    {{
#      "page_number": {page_number},
#      "page_type": "table_of_contents",
#      "sections": [
#        {{
#          "section_title": "Header Name",
#          "content_type": "toc_list",
#          "items": [
#            {{"text": "Item name", "page": "16"}},
#            {{"text": "Another item", "page": "17"}}
#          ]
#        }}
#      ]
#    }}

# 4. FOR ORGANIZATIONAL CHART:
#    {{
#      "page_number": {page_number},
#      "page_type": "organizational_chart",
#      "sections": [
#        {{
#          "section_title": "Department Name",
#          "content_type": "hierarchical",
#          "people": [
#            {{
#              "name": "Full Name",
#              "title": "Position Title",
#              "credentials": ["Credential 1", "Credential 2"]
#            }}
#          ]
#        }}
#      ]
#    }}

# CRITICAL RULES:
# - Extract ALL content - nothing should be skipped
# - For two-column layout: read LEFT column completely TOP to BOTTOM, then RIGHT column TOP to BOTTOM
# - Keep paragraphs intact - don't break mid-sentence
# - If a section starts on this page but clearly continues beyond, set "continues_on_next": true
# - If this page starts mid-paragraph from previous, set "continues_from_previous": true
# - Use consistent JSON structure for same content types
# - NO COMMENTS in JSON
# - Return ONLY valid JSON

# Return valid JSON only."""
#         return prompt

#     def detect_continuation_markers(self, content: str) -> Dict[str, bool]:
#         """
#         Detect if content continues from previous or to next page
        
#         Args:
#             content: Text content to analyze
            
#         Returns:
#             Dictionary with continuation flags
#         """
#         markers = {
#             'continues_from_previous': False,
#             'continues_on_next': False
#         }
        
#         # Check if content starts mid-sentence (no capital letter start after common headers)
#         if content and not content[0].isupper() and not content.startswith(('•', '-', '1.', '2.')):
#             markers['continues_from_previous'] = True
        
#         # Check if content ends mid-sentence
#         if content and not content.rstrip().endswith(('.', '!', '?', ':', ';')):
#             markers['continues_on_next'] = True
        
#         return markers

#     def merge_continued_sections(self, current_data: Dict, previous_data: Optional[Dict]) -> Dict:
#         """
#         Merge sections that continue across pages
        
#         Args:
#             current_data: Current page data
#             previous_data: Previous page data
            
#         Returns:
#             Updated current_data with merged content
#         """
#         if not previous_data or not previous_data.get('sections'):
#             return current_data
        
#         current_sections = current_data.get('sections', [])
#         previous_sections = previous_data.get('sections', [])
        
#         if not current_sections or not previous_sections:
#             return current_data
        
#         # Check if first section of current page continues from last section of previous page
#         first_section = current_sections[0]
#         last_prev_section = previous_sections[-1]
        
#         # Add continuation marker if detected
#         if first_section.get('continues_from_previous') or \
#            last_prev_section.get('continues_on_next'):
#             first_section['_continuation_note'] = (
#                 f"This section continues from page {previous_data.get('page_number')}. "
#                 f"Previous section: '{last_prev_section.get('section_title', 'Unknown')}'"
#             )
        
#         return current_data

#     def standardize_json_structure(self, extracted_data: Dict) -> Dict:
#         """
#         Ensure consistent JSON structure across different content types
        
#         Args:
#             extracted_data: Raw extracted data from LLM
            
#         Returns:
#             Standardized data structure
#         """
#         if not extracted_data or not extracted_data.get('sections'):
#             return extracted_data
        
#         page_type = extracted_data.get('page_type', 'general')
#         sections = extracted_data.get('sections', [])
#         standardized_sections = []
        
#         for section in sections:
#             standardized = {
#                 'section_title': section.get('section_title', ''),
#                 'content_type': section.get('content_type', 'text'),
#                 'continues_from_previous': section.get('continues_from_previous', False),
#                 'continues_on_next': section.get('continues_on_next', False)
#             }
            
#             # Standardize content based on type
#             if page_type == 'table_of_contents':
#                 # Ensure TOC items are in consistent format
#                 items = section.get('items', section.get('content', []))
#                 if isinstance(items, list):
#                     standardized['items'] = [
#                         {'text': item.get('text', str(item)), 
#                          'page': item.get('page', '')}
#                         if isinstance(item, dict) else {'text': str(item), 'page': ''}
#                         for item in items
#                     ]
                    
#             elif page_type == 'organizational_chart':
#                 # Keep hierarchical structure
#                 standardized['people'] = section.get('people', section.get('content', []))
                
#             else:  # general
#                 # Ensure content is a single string
#                 content = section.get('content', '')
#                 if isinstance(content, list):
#                     content = '\n'.join([str(item) for item in content])
#                 standardized['content'] = content
            
#             # Add continuation note if exists
#             if '_continuation_note' in section:
#                 standardized['continuation_note'] = section['_continuation_note']
            
#             standardized_sections.append(standardized)
        
#         extracted_data['sections'] = standardized_sections
#         return extracted_data

#     def extract_from_image(self, image_path: str, page_number: int) -> Dict:
#         """
#         Extract structured data from a single image
        
#         Args:
#             image_path: Path to the PNG image
#             page_number: Page number for reference
            
#         Returns:
#             Dictionary containing extracted data
#         """
#         try:
#             # Convert image to base64
#             image_data_url = self.image_to_base64(image_path)
            
#             # Create prompt with context from previous page
#             prompt = self.create_extraction_prompt(page_number, self.previous_page_context)
            
#             # Call Groq API
#             completion = self.client.chat.completions.create(
#                 model=self.model,
#                 messages=[
#                     {
#                         "role": "user",
#                         "content": [
#                             {
#                                 "type": "text",
#                                 "text": prompt
#                             },
#                             {
#                                 "type": "image_url",
#                                 "image_url": {
#                                     "url": image_data_url
#                                 }
#                             }
#                         ]
#                     }
#                 ],
#                 temperature=0.2,  # Lower temperature for more consistent output
#                 max_completion_tokens=8192,
#                 top_p=0.95,
#                 stream=False,
#                 stop=None
#             )
            
#             # Extract response
#             response_text = completion.choices[0].message.content
            
#             # Parse JSON response
#             try:
#                 # Clean up response text
#                 if "```json" in response_text:
#                     json_start = response_text.find("```json") + 7
#                     json_end = response_text.find("```", json_start)
#                     response_text = response_text[json_start:json_end].strip()
#                 elif "```" in response_text:
#                     json_start = response_text.find("```") + 3
#                     json_end = response_text.find("```", json_start)
#                     response_text = response_text[json_start:json_end].strip()
                
#                 # Remove comments
#                 lines = response_text.split('\n')
#                 cleaned_lines = []
#                 for line in lines:
#                     if '#' in line:
#                         in_string = False
#                         cleaned_line = []
#                         for i, char in enumerate(line):
#                             if char == '"' and (i == 0 or line[i-1] != '\\'):
#                                 in_string = not in_string
#                             if char == '#' and not in_string:
#                                 break
#                             cleaned_line.append(char)
#                         line = ''.join(cleaned_line).rstrip()
#                     if line.strip():
#                         cleaned_lines.append(line)
#                 response_text = '\n'.join(cleaned_lines)
                
#                 # Fix incomplete JSON
#                 if not response_text.endswith('}'):
#                     if response_text.rstrip().endswith(','):
#                         response_text = response_text.rstrip().rstrip(',') + '\n    ]\n  }\n}'
#                     else:
#                         response_text = response_text.rstrip() + '\n    ]\n  }\n}'
                
#                 extracted_data = json.loads(response_text)
                
#                 # Standardize structure
#                 extracted_data = self.standardize_json_structure(extracted_data)
                
#                 # Update context for next page
#                 sections = extracted_data.get('sections', [])
#                 if sections:
#                     last_section = sections[-1]
#                     self.previous_page_context = last_section.get('section_title', '')
                
#             except json.JSONDecodeError as e:
#                 print(f"⚠️  JSON parsing failed for page {page_number}: {str(e)}")
#                 print(f"Response preview: {response_text[:500]}")
#                 extracted_data = {
#                     "page_number": page_number,
#                     "page_type": "general",
#                     "raw_text": response_text,
#                     "extraction_status": "json_parse_failed",
#                     "parse_error": str(e)
#                 }
            
#             return extracted_data
            
#         except Exception as e:
#             return {
#                 "page_number": page_number,
#                 "error": str(e),
#                 "extraction_status": "failed"
#             }
        
#     def extract_page_range(
#         self, 
#         image_dir: str, 
#         start_page: int = 1, 
#         end_page: Optional[int] = None,
#         output_file: Optional[str] = None
#     ) -> List[Dict]:
#         """
#         Extract data from a range of pages with cross-page context
        
#         Args:
#             image_dir: Directory containing page images
#             start_page: Starting page number (1-indexed)
#             end_page: Ending page number (inclusive)
#             output_file: Optional JSON file to save results
            
#         Returns:
#             List of extracted data dictionaries
#         """
#         image_dir = Path(image_dir)
#         all_extractions = []
        
#         # Find all page images
#         page_files = sorted(image_dir.glob("page_*.png"))
        
#         if not page_files:
#             print(f"No page images found in {image_dir}")
#             return []
        
#         # Determine page range
#         total_pages = len(page_files)
#         end_page = end_page or total_pages
        
#         # Validate range
#         if start_page < 1 or start_page > total_pages:
#             print(f"Invalid start_page: {start_page}")
#             return []
        
#         if end_page > total_pages:
#             print(f"Warning: Using {total_pages} instead of {end_page}")
#             end_page = total_pages
        
#         print(f"Processing pages {start_page} to {end_page}...")
        
#         # Reset context
#         self.previous_page_context = None
#         previous_data = None
        
#         # Process each page
#         for i in range(start_page - 1, end_page):
#             page_file = page_files[i]
#             page_num = i + 1
            
#             print(f"\nProcessing page {page_num}/{end_page}...")
            
#             extracted_data = self.extract_from_image(str(page_file), page_num)
            
#             # Merge with previous page if continuation detected
#             if previous_data:
#                 extracted_data = self.merge_continued_sections(extracted_data, previous_data)
            
#             all_extractions.append(extracted_data)
#             previous_data = extracted_data
            
#             print(f"✓ Completed page {page_num}")
        
#         # Save to file
#         if output_file:
#             with open(output_file, 'w', encoding='utf-8') as f:
#                 json.dump(all_extractions, f, indent=2, ensure_ascii=False)
#             print(f"\n✓ Results saved to {output_file}")
        
#         return all_extractions


# # Example usage
# if __name__ == "__main__":
#     extractor = ImprovedPDFImageExtractor()
    
#     # Extract page range
#     results = extractor.extract_page_range(
#         image_dir="C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\chtn_test_docs\\imgs",
#         start_page=8,
#         end_page=10,
#         output_file="extracted_data_improved.json"
#     )
    
#     print("\n=== Extraction Summary ===")
#     for result in results:
#         page_num = result.get('page_number', 'Unknown')
#         page_type = result.get('page_type', 'Unknown')
#         sections = result.get('sections', [])
        
#         print(f"\nPage {page_num} ({page_type}):")
#         for section in sections:
#             title = section.get('section_title', 'No title')
#             continues_from = section.get('continues_from_previous', False)
#             continues_to = section.get('continues_on_next', False)
            
#             markers = []
#             if continues_from:
#                 markers.append("← continues from prev")
#             if continues_to:
#                 markers.append("continues to next →")
            
#             marker_str = f" [{', '.join(markers)}]" if markers else ""
#             print(f"  - {title}{marker_str}")
            
#             if 'continuation_note' in section:
#                 print(f"    Note: {section['continuation_note']}")

Processing pages 8 to 10...

Processing page 8/10...
✓ Completed page 8

Processing page 9/10...
✓ Completed page 9

Processing page 10/10...
✓ Completed page 10

✓ Results saved to extracted_data_improved.json

=== Extraction Summary ===

Page 8 (table_of_contents):
  - Associate of Applied Science Degree Programs
  - Associate of Arts Degree Programs
  - Associate of Science Degree Programs
  - Endorsement Programs
  - Minor Programs
  - Bachelor of Arts Degree Programs
  - Bachelor Fine Arts Degrees
  - Bachelor of Science Degree Programs
  - Master of Science Degree Programs
  - Course Title and Prefix Code
  - Faculty

Page 9 (general):
  - President's Message

Page 10 (general):
  - About Diné College
  - Diné College Campuses/Centers


In [2]:
#### Part 3 : 

In [ ]:
# import os
# import json
# import base64
# import re
# from pathlib import Path
# from typing import List, Dict, Optional
# from groq import Groq
# from dotenv import load_dotenv

# # Load environment variables
# load_dotenv()

# class EnhancedPDFImageExtractor:
#     def __init__(self, api_key: Optional[str] = None):
#         """
#         Initialize the PDF Image Extractor with enhanced structure detection
        
#         Args:
#             api_key: Groq API key (if not provided, loads from env)
#         """
#         self.client = Groq(api_key=api_key or os.getenv('GROQ_API_KEY'))
#         self.model = "meta-llama/llama-4-maverick-17b-128e-instruct"
#         self.previous_page_context = None
#         self.page_history = []  # Store previous pages for better context
    
#     def image_to_base64(self, image_path: str) -> str:
#         """Convert image file to base64 data URL"""
#         with open(image_path, 'rb') as image_file:
#             encoded = base64.b64encode(image_file.read()).decode('utf-8')
#             return f"data:image/png;base64,{encoded}"
    
#     def create_extraction_prompt(self, page_number: int, previous_context: Optional[str] = None) -> str:
#         """
#         Create an enhanced prompt for automatic structure detection and extraction
        
#         Args:
#             page_number: Current page number
#             previous_context: Context from previous page
            
#         Returns:
#             Formatted prompt string
#         """
#         context_info = ""
#         if previous_context:
#             context_info = f"""
# PREVIOUS PAGE CONTEXT:
# The previous page ended with: "{previous_context}"
# If this page continues from the previous content, mark it appropriately.
# """

#         prompt = f"""Analyze this PDF page (Page {page_number}) and extract ALL content with proper structure detection.

# {context_info}

# STEP 1: AUTOMATIC LAYOUT DETECTION
# First, identify the page layout:
# - "single_column": Content flows top to bottom in one column
# - "two_column": Content in LEFT column (top to bottom), then RIGHT column (top to bottom)
# - "table_of_contents": List items with page numbers
# - "mixed": Combination of layouts (describe the flow)

# STEP 2: CONTENT STRUCTURE DETECTION
# Identify all structural elements:
# - **Main Headers**: Large, bold text that starts a new major section
# - **Sub-headers**: Medium text under main headers
# - **Sub-sub-headers**: Smaller headers under sub-headers
# - **Paragraphs**: Body text content
# - **Lists**: Bulleted or numbered items
# - **Special formatting**: Italic text, bold emphasis, quotes

# STEP 3: READING ORDER (CRITICAL FOR TWO-COLUMN LAYOUTS)
# For two-column pages:
# 1. Read ENTIRE LEFT column from TOP to BOTTOM
# 2. Then read ENTIRE RIGHT column from TOP to BOTTOM
# 3. NEVER jump between columns mid-section

# STEP 4: OUTPUT FORMAT

# CRITICAL: Return VALID JSON with proper syntax:
# - ALWAYS use commas after each key-value pair (except the last one in an object)
# - ALWAYS use colons after keys
# - Use double quotes for strings
# - Example: {{"key1": "value1", "key2": "value2"}}

# For GENERAL CONTENT (paragraphs, regular text):
# {{
#   "page_number": {page_number},
#   "layout_type": "single_column",
#   "reading_order": "top_to_bottom",
#   "sections": [
#     {{
#       "level": 1,
#       "header": "Main Section Header",
#       "header_type": "main_header",
#       "continues_from_previous": false,
#       "continues_on_next": false,
#       "subsections": [
#         {{
#           "level": 2,
#           "header": "Sub-header (if any)",
#           "header_type": "sub_header",
#           "content": {{
#             "type": "paragraph",
#             "text": "Complete paragraph text...",
#             "formatting": []
#           }}
#         }}
#       ]
#     }}
#   ]
# }}

# For TABLE OF CONTENTS:
# {{
#   "page_number": {page_number},
#   "layout_type": "table_of_contents",
#   "sections": [
#     {{
#       "level": 1,
#       "header": "Main TOC Section",
#       "header_type": "toc_header",
#       "items": [
#         {{"text": "Item name", "page": "16"}},
#         {{"text": "Sub-item", "page": "17"}}
#       ]
#     }}
#   ]
# }}

# For TWO-COLUMN CONTENT (like "About Diné College"):
# {{
#   "page_number": {page_number},
#   "layout_type": "two_column",
#   "reading_order": "left_to_right",
#   "sections": [
#     {{
#       "level": 1,
#       "header": "College Mission",
#       "header_type": "main_header",
#       "column": "left",
#       "content": {{
#         "type": "multi_paragraph",
#         "paragraphs": [
#           {{"text": "First paragraph in native language...", "language": "navajo"}},
#           {{"text": "English translation...", "language": "english"}}
#         ]
#       }}
#     }},
#     {{
#       "level": 1,
#       "header": "College Vision",
#       "header_type": "main_header",
#       "column": "left",
#       "content": {{
#         "type": "multi_paragraph",
#         "paragraphs": [
#           {{"text": "Vision in native language..."}},
#           {{"text": "Vision in English..."}}
#         ]
#       }}
#     }}
#   ]
# }}

# CRITICAL JSON SYNTAX RULES:
# 1. ALWAYS add comma after key-value pairs (except last in object)
# 2. ALWAYS use colon after keys
# 3. Correct: {{"key": "value", "key2": "value2"}}
# 4. Wrong: {{"key" "value" "key2" "value2"}}

# EXTRACTION RULES:
# - Extract ALL content - nothing should be skipped
# - For two-column layout: read LEFT column completely TOP to BOTTOM, then RIGHT column TOP to BOTTOM
# - Keep paragraphs intact - don't break mid-sentence
# - If a section continues beyond this page, set "continues_on_next": true
# - If this page starts mid-paragraph from previous, set "continues_from_previous": true
# - Return ONLY valid JSON with proper commas and colons

# Return valid JSON only."""
#         return prompt

#     def fix_json_syntax(self, json_text: str) -> str:
#         """
#         Fix common JSON syntax errors from LLM output
        
#         Args:
#             json_text: Raw JSON text with potential syntax errors
            
#         Returns:
#             Fixed JSON text
#         """
#         # Remove any markdown code blocks
#         if "```json" in json_text:
#             json_start = json_text.find("```json") + 7
#             json_end = json_text.find("```", json_start)
#             json_text = json_text[json_start:json_end].strip()
#         elif "```" in json_text:
#             json_start = json_text.find("```") + 3
#             json_end = json_text.find("```", json_start)
#             json_text = json_text[json_start:json_end].strip()
        
#         # Remove comments (lines starting with # or //)
#         lines = []
#         for line in json_text.split('\n'):
#             # Skip comment lines
#             stripped = line.strip()
#             if stripped.startswith('#') or stripped.startswith('//'):
#                 continue
            
#             # Remove inline comments
#             if '#' in line or '//' in line:
#                 in_string = False
#                 cleaned = []
#                 i = 0
#                 while i < len(line):
#                     char = line[i]
                    
#                     # Track if we're inside a string
#                     if char == '"' and (i == 0 or line[i-1] != '\\'):
#                         in_string = not in_string
                    
#                     # Stop at comment if not in string
#                     if not in_string:
#                         if char == '#':
#                             break
#                         if char == '/' and i < len(line) - 1 and line[i+1] == '/':
#                             break
                    
#                     cleaned.append(char)
#                     i += 1
                
#                 line = ''.join(cleaned).rstrip()
            
#             if line.strip():
#                 lines.append(line)
        
#         json_text = '\n'.join(lines)
        
#         # Fix missing commas between key-value pairs
#         # Pattern: "key" <whitespace> "value" <newline> <whitespace> "nextkey"
#         # This matches cases where comma is missing between objects/properties
        
#         # Fix pattern: "value"\n  "key" -> "value",\n  "key"
#         json_text = re.sub(r'("\s*)\n(\s*"[^"]+"\s*:)', r'\1,\n\2', json_text)
        
#         # Fix pattern: }\n  { -> },\n  {
#         json_text = re.sub(r'(\})\n(\s*\{)', r'\1,\n\2', json_text)
        
#         # Fix pattern: ]\n  { -> ],\n  {
#         json_text = re.sub(r'(\])\n(\s*\{)', r'\1,\n\2', json_text)
        
#         # Fix pattern: }\n  ] -> }\n  ]  (this is correct, no comma needed)
#         # Fix pattern: }\n  } -> }\n  }  (this is correct, no comma needed)
        
#         # Fix missing colons: "key" "value" -> "key": "value"
#         json_text = re.sub(r'"([^"]+)"\s+"', r'"\1": "', json_text)
#         json_text = re.sub(r'"([^"]+)"\s+(\d+)', r'"\1": \2', json_text)
#         json_text = re.sub(r'"([^"]+)"\s+(true|false|null)', r'"\1": \2', json_text)
#         json_text = re.sub(r'"([^"]+)"\s+(\[|\{)', r'"\1": \2', json_text)
        
#         # Remove trailing commas before closing braces/brackets
#         json_text = re.sub(r',(\s*[\}\]])', r'\1', json_text)
        
#         # Ensure closing braces if incomplete
#         open_braces = json_text.count('{') - json_text.count('}')
#         open_brackets = json_text.count('[') - json_text.count(']')
        
#         if open_brackets > 0:
#             for _ in range(open_brackets):
#                 json_text += '\n]'
        
#         if open_braces > 0:
#             for _ in range(open_braces):
#                 json_text += '\n}'
        
#         return json_text
    
#     def validate_and_parse_json(self, json_text: str, page_number: int, max_attempts: int = 3) -> Dict:
#         """
#         Attempt to parse JSON with multiple fix strategies
        
#         Args:
#             json_text: Raw JSON text
#             page_number: Page number for error reporting
#             max_attempts: Maximum number of fix attempts
            
#         Returns:
#             Parsed JSON dictionary or error structure
#         """
#         original_text = json_text
        
#         for attempt in range(max_attempts):
#             try:
#                 # Try to parse
#                 data = json.loads(json_text)
#                 return data
            
#             except json.JSONDecodeError as e:
#                 print(f"   ⚠️  Attempt {attempt + 1}/{max_attempts}: JSON error at line {e.lineno}, col {e.colno}")
#                 print(f"       Error: {e.msg}")
                
#                 if attempt < max_attempts - 1:
#                     # Try to fix the JSON
#                     json_text = self.fix_json_syntax(json_text)
#                 else:
#                     # Last attempt failed, return error structure
#                     print(f"   ❌ Failed to parse JSON after {max_attempts} attempts")
#                     print(f"   📄 Saving raw text for manual inspection")
                    
#                     return {
#                         "page_number": page_number,
#                         "layout_type": "unknown",
#                         "raw_text": original_text,
#                         "extraction_status": "json_parse_failed",
#                         "parse_error": f"{e.msg} at line {e.lineno}",
#                         "fix_attempts": max_attempts
#                     }

#     def detect_structure_and_validate(self, extracted_data: Dict, page_number: int) -> Dict:
#         """
#         Validate and enhance the extracted structure
        
#         Args:
#             extracted_data: Raw extracted data from LLM
#             page_number: Current page number
            
#         Returns:
#             Validated and enhanced data structure
#         """
#         if not extracted_data or 'sections' not in extracted_data:
#             return extracted_data
        
#         # Add metadata
#         extracted_data['extraction_metadata'] = {
#             'page_number': page_number,
#             'validation_status': 'validated',
#             'total_sections': len(extracted_data.get('sections', [])),
#             'layout_detected': extracted_data.get('layout_type', 'unknown')
#         }
        
#         # Validate section hierarchy
#         sections = extracted_data.get('sections', [])
#         for section in sections:
#             # Ensure level is set
#             if 'level' not in section:
#                 section['level'] = 1
            
#             # Ensure header is present
#             if 'header' not in section:
#                 section['header'] = 'Untitled Section'
            
#             # Validate content structure
#             if 'content' in section:
#                 content = section['content']
#                 if isinstance(content, str):
#                     # Convert simple string to structured format
#                     section['content'] = {
#                         'type': 'paragraph',
#                         'text': content
#                     }
            
#             # Process subsections recursively
#             if 'subsections' in section:
#                 for subsection in section['subsections']:
#                     if 'level' not in subsection:
#                         subsection['level'] = section.get('level', 1) + 1
        
#         return extracted_data

#     def merge_continued_sections(self, current_data: Dict, previous_data: Optional[Dict]) -> Dict:
#         """
#         Enhanced merging of sections that continue across pages
        
#         Args:
#             current_data: Current page data
#             previous_data: Previous page data
            
#         Returns:
#             Updated current_data with proper continuation handling
#         """
#         if not previous_data or not previous_data.get('sections'):
#             return current_data
        
#         current_sections = current_data.get('sections', [])
#         previous_sections = previous_data.get('sections', [])
        
#         if not current_sections or not previous_sections:
#             return current_data
        
#         # Check first section of current page
#         first_section = current_sections[0]
#         last_prev_section = previous_sections[-1]
        
#         # Add continuation information
#         if first_section.get('continues_from_previous'):
#             first_section['continuation_info'] = {
#                 'from_page': previous_data.get('page_number'),
#                 'from_section': last_prev_section.get('header', 'Unknown'),
#                 'from_level': last_prev_section.get('level', 1)
#             }
        
#         # Mark the last section of previous page if it continues
#         if last_prev_section.get('continues_on_next'):
#             if 'continuation_info' not in last_prev_section:
#                 last_prev_section['continuation_info'] = {
#                     'to_page': current_data.get('page_number'),
#                     'to_section': first_section.get('header', 'Unknown')
#                 }
        
#         return current_data

#     def extract_from_image(self, image_path: str, page_number: int) -> Dict:
#         """
#         Extract structured data from a single image with enhanced structure detection
        
#         Args:
#             image_path: Path to the PNG image
#             page_number: Page number for reference
            
#         Returns:
#             Dictionary containing extracted data with proper structure
#         """
#         try:
#             # Convert image to base64
#             image_data_url = self.image_to_base64(image_path)
            
#             # Create enhanced prompt
#             prompt = self.create_extraction_prompt(page_number, self.previous_page_context)
            
#             # Call Groq API
#             completion = self.client.chat.completions.create(
#                 model=self.model,
#                 messages=[
#                     {
#                         "role": "user",
#                         "content": [
#                             {
#                                 "type": "text",
#                                 "text": prompt
#                             },
#                             {
#                                 "type": "image_url",
#                                 "image_url": {
#                                     "url": image_data_url
#                                 }
#                             }
#                         ]
#                     }
#                 ],
#                 temperature=0.05,  # Very low for consistent structure
#                 max_completion_tokens=8192,
#                 top_p=0.9,
#                 stream=False
#             )
            
#             # Extract response
#             response_text = completion.choices[0].message.content
            
#             # Parse JSON response with multiple fix attempts
#             extracted_data = self.validate_and_parse_json(response_text, page_number, max_attempts=3)
            
#             # Only validate if parsing succeeded
#             if extracted_data.get('extraction_status') != 'json_parse_failed':
#                 # Validate and enhance structure
#                 extracted_data = self.detect_structure_and_validate(extracted_data, page_number)
                
#                 # Update context for next page
#                 sections = extracted_data.get('sections', [])
#                 if sections:
#                     last_section = sections[-1]
#                     self.previous_page_context = last_section.get('header', '')
                
#                 # Store in history
#                 self.page_history.append({
#                     'page_number': page_number,
#                     'last_section': last_section.get('header', '') if sections else '',
#                     'layout_type': extracted_data.get('layout_type', 'unknown')
#                 })
            
#             return extracted_data
            
#         except Exception as e:
#             return {
#                 "page_number": page_number,
#                 "error": str(e),
#                 "extraction_status": "failed"
#             }
        
#     def extract_page_range(
#         self, 
#         image_dir: str, 
#         start_page: int = 1, 
#         end_page: Optional[int] = None,
#         output_file: Optional[str] = None
#     ) -> List[Dict]:
#         """
#         Extract data from a range of pages with enhanced structure detection
        
#         Args:
#             image_dir: Directory containing page images
#             start_page: Starting page number (1-indexed)
#             end_page: Ending page number (inclusive)
#             output_file: Optional JSON file to save results
            
#         Returns:
#             List of extracted data dictionaries
#         """
#         image_dir = Path(image_dir)
#         all_extractions = []
        
#         # Find all page images
#         page_files = sorted(image_dir.glob("page_*.png"))
        
#         if not page_files:
#             print(f"❌ No page images found in {image_dir}")
#             return []
        
#         # Determine page range
#         total_pages = len(page_files)
#         end_page = end_page or total_pages
        
#         # Validate range
#         if start_page < 1 or start_page > total_pages:
#             print(f"❌ Invalid start_page: {start_page}")
#             return []
        
#         if end_page > total_pages:
#             print(f"⚠️  Using {total_pages} instead of {end_page}")
#             end_page = total_pages
        
#         print(f"\n{'='*60}")
#         print(f"📄 Processing pages {start_page} to {end_page}")
#         print(f"{'='*60}\n")
        
#         # Reset context
#         self.previous_page_context = None
#         self.page_history = []
#         previous_data = None
        
#         # Process each page
#         for i in range(start_page - 1, end_page):
#             page_file = page_files[i]
#             page_num = i + 1
            
#             print(f"📖 Processing page {page_num}/{end_page}...")
            
#             extracted_data = self.extract_from_image(str(page_file), page_num)
            
#             # Merge with previous page if continuation detected
#             if previous_data and extracted_data.get('extraction_status') != 'json_parse_failed':
#                 extracted_data = self.merge_continued_sections(extracted_data, previous_data)
            
#             all_extractions.append(extracted_data)
#             previous_data = extracted_data
            
#             # Show quick summary
#             status = extracted_data.get('extraction_status', 'success')
#             if status == 'json_parse_failed':
#                 print(f"   ❌ JSON parsing failed - raw text saved")
#             else:
#                 layout = extracted_data.get('layout_type', 'unknown')
#                 num_sections = len(extracted_data.get('sections', []))
#                 print(f"   ✅ Success ({layout}, {num_sections} sections)")
        
#         # Save to file
#         if output_file:
#             output_path = Path(output_file)
#             output_path.parent.mkdir(parents=True, exist_ok=True)
            
#             with open(output_file, 'w', encoding='utf-8') as f:
#                 json.dump(all_extractions, f, indent=2, ensure_ascii=False)
#             print(f"\n✅ Results saved to {output_file}")
        
#         # Print summary
#         self._print_extraction_summary(all_extractions)
        
#         return all_extractions
    
#     def _print_extraction_summary(self, results: List[Dict]):
#         """Print a detailed summary of extraction results"""
#         print(f"\n{'='*60}")
#         print("📊 EXTRACTION SUMMARY")
#         print(f"{'='*60}\n")
        
#         success_count = 0
#         failed_count = 0
        
#         for result in results:
#             page_num = result.get('page_number', 'Unknown')
#             status = result.get('extraction_status', 'success')
            
#             if status == 'json_parse_failed':
#                 failed_count += 1
#                 print(f"❌ Page {page_num}: Parsing failed")
#                 continue
            
#             success_count += 1
#             layout = result.get('layout_type', 'Unknown')
#             sections = result.get('sections', [])
            
#             print(f"📄 Page {page_num} ({layout} layout):")
            
#             if not sections:
#                 print("   ⚠️  No sections extracted")
#                 continue
            
#             for section in sections:
#                 level = section.get('level', 1)
#                 header = section.get('header', 'No title')
#                 indent = "  " * level
                
#                 # Show continuation markers
#                 markers = []
#                 if section.get('continues_from_previous'):
#                     markers.append("⬅️ continues from prev")
#                 if section.get('continues_on_next'):
#                     markers.append("continues to next ➡️")
                
#                 marker_str = f" [{', '.join(markers)}]" if markers else ""
                
#                 print(f"{indent}{'📌' if level == 1 else '└─'} {header}{marker_str}")
                
#                 # Show subsections if any
#                 if 'subsections' in section:
#                     for subsection in section['subsections']:
#                         sub_level = subsection.get('level', level + 1)
#                         sub_header = subsection.get('header', 'No title')
#                         sub_indent = "  " * sub_level
#                         print(f"{sub_indent}└─ {sub_header}")
                
#                 # Show continuation info if available
#                 if 'continuation_info' in section:
#                     info = section['continuation_info']
#                     if 'from_page' in info:
#                         print(f"{indent}   ℹ️  Continues from page {info['from_page']}")
            
#             print()
        
#         print(f"{'='*60}")
#         print(f"✅ Successfully extracted: {success_count} pages")
#         print(f"❌ Failed to parse: {failed_count} pages")
#         print(f"{'='*60}\n")


# # Example usage
# if __name__ == "__main__":
#     extractor = EnhancedPDFImageExtractor()
    
#     # Extract page range
#     results = extractor.extract_page_range(
#         image_dir="C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\chtn_test_docs\\imgs",
#         start_page=42,
#         end_page=45,
#         output_file="extracted_data_enhanced.json"
#     )
    
#     print("\n✨ Extraction complete!")
#     print(f"📁 Total pages processed: {len(results)}")


📄 Processing pages 42 to 45

📖 Processing page 42/45...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (two_column, 4 sections)
📖 Processing page 43/45...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (mixed, 1 sections)
📖 Processing page 44/45...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (single_column, 1 sections)
📖 Processing page 45/45...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (two_column, 4 sections)

✅ Results saved to extracted_data_enhanced.json

📊 EXTRACTION SUMMARY

📄 Page 42 (two_column layout):
  📌 Pre-Engineering (A.S.)
    └─ Program Requirements
    └─ Degree Earned
  📌 Public Health (A.S.)
    └─ Program Requirements
  📌 Art Endorsement
    └─ Program Requirements
  📌 Math Endorsement
    └─ Program Requirements

📄 Page 43 (mixed layout):
  📌 Program Requirements [⬅️ continues from 

#### part 4 : page 8 to 10 was working , problem faced in 42 number page.

In [ ]:
# import os
# import json
# import base64
# import re
# from pathlib import Path
# from typing import List, Dict, Optional
# from groq import Groq
# from dotenv import load_dotenv

# # Load environment variables
# load_dotenv()

# class EnhancedPDFImageExtractor:
#     def __init__(self, api_key: Optional[str] = None):
#         """
#         Initialize the PDF Image Extractor with enhanced structure detection
        
#         Args:
#             api_key: Groq API key (if not provided, loads from env)
#         """
#         self.client = Groq(api_key=api_key or os.getenv('GROQ_API_KEY'))
#         self.model = "meta-llama/llama-4-maverick-17b-128e-instruct"
#         self.previous_page_context = None
#         self.page_history = []  # Store previous pages for better context
    
#     def image_to_base64(self, image_path: str) -> str:
#         """Convert image file to base64 data URL"""
#         with open(image_path, 'rb') as image_file:
#             encoded = base64.b64encode(image_file.read()).decode('utf-8')
#             return f"data:image/png;base64,{encoded}"
    
#     def create_extraction_prompt(self, page_number: int, previous_context: Optional[str] = None) -> str:
#         """
#         Create an enhanced prompt for automatic structure detection and extraction
        
#         Args:
#             page_number: Current page number
#             previous_context: Context from previous page
            
#         Returns:
#             Formatted prompt string
#         """
#         context_info = ""
#         if previous_context:
#             context_info = f"""
# PREVIOUS PAGE CONTEXT:
# The previous page ended with: "{previous_context}"
# If this page continues from the previous content, mark it appropriately.
# """

#         prompt = f"""Analyze this PDF page (Page {page_number}) and extract ALL content with proper structure detection.

# {context_info}

# CRITICAL: EXTRACT EVERYTHING - START FROM THE VERY TOP
# - Look at the TOP of the page first
# - Check for headers, titles, or table headings at the very top
# - Read from TOP to BOTTOM - do not skip any content
# - For tables: extract ALL rows including headers
# - For degree programs: extract the complete table with Program Requirements, Credits, etc.

# STEP 1: AUTOMATIC LAYOUT DETECTION
# First, identify the page layout:
# - "single_column": Content flows top to bottom in one column
# - "two_column": Content in LEFT column (top to bottom), then RIGHT column (top to bottom)
# - "table_of_contents": List items with page numbers
# - "mixed": Combination of layouts (describe the flow)

# STEP 2: CONTENT STRUCTURE DETECTION
# Identify all structural elements:
# - **Main Headers**: Large, bold text that starts a new major section
# - **Sub-headers**: Medium text under main headers
# - **Sub-sub-headers**: Smaller headers under sub-headers
# - **Paragraphs**: Body text content
# - **Lists**: Bulleted or numbered items
# - **Special formatting**: Italic text, bold emphasis, quotes

# STEP 3: READING ORDER (CRITICAL FOR TWO-COLUMN LAYOUTS)
# For two-column pages:
# 1. Read ENTIRE LEFT column from TOP to BOTTOM
# 2. Then read ENTIRE RIGHT column from TOP to BOTTOM
# 3. NEVER jump between columns mid-section

# STEP 4: OUTPUT FORMAT

# CRITICAL: Return VALID JSON with proper syntax:
# - ALWAYS use commas after each key-value pair (except the last one in an object)
# - ALWAYS use colons after keys
# - Use double quotes for strings
# - Example: {{"key1": "value1", "key2": "value2"}}

# For GENERAL CONTENT (paragraphs, regular text):
# {{
#   "page_number": {page_number},
#   "layout_type": "single_column",
#   "reading_order": "top_to_bottom",
#   "sections": [
#     {{
#       "level": 1,
#       "header": "Main Section Header",
#       "header_type": "main_header",
#       "continues_from_previous": false,
#       "continues_on_next": false,
#       "subsections": [
#         {{
#           "level": 2,
#           "header": "Sub-header (if any)",
#           "header_type": "sub_header",
#           "content": {{
#             "type": "paragraph",
#             "text": "Complete paragraph text...",
#             "formatting": []
#           }}
#         }}
#       ]
#     }}
#   ]
# }}

# For TABLE OF CONTENTS:
# {{
#   "page_number": {page_number},
#   "layout_type": "table_of_contents",
#   "sections": [
#     {{
#       "level": 1,
#       "header": "Main TOC Section",
#       "header_type": "toc_header",
#       "items": [
#         {{"text": "Item name", "page": "16"}},
#         {{"text": "Sub-item", "page": "17"}}
#       ]
#     }}
#   ]
# }}

# For DEGREE PROGRAM PAGES (with tables and program requirements):
# {{
#   "page_number": {page_number},
#   "layout_type": "two_column",
#   "reading_order": "left_to_right",
#   "sections": [
#     {{
#       "level": 1,
#       "header": "Program Name (e.g., Pre-Engineering A.S.)",
#       "header_type": "main_header",
#       "column": "left",
#       "subsections": [
#         {{
#           "level": 2,
#           "header": "Program Requirements",
#           "header_type": "table_header",
#           "content": {{
#             "type": "table",
#             "headers": ["Course/Item", "Credits"],
#             "rows": [
#               {{"course": "CHM 151 General Chemistry I", "credits": "5"}},
#               {{"course": "CHM 152 General Chemistry II", "credits": "4"}},
#               {{"total": "Program Credits:", "credits": "32"}}
#             ]
#           }}
#         }},
#         {{
#           "level": 2,
#           "header": "Degree Earned",
#           "header_type": "table_header",
#           "content": {{
#             "type": "table",
#             "rows": [
#               {{"item": "General Education", "credits": "39-40"}},
#               {{"item": "Program Requirements", "credits": "32"}},
#               {{"total": "Total Credits Earned:", "credits": "71-72"}}
#             ]
#           }}
#         }},
#         {{
#           "level": 2,
#           "header": "Description Paragraph",
#           "header_type": "description",
#           "content": {{
#             "type": "paragraph",
#             "text": "This degree is designed for students who..."
#           }}
#         }}
#       ]
#     }},
#     {{
#       "level": 1,
#       "header": "Choose Option A or Option B",
#       "header_type": "table_header",
#       "column": "right",
#       "subsections": [
#         {{
#           "level": 2,
#           "header": "Option A",
#           "content": {{
#             "type": "table",
#             "rows": [
#               {{"course": "PUH 220 Health & Human Disease", "credits": "3"}},
#               {{"course": "MTH 212MSP 213 Statistics", "credits": "4"}},
#               {{"total": "Program Credits:", "credits": "7"}}
#             ]
#           }}
#         }},
#         {{
#           "level": 2,
#           "header": "Option B",
#           "content": {{
#             "type": "table",
#             "rows": [
#               {{"course": "PUH 290 Introductory Public Health Research Methods", "credits": "4"}}
#             ]
#           }}
#         }}
#       ]
#     }}
#   ]
# }}

# For TWO-COLUMN NARRATIVE CONTENT (like "About Diné College"):
# {{
#   "page_number": {page_number},
#   "layout_type": "two_column",
#   "reading_order": "left_to_right",
#   "sections": [
#     {{
#       "level": 1,
#       "header": "College Mission",
#       "header_type": "main_header",
#       "column": "left",
#       "content": {{
#         "type": "multi_paragraph",
#         "paragraphs": [
#           {{"text": "First paragraph in native language...", "language": "navajo"}},
#           {{"text": "English translation...", "language": "english"}}
#         ]
#       }}
#     }}
#   ]
# }}

# CRITICAL JSON SYNTAX RULES:
# 1. ALWAYS add comma after key-value pairs (except last in object)
# 2. ALWAYS use colon after keys
# 3. Correct: {{"key": "value", "key2": "value2"}}
# 4. Wrong: {{"key" "value" "key2" "value2"}}

# CRITICAL EXTRACTION RULES:

# 1. **START FROM THE VERY TOP**: 
#    - Look at the TOP of the page FIRST
#    - Check for blue headers, table titles, or section headers at the top
#    - For degree program pages: Extract the COMPLETE tables with headers like "Program Requirements", "Credits", "Choose Option A or Option B"
   
# 2. **EXTRACT COMPLETE TABLES**:
#    - Table headers (e.g., "Program Requirements", "Credits", "Choose Option A or Option B")
#    - ALL table rows including course codes, names, and credit numbers
#    - Table totals (e.g., "Program Credits: 32", "Total Credits Earned: 71-72")
#    - Don't skip any rows or sections

# 3. **READING ORDER FOR TWO-COLUMN PAGES**:
#    - Complete LEFT column from TOP to BOTTOM (including all tables, headers, content)
#    - Then complete RIGHT column from TOP to BOTTOM
#    - NEVER skip content at the top of columns
   
# 4. **STRUCTURE PRESERVATION**:
#    - Keep table structure intact with headers and rows
#    - Maintain hierarchy: main section → subsections → tables → rows
#    - Preserve all section headers and sub-headers

# 5. **CONTENT COMPLETENESS**:
#    - Extract ALL text - nothing should be missing
#    - Include program names, descriptions, requirements, credits, options
#    - Capture "Degree Earned" sections, "Program Credits" totals, etc.

# 6. **FOR DEGREE PROGRAM PAGES SPECIFICALLY**:
#    - Extract the complete program requirements table at the top
#    - Extract the "Degree Earned" summary table
#    - Extract the program description paragraph
#    - Extract "Choose Option A or Option B" section in right column (if present)
#    - Extract any additional sections like "Art Endorsement", "Math Endorsement", etc.

# VALIDATION CHECKLIST BEFORE RETURNING:
# - [ ] All table headers extracted (Program Requirements, Credits, Choose Option A or B, Degree Earned)
# - [ ] All table rows extracted with course codes and credits
# - [ ] All section headers extracted
# - [ ] All body text extracted
# - [ ] Correct reading order (left-to-right for two columns)
# - [ ] No content from top of page is missing
# - [ ] Proper hierarchy levels assigned

# Return ONLY valid JSON with NO comments, NO trailing commas, and proper structure."""
#         return prompt

#     def fix_json_syntax(self, json_text: str) -> str:
#         """
#         Fix common JSON syntax errors from LLM output
        
#         Args:
#             json_text: Raw JSON text with potential syntax errors
            
#         Returns:
#             Fixed JSON text
#         """
#         # Remove any markdown code blocks
#         if "```json" in json_text:
#             json_start = json_text.find("```json") + 7
#             json_end = json_text.find("```", json_start)
#             json_text = json_text[json_start:json_end].strip()
#         elif "```" in json_text:
#             json_start = json_text.find("```") + 3
#             json_end = json_text.find("```", json_start)
#             json_text = json_text[json_start:json_end].strip()
        
#         # Remove comments (lines starting with # or //)
#         lines = []
#         for line in json_text.split('\n'):
#             # Skip comment lines
#             stripped = line.strip()
#             if stripped.startswith('#') or stripped.startswith('//'):
#                 continue
            
#             # Remove inline comments
#             if '#' in line or '//' in line:
#                 in_string = False
#                 cleaned = []
#                 i = 0
#                 while i < len(line):
#                     char = line[i]
                    
#                     # Track if we're inside a string
#                     if char == '"' and (i == 0 or line[i-1] != '\\'):
#                         in_string = not in_string
                    
#                     # Stop at comment if not in string
#                     if not in_string:
#                         if char == '#':
#                             break
#                         if char == '/' and i < len(line) - 1 and line[i+1] == '/':
#                             break
                    
#                     cleaned.append(char)
#                     i += 1
                
#                 line = ''.join(cleaned).rstrip()
            
#             if line.strip():
#                 lines.append(line)
        
#         json_text = '\n'.join(lines)
        
#         # Fix missing commas between key-value pairs
#         # Pattern: "key" <whitespace> "value" <newline> <whitespace> "nextkey"
#         # This matches cases where comma is missing between objects/properties
        
#         # Fix pattern: "value"\n  "key" -> "value",\n  "key"
#         json_text = re.sub(r'("\s*)\n(\s*"[^"]+"\s*:)', r'\1,\n\2', json_text)
        
#         # Fix pattern: }\n  { -> },\n  {
#         json_text = re.sub(r'(\})\n(\s*\{)', r'\1,\n\2', json_text)
        
#         # Fix pattern: ]\n  { -> ],\n  {
#         json_text = re.sub(r'(\])\n(\s*\{)', r'\1,\n\2', json_text)
        
#         # Fix pattern: }\n  ] -> }\n  ]  (this is correct, no comma needed)
#         # Fix pattern: }\n  } -> }\n  }  (this is correct, no comma needed)
        
#         # Fix missing colons: "key" "value" -> "key": "value"
#         json_text = re.sub(r'"([^"]+)"\s+"', r'"\1": "', json_text)
#         json_text = re.sub(r'"([^"]+)"\s+(\d+)', r'"\1": \2', json_text)
#         json_text = re.sub(r'"([^"]+)"\s+(true|false|null)', r'"\1": \2', json_text)
#         json_text = re.sub(r'"([^"]+)"\s+(\[|\{)', r'"\1": \2', json_text)
        
#         # Remove trailing commas before closing braces/brackets
#         json_text = re.sub(r',(\s*[\}\]])', r'\1', json_text)
        
#         # Ensure closing braces if incomplete
#         open_braces = json_text.count('{') - json_text.count('}')
#         open_brackets = json_text.count('[') - json_text.count(']')
        
#         if open_brackets > 0:
#             for _ in range(open_brackets):
#                 json_text += '\n]'
        
#         if open_braces > 0:
#             for _ in range(open_braces):
#                 json_text += '\n}'
        
#         return json_text
    
#     def validate_and_parse_json(self, json_text: str, page_number: int, max_attempts: int = 3) -> Dict:
#         """
#         Attempt to parse JSON with multiple fix strategies
        
#         Args:
#             json_text: Raw JSON text
#             page_number: Page number for error reporting
#             max_attempts: Maximum number of fix attempts
            
#         Returns:
#             Parsed JSON dictionary or error structure
#         """
#         original_text = json_text
        
#         for attempt in range(max_attempts):
#             try:
#                 # Try to parse
#                 data = json.loads(json_text)
#                 return data
            
#             except json.JSONDecodeError as e:
#                 print(f"   ⚠️  Attempt {attempt + 1}/{max_attempts}: JSON error at line {e.lineno}, col {e.colno}")
#                 print(f"       Error: {e.msg}")
                
#                 if attempt < max_attempts - 1:
#                     # Try to fix the JSON
#                     json_text = self.fix_json_syntax(json_text)
#                 else:
#                     # Last attempt failed, return error structure
#                     print(f"   ❌ Failed to parse JSON after {max_attempts} attempts")
#                     print(f"   📄 Saving raw text for manual inspection")
                    
#                     return {
#                         "page_number": page_number,
#                         "layout_type": "unknown",
#                         "raw_text": original_text,
#                         "extraction_status": "json_parse_failed",
#                         "parse_error": f"{e.msg} at line {e.lineno}",
#                         "fix_attempts": max_attempts
#                     }

#     def detect_structure_and_validate(self, extracted_data: Dict, page_number: int) -> Dict:
#         """
#         Validate and enhance the extracted structure
        
#         Args:
#             extracted_data: Raw extracted data from LLM
#             page_number: Current page number
            
#         Returns:
#             Validated and enhanced data structure
#         """
#         if not extracted_data or 'sections' not in extracted_data:
#             return extracted_data
        
#         # Add metadata
#         extracted_data['extraction_metadata'] = {
#             'page_number': page_number,
#             'validation_status': 'validated',
#             'total_sections': len(extracted_data.get('sections', [])),
#             'layout_detected': extracted_data.get('layout_type', 'unknown')
#         }
        
#         # Validate section hierarchy
#         sections = extracted_data.get('sections', [])
        
#         # Check if this looks like a degree program page
#         is_degree_page = self._is_degree_program_page(sections)
#         if is_degree_page:
#             # Add warning if key sections are missing
#             missing_sections = self._check_degree_page_completeness(sections)
#             if missing_sections:
#                 extracted_data['extraction_metadata']['warnings'] = missing_sections
        
#         for section in sections:
#             # Ensure level is set
#             if 'level' not in section:
#                 section['level'] = 1
            
#             # Ensure header is present
#             if 'header' not in section:
#                 section['header'] = 'Untitled Section'
            
#             # Validate content structure
#             if 'content' in section:
#                 content = section['content']
#                 if isinstance(content, str):
#                     # Convert simple string to structured format
#                     section['content'] = {
#                         'type': 'paragraph',
#                         'text': content
#                     }
            
#             # Process subsections recursively
#             if 'subsections' in section:
#                 for subsection in section['subsections']:
#                     if 'level' not in subsection:
#                         subsection['level'] = section.get('level', 1) + 1
        
#         return extracted_data
    
#     def _is_degree_program_page(self, sections: List[Dict]) -> bool:
#         """Check if this appears to be a degree program page"""
#         if not sections:
#             return False
        
#         # Look for keywords that indicate degree programs
#         degree_keywords = ['program requirements', 'degree earned', 'credits', 
#                           'bachelor', 'associate', 'master', 'endorsement']
        
#         for section in sections:
#             header = section.get('header', '').lower()
#             if any(keyword in header for keyword in degree_keywords):
#                 return True
            
#             # Check subsections
#             subsections = section.get('subsections', [])
#             for subsection in subsections:
#                 sub_header = subsection.get('header', '').lower()
#                 if any(keyword in sub_header for keyword in degree_keywords):
#                     return True
        
#         return False
    
#     def _check_degree_page_completeness(self, sections: List[Dict]) -> List[str]:
#         """Check if degree program page has all expected sections"""
#         warnings = []
        
#         # Expected sections for degree program pages
#         expected_sections = [
#             'program requirements',
#             'degree earned',
#             'credits'
#         ]
        
#         found_sections = set()
        
#         # Check all sections and subsections
#         for section in sections:
#             header = section.get('header', '').lower()
#             found_sections.add(header)
            
#             subsections = section.get('subsections', [])
#             for subsection in subsections:
#                 sub_header = subsection.get('header', '').lower()
#                 found_sections.add(sub_header)
        
#         # Check for missing expected sections
#         for expected in expected_sections:
#             if not any(expected in found for found in found_sections):
#                 warnings.append(f"Potentially missing section: '{expected}' - may be incomplete extraction")
        
#         return warnings

#     def merge_continued_sections(self, current_data: Dict, previous_data: Optional[Dict]) -> Dict:
#         """
#         Enhanced merging of sections that continue across pages
        
#         Args:
#             current_data: Current page data
#             previous_data: Previous page data
            
#         Returns:
#             Updated current_data with proper continuation handling
#         """
#         if not previous_data or not previous_data.get('sections'):
#             return current_data
        
#         current_sections = current_data.get('sections', [])
#         previous_sections = previous_data.get('sections', [])
        
#         if not current_sections or not previous_sections:
#             return current_data
        
#         # Check first section of current page
#         first_section = current_sections[0]
#         last_prev_section = previous_sections[-1]
        
#         # Add continuation information
#         if first_section.get('continues_from_previous'):
#             first_section['continuation_info'] = {
#                 'from_page': previous_data.get('page_number'),
#                 'from_section': last_prev_section.get('header', 'Unknown'),
#                 'from_level': last_prev_section.get('level', 1)
#             }
        
#         # Mark the last section of previous page if it continues
#         if last_prev_section.get('continues_on_next'):
#             if 'continuation_info' not in last_prev_section:
#                 last_prev_section['continuation_info'] = {
#                     'to_page': current_data.get('page_number'),
#                     'to_section': first_section.get('header', 'Unknown')
#                 }
        
#         return current_data

#     def extract_from_image(self, image_path: str, page_number: int) -> Dict:
#         """
#         Extract structured data from a single image with enhanced structure detection
        
#         Args:
#             image_path: Path to the PNG image
#             page_number: Page number for reference
            
#         Returns:
#             Dictionary containing extracted data with proper structure
#         """
#         try:
#             # Convert image to base64
#             image_data_url = self.image_to_base64(image_path)
            
#             # Create enhanced prompt
#             prompt = self.create_extraction_prompt(page_number, self.previous_page_context)
            
#             # Call Groq API
#             completion = self.client.chat.completions.create(
#                 model=self.model,
#                 messages=[
#                     {
#                         "role": "user",
#                         "content": [
#                             {
#                                 "type": "text",
#                                 "text": prompt
#                             },
#                             {
#                                 "type": "image_url",
#                                 "image_url": {
#                                     "url": image_data_url
#                                 }
#                             }
#                         ]
#                     }
#                 ],
#                 temperature=0.05,  # Very low for consistent structure
#                 max_completion_tokens=8192,
#                 top_p=0.9,
#                 stream=False
#             )
            
#             # Extract response
#             response_text = completion.choices[0].message.content
            
#             # Parse JSON response with multiple fix attempts
#             extracted_data = self.validate_and_parse_json(response_text, page_number, max_attempts=3)
            
#             # Only validate if parsing succeeded
#             if extracted_data.get('extraction_status') != 'json_parse_failed':
#                 # Validate and enhance structure
#                 extracted_data = self.detect_structure_and_validate(extracted_data, page_number)
                
#                 # Update context for next page
#                 sections = extracted_data.get('sections', [])
#                 if sections:
#                     last_section = sections[-1]
#                     self.previous_page_context = last_section.get('header', '')
                
#                 # Store in history
#                 self.page_history.append({
#                     'page_number': page_number,
#                     'last_section': last_section.get('header', '') if sections else '',
#                     'layout_type': extracted_data.get('layout_type', 'unknown')
#                 })
            
#             return extracted_data
            
#         except Exception as e:
#             return {
#                 "page_number": page_number,
#                 "error": str(e),
#                 "extraction_status": "failed"
#             }
        
#     def extract_page_range(
#         self, 
#         image_dir: str, 
#         start_page: int = 1, 
#         end_page: Optional[int] = None,
#         output_file: Optional[str] = None
#     ) -> List[Dict]:
#         """
#         Extract data from a range of pages with enhanced structure detection
        
#         Args:
#             image_dir: Directory containing page images
#             start_page: Starting page number (1-indexed)
#             end_page: Ending page number (inclusive)
#             output_file: Optional JSON file to save results
            
#         Returns:
#             List of extracted data dictionaries
#         """
#         image_dir = Path(image_dir)
#         all_extractions = []
        
#         # Find all page images
#         page_files = sorted(image_dir.glob("page_*.png"))
        
#         if not page_files:
#             print(f"❌ No page images found in {image_dir}")
#             return []
        
#         # Determine page range
#         total_pages = len(page_files)
#         end_page = end_page or total_pages
        
#         # Validate range
#         if start_page < 1 or start_page > total_pages:
#             print(f"❌ Invalid start_page: {start_page}")
#             return []
        
#         if end_page > total_pages:
#             print(f"⚠️  Using {total_pages} instead of {end_page}")
#             end_page = total_pages
        
#         print(f"\n{'='*60}")
#         print(f"📄 Processing pages {start_page} to {end_page}")
#         print(f"{'='*60}\n")
        
#         # Reset context
#         self.previous_page_context = None
#         self.page_history = []
#         previous_data = None
        
#         # Process each page
#         for i in range(start_page - 1, end_page):
#             page_file = page_files[i]
#             page_num = i + 1
            
#             print(f"📖 Processing page {page_num}/{end_page}...")
            
#             extracted_data = self.extract_from_image(str(page_file), page_num)
            
#             # Merge with previous page if continuation detected
#             if previous_data and extracted_data.get('extraction_status') != 'json_parse_failed':
#                 extracted_data = self.merge_continued_sections(extracted_data, previous_data)
            
#             all_extractions.append(extracted_data)
#             previous_data = extracted_data
            
#             # Show quick summary
#             status = extracted_data.get('extraction_status', 'success')
#             if status == 'json_parse_failed':
#                 print(f"   ❌ JSON parsing failed - raw text saved")
#             else:
#                 layout = extracted_data.get('layout_type', 'unknown')
#                 num_sections = len(extracted_data.get('sections', []))
#                 print(f"   ✅ Success ({layout}, {num_sections} sections)")
        
#         # Save to file
#         if output_file:
#             output_path = Path(output_file)
#             output_path.parent.mkdir(parents=True, exist_ok=True)
            
#             with open(output_file, 'w', encoding='utf-8') as f:
#                 json.dump(all_extractions, f, indent=2, ensure_ascii=False)
#             print(f"\n✅ Results saved to {output_file}")
        
#         # Print summary
#         self._print_extraction_summary(all_extractions)
        
#         return all_extractions
    
#     def _print_extraction_summary(self, results: List[Dict]):
#         """Print a detailed summary of extraction results"""
#         print(f"\n{'='*60}")
#         print("📊 EXTRACTION SUMMARY")
#         print(f"{'='*60}\n")
        
#         success_count = 0
#         failed_count = 0
#         warning_count = 0
        
#         for result in results:
#             page_num = result.get('page_number', 'Unknown')
#             status = result.get('extraction_status', 'success')
            
#             if status == 'json_parse_failed':
#                 failed_count += 1
#                 print(f"❌ Page {page_num}: Parsing failed")
#                 continue
            
#             success_count += 1
#             layout = result.get('layout_type', 'Unknown')
#             sections = result.get('sections', [])
            
#             # Check for warnings
#             metadata = result.get('extraction_metadata', {})
#             warnings = metadata.get('warnings', [])
            
#             warning_indicator = " ⚠️" if warnings else ""
#             print(f"📄 Page {page_num} ({layout} layout){warning_indicator}:")
            
#             if warnings:
#                 warning_count += 1
#                 for warning in warnings:
#                     print(f"   ⚠️  {warning}")
            
#             if not sections:
#                 print("   ⚠️  No sections extracted")
#                 continue
            
#             for section in sections:
#                 level = section.get('level', 1)
#                 header = section.get('header', 'No title')
#                 column = section.get('column', '')
#                 indent = "  " * level
                
#                 # Show column info for two-column layouts
#                 column_info = f" [{column}]" if column else ""
                
#                 # Show continuation markers
#                 markers = []
#                 if section.get('continues_from_previous'):
#                     markers.append("⬅️ continues from prev")
#                 if section.get('continues_on_next'):
#                     markers.append("continues to next ➡️")
                
#                 marker_str = f" [{', '.join(markers)}]" if markers else ""
                
#                 print(f"{indent}{'📌' if level == 1 else '└─'} {header}{column_info}{marker_str}")
                
#                 # Show subsections if any
#                 if 'subsections' in section:
#                     for subsection in section['subsections']:
#                         sub_level = subsection.get('level', level + 1)
#                         sub_header = subsection.get('header', 'No title')
#                         sub_indent = "  " * sub_level
                        
#                         # Check if it's a table
#                         content = subsection.get('content', {})
#                         content_type = content.get('type', 'unknown') if isinstance(content, dict) else 'unknown'
#                         type_indicator = " 📊" if content_type == 'table' else ""
                        
#                         print(f"{sub_indent}└─ {sub_header}{type_indicator}")
                        
#                         # Show table row count if it's a table
#                         if content_type == 'table':
#                             rows = content.get('rows', [])
#                             print(f"{sub_indent}   ({len(rows)} rows)")
                
#                 # Show continuation info if available
#                 if 'continuation_info' in section:
#                     info = section['continuation_info']
#                     if 'from_page' in info:
#                         print(f"{indent}   ℹ️  Continues from page {info['from_page']}")
            
#             print()
        
#         print(f"{'='*60}")
#         print(f"✅ Successfully extracted: {success_count} pages")
#         if warning_count > 0:
#             print(f"⚠️  Pages with warnings: {warning_count} (may be incomplete)")
#         print(f"❌ Failed to parse: {failed_count} pages")
#         print(f"{'='*60}\n")


# if __name__ == "__main__":
#     extractor = EnhancedPDFImageExtractor()
    
#     # Extract page range
#     results = extractor.extract_page_range(
#         image_dir="C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\chtn_test_docs\\imgs",
#         start_page=60,
#         end_page=64,
#         output_file="extracted_data_enhanced.json"
#     )
    
#     print("\n✨ Extraction complete!")
#     print(f"📁 Total pages processed: {len(results)}")


📄 Processing pages 60 to 64

📖 Processing page 60/64...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (two_column, 2 sections)
📖 Processing page 61/64...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (two_column, 2 sections)
📖 Processing page 62/64...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (two_column, 2 sections)
📖 Processing page 63/64...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (two_column, 2 sections)
📖 Processing page 64/64...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (two_column, 2 sections)

✅ Results saved to extracted_data_enhanced.json

📊 EXTRACTION SUMMARY

📄 Page 60 (two_column layout) ⚠️:
   ⚠️  Potentially missing section: 'program requirements' - may be incomplete extraction
   ⚠️  Potentially missing section: 'credits' - may be inco

#### Part 5:- Below code is working fine. i am going to use the output to structure that and then it will be converted to emebeddings. 

In [ ]:
# import os
# import json
# import base64
# import re
# from pathlib import Path
# from typing import List, Dict, Optional
# from groq import Groq
# from dotenv import load_dotenv

# # Load environment variables
# load_dotenv()

# class EnhancedPDFImageExtractor:
#     def __init__(self, api_key: Optional[str] = None):
#         """
#         Initialize the PDF Image Extractor with enhanced structure detection
        
#         Args:
#             api_key: Groq API key (if not provided, loads from env)
#         """
#         self.client = Groq(api_key=api_key or os.getenv('GROQ_API_KEY'))
#         self.model = "meta-llama/llama-4-maverick-17b-128e-instruct"
#         self.previous_page_context = None
#         self.page_history = []  # Store previous pages for better context
    
#     def image_to_base64(self, image_path: str) -> str:
#         """Convert image file to base64 data URL"""
#         with open(image_path, 'rb') as image_file:
#             encoded = base64.b64encode(image_file.read()).decode('utf-8')
#             return f"data:image/png;base64,{encoded}"
    
#     def create_extraction_prompt(self, page_number: int, previous_context: Optional[str] = None) -> str:
#         """
#         Create an enhanced prompt for automatic structure detection and extraction
        
#         Args:
#             page_number: Current page number
#             previous_context: Context from previous page
            
#         Returns:
#             Formatted prompt string
#         """
#         context_info = ""
#         if previous_context:
#             context_info = f"""
# PREVIOUS PAGE CONTEXT:
# The previous page ended with: "{previous_context}"
# If this page continues from the previous content, mark it appropriately.
# """

#         prompt = f"""Analyze this PDF page (Page {page_number}) and extract ALL content with proper structure detection.

# {context_info}

# CRITICAL: EXTRACT EVERYTHING - START FROM THE VERY TOP
# - Look at the TOP of the page first
# - Check for headers, titles, or table headings at the very top
# - Read from TOP to BOTTOM - do not skip any content
# - For tables: extract ALL rows including headers
# - For degree programs: extract the complete table with Program Requirements, Credits, etc.

# STEP 1: AUTOMATIC LAYOUT DETECTION
# First, identify the page layout:
# - "single_column": Content flows top to bottom in one column
# - "two_column": Content in LEFT column (top to bottom), then RIGHT column (top to bottom)
# - "table_of_contents": List items with page numbers
# - "mixed": Combination of layouts (describe the flow)

# STEP 2: CONTENT STRUCTURE DETECTION
# Identify all structural elements:
# - **Main Headers**: Large, bold text that starts a new major section
# - **Sub-headers**: Medium text under main headers
# - **Sub-sub-headers**: Smaller headers under sub-headers
# - **Paragraphs**: Body text content
# - **Lists**: Bulleted or numbered items
# - **Special formatting**: Italic text, bold emphasis, quotes

# STEP 3: READING ORDER (CRITICAL FOR TWO-COLUMN LAYOUTS)
# For two-column pages:
# 1. Read ENTIRE LEFT column from TOP to BOTTOM
# 2. Then read ENTIRE RIGHT column from TOP to BOTTOM
# 3. NEVER jump between columns mid-section

# STEP 4: OUTPUT FORMAT

# CRITICAL: Return VALID JSON with proper syntax:
# - ALWAYS use commas after each key-value pair (except the last one in an object)
# - ALWAYS use colons after keys
# - Use double quotes for strings
# - Example: {{"key1": "value1", "key2": "value2"}}

# For GENERAL CONTENT (paragraphs, regular text):
# {{
#   "page_number": {page_number},
#   "layout_type": "single_column",
#   "reading_order": "top_to_bottom",
#   "sections": [
#     {{
#       "level": 1,
#       "header": "Main Section Header",
#       "header_type": "main_header",
#       "continues_from_previous": false,
#       "continues_on_next": false,
#       "subsections": [
#         {{
#           "level": 2,
#           "header": "Sub-header (if any)",
#           "header_type": "sub_header",
#           "content": {{
#             "type": "paragraph",
#             "text": "Complete paragraph text...",
#             "formatting": []
#           }}
#         }}
#       ]
#     }}
#   ]
# }}

# For TABLE OF CONTENTS:
# {{
#   "page_number": {page_number},
#   "layout_type": "table_of_contents",
#   "sections": [
#     {{
#       "level": 1,
#       "header": "Main TOC Section",
#       "header_type": "toc_header",
#       "items": [
#         {{"text": "Item name", "page": "16"}},
#         {{"text": "Sub-item", "page": "17"}}
#       ]
#     }}
#   ]
# }}

# For DEGREE PROGRAM PAGES (with tables and program requirements):
# {{
#   "page_number": {page_number},
#   "layout_type": "two_column",
#   "reading_order": "left_to_right",
#   "sections": [
#     {{
#       "level": 1,
#       "header": "Program Name (e.g., Pre-Engineering A.S.)",
#       "header_type": "main_header",
#       "column": "left",
#       "subsections": [
#         {{
#           "level": 2,
#           "header": "Program Requirements",
#           "header_type": "table_header",
#           "content": {{
#             "type": "table",
#             "headers": ["Course/Item", "Credits"],
#             "rows": [
#               {{"course": "CHM 151 General Chemistry I", "credits": "5"}},
#               {{"course": "CHM 152 General Chemistry II", "credits": "4"}},
#               {{"total": "Program Credits:", "credits": "32"}}
#             ]
#           }}
#         }},
#         {{
#           "level": 2,
#           "header": "Degree Earned",
#           "header_type": "table_header",
#           "content": {{
#             "type": "table",
#             "rows": [
#               {{"item": "General Education", "credits": "39-40"}},
#               {{"item": "Program Requirements", "credits": "32"}},
#               {{"total": "Total Credits Earned:", "credits": "71-72"}}
#             ]
#           }}
#         }},
#         {{
#           "level": 2,
#           "header": "Description Paragraph",
#           "header_type": "description",
#           "content": {{
#             "type": "paragraph",
#             "text": "This degree is designed for students who..."
#           }}
#         }}
#       ]
#     }},
#     {{
#       "level": 1,
#       "header": "Choose Option A or Option B",
#       "header_type": "table_header",
#       "column": "right",
#       "subsections": [
#         {{
#           "level": 2,
#           "header": "Option A",
#           "content": {{
#             "type": "table",
#             "rows": [
#               {{"course": "PUH 220 Health & Human Disease", "credits": "3"}},
#               {{"course": "MTH 212MSP 213 Statistics", "credits": "4"}},
#               {{"total": "Program Credits:", "credits": "7"}}
#             ]
#           }}
#         }},
#         {{
#           "level": 2,
#           "header": "Option B",
#           "content": {{
#             "type": "table",
#             "rows": [
#               {{"course": "PUH 290 Introductory Public Health Research Methods", "credits": "4"}}
#             ]
#           }}
#         }}
#       ]
#     }}
#   ]
# }}

# For TWO-COLUMN NARRATIVE CONTENT (like "About Diné College"):
# {{
#   "page_number": {page_number},
#   "layout_type": "two_column",
#   "reading_order": "left_to_right",
#   "sections": [
#     {{
#       "level": 1,
#       "header": "College Mission",
#       "header_type": "main_header",
#       "column": "left",
#       "content": {{
#         "type": "multi_paragraph",
#         "paragraphs": [
#           {{"text": "First paragraph in native language...", "language": "navajo"}},
#           {{"text": "English translation...", "language": "english"}}
#         ]
#       }}
#     }}
#   ]
# }}

# CRITICAL JSON SYNTAX RULES:
# 1. ALWAYS add comma after key-value pairs (except last in object)
# 2. ALWAYS use colon after keys
# 3. Correct: {{"key": "value", "key2": "value2"}}
# 4. Wrong: {{"key" "value" "key2" "value2"}}

# CRITICAL EXTRACTION RULES:

# 1. **START FROM THE VERY TOP**: 
#    - Look at the TOP of the page FIRST
#    - Check for blue headers, table titles, or section headers at the top
#    - For degree program pages: Extract the COMPLETE tables with headers like "Program Requirements", "Credits", "Choose Option A or Option B"

# 2. **DETECT INCOMPLETE CONTENT**:
#    - If a course description is missing or empty, mark "continues_on_next": true
#    - If a paragraph starts mid-sentence without context, mark "continues_from_previous": true
#    - For course descriptions: if description is empty or very short (< 20 words), it likely continues on next page
   
# 3. **EXTRACT COMPLETE TABLES**:
#    - Table headers (e.g., "Program Requirements", "Credits", "Choose Option A or Option B")
#    - ALL table rows including course codes, names, and credit numbers
#    - Table totals (e.g., "Program Credits: 32", "Total Credits Earned: 71-72")
#    - Don't skip any rows or sections

# 4. **READING ORDER FOR TWO-COLUMN PAGES**:
#    - Complete LEFT column from TOP to BOTTOM (including all tables, headers, content)
#    - Then complete RIGHT column from TOP to BOTTOM
#    - NEVER skip content at the top of columns
   
# 5. **STRUCTURE PRESERVATION**:
#    - Keep table structure intact with headers and rows
#    - Maintain hierarchy: main section → subsections → tables → rows
#    - Preserve all section headers and sub-headers

# 6. **CONTENT COMPLETENESS**:
#    - Extract ALL text - nothing should be missing
#    - Include program names, descriptions, requirements, credits, options
#    - Capture "Degree Earned" sections, "Program Credits" totals, etc.
#    - For course descriptions: include COMPLETE description text, even if partial

# 7. **FOR COURSE DESCRIPTION PAGES**:
#    - Extract course code (e.g., "AGR 313")
#    - Extract course title (e.g., "Agricultural Genetics")
#    - Extract credits (e.g., "(3)")
#    - Extract COMPLETE description paragraph
#    - If description is cut off or appears on next page, extract what's visible and mark "continues_on_next": true

# 8. **FOR DEGREE PROGRAM PAGES SPECIFICALLY**:
#    - Extract the complete program requirements table at the top
#    - Extract the "Degree Earned" summary table
#    - Extract the program description paragraph
#    - Extract "Choose Option A or Option B" section in right column (if present)
#    - Extract any additional sections like "Art Endorsement", "Math Endorsement", etc.

# VALIDATION CHECKLIST BEFORE RETURNING:
# - [ ] All table headers extracted (Program Requirements, Credits, Choose Option A or B, Degree Earned)
# - [ ] All table rows extracted with course codes and credits
# - [ ] All section headers extracted
# - [ ] All body text extracted
# - [ ] Check if last item has empty/short description - mark as "continues_on_next": true
# - [ ] Correct reading order (left-to-right for two columns)
# - [ ] No content from top of page is missing
# - [ ] Proper hierarchy levels assigned

# Return ONLY valid JSON with NO comments, NO trailing commas, and proper structure."""
#         return prompt

#     def fix_json_syntax(self, json_text: str) -> str:
#         """
#         Fix common JSON syntax errors from LLM output
        
#         Args:
#             json_text: Raw JSON text with potential syntax errors
            
#         Returns:
#             Fixed JSON text
#         """
#         # Remove any markdown code blocks
#         if "```json" in json_text:
#             json_start = json_text.find("```json") + 7
#             json_end = json_text.find("```", json_start)
#             json_text = json_text[json_start:json_end].strip()
#         elif "```" in json_text:
#             json_start = json_text.find("```") + 3
#             json_end = json_text.find("```", json_start)
#             json_text = json_text[json_start:json_end].strip()
        
#         # Remove comments (lines starting with # or //)
#         lines = []
#         for line in json_text.split('\n'):
#             # Skip comment lines
#             stripped = line.strip()
#             if stripped.startswith('#') or stripped.startswith('//'):
#                 continue
            
#             # Remove inline comments
#             if '#' in line or '//' in line:
#                 in_string = False
#                 cleaned = []
#                 i = 0
#                 while i < len(line):
#                     char = line[i]
                    
#                     # Track if we're inside a string
#                     if char == '"' and (i == 0 or line[i-1] != '\\'):
#                         in_string = not in_string
                    
#                     # Stop at comment if not in string
#                     if not in_string:
#                         if char == '#':
#                             break
#                         if char == '/' and i < len(line) - 1 and line[i+1] == '/':
#                             break
                    
#                     cleaned.append(char)
#                     i += 1
                
#                 line = ''.join(cleaned).rstrip()
            
#             if line.strip():
#                 lines.append(line)
        
#         json_text = '\n'.join(lines)
        
#         # Fix missing commas between key-value pairs
#         # Pattern: "key" <whitespace> "value" <newline> <whitespace> "nextkey"
#         # This matches cases where comma is missing between objects/properties
        
#         # Fix pattern: "value"\n  "key" -> "value",\n  "key"
#         json_text = re.sub(r'("\s*)\n(\s*"[^"]+"\s*:)', r'\1,\n\2', json_text)
        
#         # Fix pattern: }\n  { -> },\n  {
#         json_text = re.sub(r'(\})\n(\s*\{)', r'\1,\n\2', json_text)
        
#         # Fix pattern: ]\n  { -> ],\n  {
#         json_text = re.sub(r'(\])\n(\s*\{)', r'\1,\n\2', json_text)
        
#         # Fix pattern: }\n  ] -> }\n  ]  (this is correct, no comma needed)
#         # Fix pattern: }\n  } -> }\n  }  (this is correct, no comma needed)
        
#         # Fix missing colons: "key" "value" -> "key": "value"
#         json_text = re.sub(r'"([^"]+)"\s+"', r'"\1": "', json_text)
#         json_text = re.sub(r'"([^"]+)"\s+(\d+)', r'"\1": \2', json_text)
#         json_text = re.sub(r'"([^"]+)"\s+(true|false|null)', r'"\1": \2', json_text)
#         json_text = re.sub(r'"([^"]+)"\s+(\[|\{)', r'"\1": \2', json_text)
        
#         # Remove trailing commas before closing braces/brackets
#         json_text = re.sub(r',(\s*[\}\]])', r'\1', json_text)
        
#         # Ensure closing braces if incomplete
#         open_braces = json_text.count('{') - json_text.count('}')
#         open_brackets = json_text.count('[') - json_text.count(']')
        
#         if open_brackets > 0:
#             for _ in range(open_brackets):
#                 json_text += '\n]'
        
#         if open_braces > 0:
#             for _ in range(open_braces):
#                 json_text += '\n}'
        
#         return json_text
    
#     def validate_and_parse_json(self, json_text: str, page_number: int, max_attempts: int = 3) -> Dict:
#         """
#         Attempt to parse JSON with multiple fix strategies
        
#         Args:
#             json_text: Raw JSON text
#             page_number: Page number for error reporting
#             max_attempts: Maximum number of fix attempts
            
#         Returns:
#             Parsed JSON dictionary or error structure
#         """
#         original_text = json_text
        
#         for attempt in range(max_attempts):
#             try:
#                 # Try to parse
#                 data = json.loads(json_text)
#                 return data
            
#             except json.JSONDecodeError as e:
#                 print(f"   ⚠️  Attempt {attempt + 1}/{max_attempts}: JSON error at line {e.lineno}, col {e.colno}")
#                 print(f"       Error: {e.msg}")
                
#                 if attempt < max_attempts - 1:
#                     # Try to fix the JSON
#                     json_text = self.fix_json_syntax(json_text)
#                 else:
#                     # Last attempt failed, return error structure
#                     print(f"   ❌ Failed to parse JSON after {max_attempts} attempts")
#                     print(f"   📄 Saving raw text for manual inspection")
                    
#                     return {
#                         "page_number": page_number,
#                         "layout_type": "unknown",
#                         "raw_text": original_text,
#                         "extraction_status": "json_parse_failed",
#                         "parse_error": f"{e.msg} at line {e.lineno}",
#                         "fix_attempts": max_attempts
#                     }

#     def detect_structure_and_validate(self, extracted_data: Dict, page_number: int) -> Dict:
#         """
#         Validate and enhance the extracted structure
        
#         Args:
#             extracted_data: Raw extracted data from LLM
#             page_number: Current page number
            
#         Returns:
#             Validated and enhanced data structure
#         """
#         if not extracted_data or 'sections' not in extracted_data:
#             return extracted_data
        
#         # Add metadata
#         extracted_data['extraction_metadata'] = {
#             'page_number': page_number,
#             'validation_status': 'validated',
#             'total_sections': len(extracted_data.get('sections', [])),
#             'layout_detected': extracted_data.get('layout_type', 'unknown')
#         }
        
#         # Validate section hierarchy
#         sections = extracted_data.get('sections', [])
        
#         # Check if this looks like a degree program page
#         is_degree_page = self._is_degree_program_page(sections)
#         if is_degree_page:
#             # Add warning if key sections are missing
#             missing_sections = self._check_degree_page_completeness(sections)
#             if missing_sections:
#                 extracted_data['extraction_metadata']['warnings'] = missing_sections
        
#         for section in sections:
#             # Ensure level is set
#             if 'level' not in section:
#                 section['level'] = 1
            
#             # Ensure header is present
#             if 'header' not in section:
#                 section['header'] = 'Untitled Section'
            
#             # Validate content structure
#             if 'content' in section:
#                 content = section['content']
#                 if isinstance(content, str):
#                     # Convert simple string to structured format
#                     section['content'] = {
#                         'type': 'paragraph',
#                         'text': content
#                     }
            
#             # Process subsections recursively
#             if 'subsections' in section:
#                 for subsection in section['subsections']:
#                     if 'level' not in subsection:
#                         subsection['level'] = section.get('level', 1) + 1
        
#         return extracted_data
    
#     def _is_degree_program_page(self, sections: List[Dict]) -> bool:
#         """Check if this appears to be a degree program page"""
#         if not sections:
#             return False
        
#         # Look for keywords that indicate degree programs
#         degree_keywords = ['program requirements', 'degree earned', 'credits', 
#                           'bachelor', 'associate', 'master', 'endorsement']
        
#         for section in sections:
#             header = section.get('header', '').lower()
#             if any(keyword in header for keyword in degree_keywords):
#                 return True
            
#             # Check subsections
#             subsections = section.get('subsections', [])
#             for subsection in subsections:
#                 sub_header = subsection.get('header', '').lower()
#                 if any(keyword in sub_header for keyword in degree_keywords):
#                     return True
        
#         return False
    
#     def _check_degree_page_completeness(self, sections: List[Dict]) -> List[str]:
#         """Check if degree program page has all expected sections"""
#         warnings = []
        
#         # Expected sections for degree program pages
#         expected_sections = [
#             'program requirements',
#             'degree earned',
#             'credits'
#         ]
        
#         found_sections = set()
        
#         # Check all sections and subsections
#         for section in sections:
#             header = section.get('header', '').lower()
#             found_sections.add(header)
            
#             subsections = section.get('subsections', [])
#             for subsection in subsections:
#                 sub_header = subsection.get('header', '').lower()
#                 found_sections.add(sub_header)
        
#         # Check for missing expected sections
#         for expected in expected_sections:
#             if not any(expected in found for found in found_sections):
#                 warnings.append(f"Potentially missing section: '{expected}' - may be incomplete extraction")
        
#         return warnings

#     def merge_continued_sections(self, current_data: Dict, previous_data: Optional[Dict]) -> Dict:
#         """
#         Enhanced merging of sections that continue across pages
        
#         Args:
#             current_data: Current page data
#             previous_data: Previous page data
            
#         Returns:
#             Updated current_data with proper continuation handling
#         """
#         if not previous_data or not previous_data.get('sections'):
#             return current_data
        
#         current_sections = current_data.get('sections', [])
#         previous_sections = previous_data.get('sections', [])
        
#         if not current_sections or not previous_sections:
#             return current_data
        
#         # Check first section of current page
#         first_section = current_sections[0]
#         last_prev_section = previous_sections[-1]
        
#         # Add continuation information
#         if first_section.get('continues_from_previous'):
#             first_section['continuation_info'] = {
#                 'from_page': previous_data.get('page_number'),
#                 'from_section': last_prev_section.get('header', 'Unknown'),
#                 'from_level': last_prev_section.get('level', 1)
#             }
        
#         # Mark the last section of previous page if it continues
#         if last_prev_section.get('continues_on_next'):
#             if 'continuation_info' not in last_prev_section:
#                 last_prev_section['continuation_info'] = {
#                     'to_page': current_data.get('page_number'),
#                     'to_section': first_section.get('header', 'Unknown')
#                 }
        
#         # SPECIAL HANDLING FOR COURSE DESCRIPTIONS
#         # Check if the last course in previous page has an empty or incomplete description
#         prev_courses = self._extract_course_list(previous_sections)
#         curr_courses = self._extract_course_list(current_sections)
        
#         if prev_courses and curr_courses:
#             last_prev_course = prev_courses[-1]
#             first_curr_course = curr_courses[0]
            
#             # Check if last course from previous page has empty or very short description
#             prev_desc = last_prev_course.get('description', '').strip()
            
#             if len(prev_desc) < 50:  # Likely incomplete or empty
#                 # Check if first section of current page is NOT a course header
#                 # This might be the continuation of the description
#                 first_section_header = first_section.get('header', '')
                
#                 # If first section doesn't look like a course code, it might be continuation
#                 if not self._is_course_header(first_section_header):
#                     # Mark this as a continuation case that needs attention
#                     last_prev_course['continuation_warning'] = {
#                         'status': 'incomplete_description',
#                         'continues_on_page': current_data.get('page_number'),
#                         'message': f"Description appears to continue on page {current_data.get('page_number')}. Manual merge may be needed."
#                     }
                
#                 # Alternative: if first current course has same code, merge descriptions
#                 elif first_curr_course.get('course_code') == last_prev_course.get('course_code'):
#                     # Same course appears on both pages - merge descriptions
#                     curr_desc = first_curr_course.get('description', '').strip()
#                     merged_desc = f"{prev_desc} {curr_desc}".strip()
                    
#                     # Update the previous course's description
#                     last_prev_course['description'] = merged_desc
#                     last_prev_course['merged_from_pages'] = [
#                         previous_data.get('page_number'),
#                         current_data.get('page_number')
#                     ]
                    
#                     # Mark the current course as merged
#                     first_curr_course['merged_into_previous_page'] = True
#                     first_curr_course['merge_note'] = f"This course was merged with the same course from page {previous_data.get('page_number')}"
        
#         return current_data
    
#     def _extract_course_list(self, sections: List[Dict]) -> List[Dict]:
#         """
#         Extract all course entries from sections
        
#         Args:
#             sections: List of section dictionaries
            
#         Returns:
#             List of course dictionaries with course_code, course_title, description
#         """
#         courses = []
        
#         def extract_from_section(section):
#             # Check if this section has course content
#             subsections = section.get('subsections', [])
#             for subsection in subsections:
#                 content = subsection.get('content', {})
                
#                 if isinstance(content, dict):
#                     # Check for course list in table format
#                     if content.get('type') == 'course_list':
#                         courses.extend(content.get('courses', []))
                    
#                     # Check for individual course
#                     elif 'course_code' in content or 'course_title' in content:
#                         courses.append(content)
                
#                 # Recursively check nested subsections
#                 if 'subsections' in subsection:
#                     for nested in subsection['subsections']:
#                         extract_from_section(nested)
        
#         for section in sections:
#             extract_from_section(section)
            
#             # Also check direct content in sections
#             content = section.get('content', {})
#             if isinstance(content, dict):
#                 if content.get('type') == 'course_list':
#                     courses.extend(content.get('courses', []))
#                 elif 'course_code' in content:
#                     courses.append(content)
        
#         return courses
    
#     def _is_course_header(self, header: str) -> bool:
#         """
#         Check if a header looks like a course code/title
        
#         Args:
#             header: Header string to check
            
#         Returns:
#             True if it looks like a course header (e.g., "AGR 323 Mushroom and Molds (3)")
#         """
#         if not header:
#             return False
        
#         # Common patterns for course headers:
#         # "AGR 323 Course Name (3)"
#         # "BIO 101 Introduction to Biology"
#         # Course codes typically have letters followed by numbers
        
#         import re
#         # Pattern: 2-4 letters, space, 2-4 digits
#         course_code_pattern = r'^[A-Z]{2,4}\s+\d{2,4}\b'
        
#         return bool(re.match(course_code_pattern, header.strip()))

#     def extract_from_image(self, image_path: str, page_number: int) -> Dict:
#         """
#         Extract structured data from a single image with enhanced structure detection
        
#         Args:
#             image_path: Path to the PNG image
#             page_number: Page number for reference
            
#         Returns:
#             Dictionary containing extracted data with proper structure
#         """
#         try:
#             # Convert image to base64
#             image_data_url = self.image_to_base64(image_path)
            
#             # Create enhanced prompt
#             prompt = self.create_extraction_prompt(page_number, self.previous_page_context)
            
#             # Call Groq API
#             completion = self.client.chat.completions.create(
#                 model=self.model,
#                 messages=[
#                     {
#                         "role": "user",
#                         "content": [
#                             {
#                                 "type": "text",
#                                 "text": prompt
#                             },
#                             {
#                                 "type": "image_url",
#                                 "image_url": {
#                                     "url": image_data_url
#                                 }
#                             }
#                         ]
#                     }
#                 ],
#                 temperature=0.05,  # Very low for consistent structure
#                 max_completion_tokens=8192,
#                 top_p=0.9,
#                 stream=False
#             )
            
#             # Extract response
#             response_text = completion.choices[0].message.content
            
#             # Parse JSON response with multiple fix attempts
#             extracted_data = self.validate_and_parse_json(response_text, page_number, max_attempts=3)
            
#             # Only validate if parsing succeeded
#             if extracted_data.get('extraction_status') != 'json_parse_failed':
#                 # Validate and enhance structure
#                 extracted_data = self.detect_structure_and_validate(extracted_data, page_number)
                
#                 # Update context for next page
#                 sections = extracted_data.get('sections', [])
#                 if sections:
#                     last_section = sections[-1]
#                     self.previous_page_context = last_section.get('header', '')
                
#                 # Store in history
#                 self.page_history.append({
#                     'page_number': page_number,
#                     'last_section': last_section.get('header', '') if sections else '',
#                     'layout_type': extracted_data.get('layout_type', 'unknown')
#                 })
            
#             return extracted_data
            
#         except Exception as e:
#             return {
#                 "page_number": page_number,
#                 "error": str(e),
#                 "extraction_status": "failed"
#             }
        
#     def extract_page_range(
#         self, 
#         image_dir: str, 
#         start_page: int = 1, 
#         end_page: Optional[int] = None,
#         output_file: Optional[str] = None
#     ) -> List[Dict]:
#         """
#         Extract data from a range of pages with enhanced structure detection
        
#         Args:
#             image_dir: Directory containing page images
#             start_page: Starting page number (1-indexed)
#             end_page: Ending page number (inclusive)
#             output_file: Optional JSON file to save results
            
#         Returns:
#             List of extracted data dictionaries
#         """
#         image_dir = Path(image_dir)
#         all_extractions = []
        
#         # Find all page images
#         page_files = sorted(image_dir.glob("page_*.png"))
        
#         if not page_files:
#             print(f"❌ No page images found in {image_dir}")
#             return []
        
#         # Determine page range
#         total_pages = len(page_files)
#         end_page = end_page or total_pages
        
#         # Validate range
#         if start_page < 1 or start_page > total_pages:
#             print(f"❌ Invalid start_page: {start_page}")
#             return []
        
#         if end_page > total_pages:
#             print(f"⚠️  Using {total_pages} instead of {end_page}")
#             end_page = total_pages
        
#         print(f"\n{'='*60}")
#         print(f"📄 Processing pages {start_page} to {end_page}")
#         print(f"{'='*60}\n")
        
#         # Reset context
#         self.previous_page_context = None
#         self.page_history = []
#         previous_data = None
        
#         # Process each page
#         for i in range(start_page - 1, end_page):
#             page_file = page_files[i]
#             page_num = i + 1
            
#             print(f"📖 Processing page {page_num}/{end_page}...")
            
#             extracted_data = self.extract_from_image(str(page_file), page_num)
            
#             # Merge with previous page if continuation detected
#             if previous_data and extracted_data.get('extraction_status') != 'json_parse_failed':
#                 extracted_data = self.merge_continued_sections(extracted_data, previous_data)
            
#             all_extractions.append(extracted_data)
#             previous_data = extracted_data
            
#             # Show quick summary
#             status = extracted_data.get('extraction_status', 'success')
#             if status == 'json_parse_failed':
#                 print(f"   ❌ JSON parsing failed - raw text saved")
#             else:
#                 layout = extracted_data.get('layout_type', 'unknown')
#                 num_sections = len(extracted_data.get('sections', []))
#                 print(f"   ✅ Success ({layout}, {num_sections} sections)")
        
#         # Save to file
#         if output_file:
#             output_path = Path(output_file)
#             output_path.parent.mkdir(parents=True, exist_ok=True)
            
#             # Save raw extraction
#             with open(output_file, 'w', encoding='utf-8') as f:
#                 json.dump(all_extractions, f, indent=2, ensure_ascii=False)
#             print(f"\n✅ Raw results saved to {output_file}")
            
#             # Generate merged output for easier consumption
#             merged_file = output_path.parent / f"{output_path.stem}_merged{output_path.suffix}"
#             merged_data = self._generate_merged_output(all_extractions)
            
#             with open(merged_file, 'w', encoding='utf-8') as f:
#                 json.dump(merged_data, f, indent=2, ensure_ascii=False)
#             print(f"✅ Merged results saved to {merged_file}")
            
#             # Generate continuation report
#             continuation_report = self._generate_continuation_report(all_extractions)
#             if continuation_report:
#                 report_file = output_path.parent / f"{output_path.stem}_continuation_report.txt"
#                 with open(report_file, 'w', encoding='utf-8') as f:
#                     f.write(continuation_report)
#                 print(f"ℹ️  Continuation report saved to {report_file}")
        
#         # Print summary
#         self._print_extraction_summary(all_extractions)
        
#         return all_extractions
    
#     def _generate_merged_output(self, extractions: List[Dict]) -> Dict:
#         """
#         Generate a merged output where course descriptions are combined across pages
        
#         Args:
#             extractions: List of page extractions
            
#         Returns:
#             Merged data structure
#         """
#         merged = {
#             'metadata': {
#                 'total_pages': len(extractions),
#                 'merge_notes': 'Course descriptions have been merged across page boundaries'
#             },
#             'pages': []
#         }
        
#         for i, page_data in enumerate(extractions):
#             page_copy = page_data.copy()
            
#             # Process courses in this page
#             courses = self._extract_course_list([page_data] if isinstance(page_data, dict) else page_data.get('sections', []))
            
#             # Mark courses that were merged
#             for course in courses:
#                 if 'continuation_warning' in course:
#                     course['_merge_note'] = course['continuation_warning']['message']
                
#                 if 'merged_from_pages' in course:
#                     course['_merge_note'] = f"Description merged from pages {course['merged_from_pages']}"
                
#                 if course.get('merged_into_previous_page'):
#                     course['_merge_note'] = "This course entry was merged into previous page"
            
#             merged['pages'].append(page_copy)
        
#         return merged
    
#     def _generate_continuation_report(self, extractions: List[Dict]) -> str:
#         """
#         Generate a text report of all continuations and potential merge issues
        
#         Args:
#             extractions: List of page extractions
            
#         Returns:
#             Report text or empty string if no issues
#         """
#         issues = []
        
#         for page_data in extractions:
#             page_num = page_data.get('page_number', 'Unknown')
            
#             # Check for courses with continuation warnings
#             courses = self._extract_course_list(page_data.get('sections', []))
            
#             for course in courses:
#                 if 'continuation_warning' in course:
#                     warning = course['continuation_warning']
#                     issues.append({
#                         'page': page_num,
#                         'course_code': course.get('course_code', 'Unknown'),
#                         'course_title': course.get('course_title', 'Unknown'),
#                         'issue': warning.get('message', 'Unknown issue')
#                     })
        
#         if not issues:
#             return ""
        
#         report_lines = [
#             "=" * 80,
#             "CONTINUATION REPORT - MANUAL REVIEW RECOMMENDED",
#             "=" * 80,
#             "",
#             "The following courses have incomplete descriptions that may span across pages:",
#             ""
#         ]
        
#         for i, issue in enumerate(issues, 1):
#             report_lines.extend([
#                 f"{i}. Page {issue['page']}:",
#                 f"   Course: {issue['course_code']} - {issue['course_title']}",
#                 f"   Issue: {issue['issue']}",
#                 ""
#             ])
        
#         report_lines.extend([
#             "=" * 80,
#             "RECOMMENDED ACTIONS:",
#             "1. Review the merged output file for complete course descriptions",
#             "2. Check the continuation_warning fields in the JSON for details",
#             "3. Manually verify courses listed above have complete descriptions",
#             "=" * 80
#         ])
        
#         return "\n".join(report_lines)
    
#     def _print_extraction_summary(self, results: List[Dict]):
#         """Print a detailed summary of extraction results"""
#         print(f"\n{'='*60}")
#         print("📊 EXTRACTION SUMMARY")
#         print(f"{'='*60}\n")
        
#         success_count = 0
#         failed_count = 0
#         warning_count = 0
#         continuation_count = 0
        
#         for result in results:
#             page_num = result.get('page_number', 'Unknown')
#             status = result.get('extraction_status', 'success')
            
#             if status == 'json_parse_failed':
#                 failed_count += 1
#                 print(f"❌ Page {page_num}: Parsing failed")
#                 continue
            
#             success_count += 1
#             layout = result.get('layout_type', 'Unknown')
#             sections = result.get('sections', [])
            
#             # Check for warnings
#             metadata = result.get('extraction_metadata', {})
#             warnings = metadata.get('warnings', [])
            
#             # Check for continuation warnings in courses
#             courses = self._extract_course_list(sections)
#             has_continuation = any('continuation_warning' in c for c in courses)
            
#             warning_indicator = " ⚠️" if warnings or has_continuation else ""
#             print(f"📄 Page {page_num} ({layout} layout){warning_indicator}:")
            
#             if warnings:
#                 warning_count += 1
#                 for warning in warnings:
#                     print(f"   ⚠️  {warning}")
            
#             if has_continuation:
#                 continuation_count += 1
#                 for course in courses:
#                     if 'continuation_warning' in course:
#                         code = course.get('course_code', 'Unknown')
#                         title = course.get('course_title', 'Unknown')
#                         print(f"   🔗 {code} {title}: Description may continue on next page")
            
#             if not sections:
#                 print("   ⚠️  No sections extracted")
#                 continue
            
#             for section in sections:
#                 level = section.get('level', 1)
#                 header = section.get('header', 'No title')
#                 column = section.get('column', '')
#                 indent = "  " * level
                
#                 # Show column info for two-column layouts
#                 column_info = f" [{column}]" if column else ""
                
#                 # Show continuation markers
#                 markers = []
#                 if section.get('continues_from_previous'):
#                     markers.append("⬅️ continues from prev")
#                 if section.get('continues_on_next'):
#                     markers.append("continues to next ➡️")
                
#                 marker_str = f" [{', '.join(markers)}]" if markers else ""
                
#                 print(f"{indent}{'📌' if level == 1 else '└─'} {header}{column_info}{marker_str}")
                
#                 # Show subsections if any
#                 if 'subsections' in section:
#                     for subsection in section['subsections']:
#                         sub_level = subsection.get('level', level + 1)
#                         sub_header = subsection.get('header', 'No title')
#                         sub_indent = "  " * sub_level
                        
#                         # Check if it's a table
#                         content = subsection.get('content', {})
#                         content_type = content.get('type', 'unknown') if isinstance(content, dict) else 'unknown'
#                         type_indicator = " 📊" if content_type == 'table' else ""
                        
#                         print(f"{sub_indent}└─ {sub_header}{type_indicator}")
                        
#                         # Show table row count if it's a table
#                         if content_type == 'table':
#                             rows = content.get('rows', [])
#                             print(f"{sub_indent}   ({len(rows)} rows)")
                
#                 # Show continuation info if available
#                 if 'continuation_info' in section:
#                     info = section['continuation_info']
#                     if 'from_page' in info:
#                         print(f"{indent}   ℹ️  Continues from page {info['from_page']}")
            
#             print()
        
#         print(f"{'='*60}")
#         print(f"✅ Successfully extracted: {success_count} pages")
#         if warning_count > 0:
#             print(f"⚠️  Pages with warnings: {warning_count} (may be incomplete)")
#         if continuation_count > 0:
#             print(f"🔗 Pages with continuations: {continuation_count}")
#             print(f"   → Check the continuation report for details")
#         print(f"❌ Failed to parse: {failed_count} pages")
#         print(f"{'='*60}\n")


# # Example usage
# if __name__ == "__main__":
#     extractor = EnhancedPDFImageExtractor()
    
#     results = extractor.extract_page_range(
#         image_dir="C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\chtn_test_docs\\imgs",
#         start_page=1,
#         end_page=10,
#         output_file="extracted_data_enhanced.json"
#     )
    
#     print("\n✨ Extraction complete!")
#     print(f"📁 Total pages processed: {len(results)}")


📄 Processing pages 1 to 10

📖 Processing page 1/10...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (single_column, 2 sections)
📖 Processing page 2/10...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (single_column, 1 sections)
📖 Processing page 3/10...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (single_column, 1 sections)
📖 Processing page 4/10...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (single_column, 1 sections)
📖 Processing page 5/10...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (two_column, 6 sections)
📖 Processing page 6/10...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting value
   ✅ Success (table_of_contents, 6 sections)
📖 Processing page 7/10...
   ⚠️  Attempt 1/3: JSON error at line 1, col 1
       Error: Expecting va

#### was using below code to extarct data as few things got updated like checkpointing where extraction was paused due to token limit hit problem.

#### Still data extarction is not perfect when data moves from left to right and one page to another. will have to fix this later

In [2]:
import os
import json
import base64
import re
import time
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from groq import Groq
from dotenv import load_dotenv
from datetime import datetime

# Load environment variables
load_dotenv()

class EnhancedPDFImageExtractor:
    def __init__(self, api_key: Optional[str] = None, delay_between_calls: float = 2.0):
        """
        Initialize the PDF Image Extractor with enhanced structure detection
        
        Args:
            api_key: Groq API key (if not provided, loads from env)
            delay_between_calls: Delay in seconds between API calls (default: 2.0)
        """
        self.client = Groq(api_key=api_key or os.getenv('GROQ_API_KEY'))
        self.model = "meta-llama/llama-4-maverick-17b-128e-instruct"
        self.previous_page_context = None
        self.page_history = []  # Store previous pages for better context
        
        # Rate limiting settings
        self.delay_between_calls = delay_between_calls
        self.last_api_call_time = 0
        self.total_tokens_used = 0
        self.daily_token_limit = 500000
        self.token_safety_margin = 0.9  # Stop at 90% of limit
        
        # Checkpointing settings
        self.checkpoint_file = None
        self.checkpoint_data = {
            'completed_pages': [],
            'failed_pages': [],
            'last_processed_page': 0,
            'total_tokens_used': 0,
            'last_checkpoint_time': None
        }
    
    def _wait_for_rate_limit(self):
        """Implement rate limiting by adding delay between API calls"""
        current_time = time.time()
        time_since_last_call = current_time - self.last_api_call_time
        
        if time_since_last_call < self.delay_between_calls:
            sleep_time = self.delay_between_calls - time_since_last_call
            print(f"   ⏳ Rate limiting: waiting {sleep_time:.2f}s...")
            time.sleep(sleep_time)
        
        self.last_api_call_time = time.time()
    
    def _check_token_budget(self, estimated_tokens: int = 10000) -> bool:
        """
        Check if we have enough token budget remaining
        
        Args:
            estimated_tokens: Estimated tokens for next request
            
        Returns:
            True if safe to proceed, False if approaching limit
        """
        safe_limit = self.daily_token_limit * self.token_safety_margin
        
        if self.total_tokens_used + estimated_tokens > safe_limit:
            print(f"\n⚠️  Approaching token limit!")
            print(f"   Used: {self.total_tokens_used:,} / {self.daily_token_limit:,}")
            print(f"   Safety limit: {safe_limit:,}")
            return False
        
        return True
    
    def _parse_retry_time(self, error_message: str) -> Optional[float]:
        """
        Parse retry time from rate limit error message
        
        Args:
            error_message: Error message from API
            
        Returns:
            Wait time in seconds, or None if not found
        """
        # Pattern: "Please try again in 6m13.7664s"
        match = re.search(r'try again in (\d+)m([\d.]+)s', error_message)
        if match:
            minutes = int(match.group(1))
            seconds = float(match.group(2))
            return minutes * 60 + seconds
        
        # Pattern: "Please try again in 45.5s"
        match = re.search(r'try again in ([\d.]+)s', error_message)
        if match:
            return float(match.group(1))
        
        return None
    
    def _extract_with_retry(
        self, 
        image_path: str, 
        page_number: int,
        max_retries: int = 3,
        base_wait_time: float = 60.0
    ) -> Tuple[Dict, bool]:
        """
        Extract data with exponential backoff retry logic
        
        Args:
            image_path: Path to image file
            page_number: Page number
            max_retries: Maximum number of retry attempts
            base_wait_time: Base wait time for exponential backoff
            
        Returns:
            Tuple of (extracted_data, success_flag)
        """
        for attempt in range(max_retries + 1):
            try:
                # Check token budget before attempting
                if not self._check_token_budget():
                    return {
                        "page_number": page_number,
                        "error": "Token budget limit reached for safety",
                        "extraction_status": "skipped_budget_limit"
                    }, False
                
                # Apply rate limiting
                self._wait_for_rate_limit()
                
                # Attempt extraction
                result = self._extract_from_image_internal(image_path, page_number)
                
                # Track token usage if available
                if 'usage' in result:
                    tokens_used = result['usage'].get('total_tokens', 0)
                    self.total_tokens_used += tokens_used
                    print(f"   📊 Tokens used: {tokens_used:,} (Total: {self.total_tokens_used:,})")
                
                return result, True
                
            except Exception as e:
                error_str = str(e)
                
                # Check if it's a rate limit error
                if 'rate_limit_exceeded' in error_str or 'Error code: 429' in error_str:
                    retry_time = self._parse_retry_time(error_str)
                    
                    if attempt < max_retries:
                        if retry_time:
                            wait_time = retry_time + 5  # Add 5 second buffer
                            print(f"   ⚠️  Rate limit hit! Waiting {wait_time:.0f}s (attempt {attempt + 1}/{max_retries})...")
                        else:
                            # Exponential backoff: 60s, 120s, 240s
                            wait_time = base_wait_time * (2 ** attempt)
                            print(f"   ⚠️  Rate limit hit! Exponential backoff: {wait_time:.0f}s (attempt {attempt + 1}/{max_retries})...")
                        
                        time.sleep(wait_time)
                        continue
                    else:
                        print(f"   ❌ Max retries reached for page {page_number}")
                        return {
                            "page_number": page_number,
                            "error": error_str,
                            "extraction_status": "failed_rate_limit",
                            "retry_attempts": max_retries
                        }, False
                else:
                    # Non-rate-limit error
                    if attempt < max_retries:
                        wait_time = 5 * (attempt + 1)  # 5s, 10s, 15s
                        print(f"   ⚠️  Error occurred, retrying in {wait_time}s... (attempt {attempt + 1}/{max_retries})")
                        time.sleep(wait_time)
                        continue
                    else:
                        return {
                            "page_number": page_number,
                            "error": error_str,
                            "extraction_status": "failed",
                            "retry_attempts": max_retries
                        }, False
        
        # Should not reach here
        return {
            "page_number": page_number,
            "error": "Unknown error in retry logic",
            "extraction_status": "failed"
        }, False
    
    def _load_checkpoint(self, checkpoint_file: str) -> bool:
        """
        Load checkpoint data from file
        
        Args:
            checkpoint_file: Path to checkpoint file
            
        Returns:
            True if checkpoint loaded successfully
        """
        checkpoint_path = Path(checkpoint_file)
        
        if not checkpoint_path.exists():
            print(f"ℹ️  No checkpoint found at {checkpoint_file}, starting fresh")
            return False
        
        try:
            with open(checkpoint_path, 'r', encoding='utf-8') as f:
                self.checkpoint_data = json.load(f)
            
            self.total_tokens_used = self.checkpoint_data.get('total_tokens_used', 0)
            
            print(f"✅ Checkpoint loaded from {checkpoint_file}")
            print(f"   Last processed page: {self.checkpoint_data['last_processed_page']}")
            print(f"   Completed pages: {len(self.checkpoint_data['completed_pages'])}")
            print(f"   Failed pages: {len(self.checkpoint_data['failed_pages'])}")
            print(f"   Tokens used: {self.total_tokens_used:,}")
            
            return True
            
        except Exception as e:
            print(f"⚠️  Failed to load checkpoint: {e}")
            return False
    
    def _save_checkpoint(self):
        """Save current progress to checkpoint file"""
        if not self.checkpoint_file:
            return
        
        self.checkpoint_data['total_tokens_used'] = self.total_tokens_used
        self.checkpoint_data['last_checkpoint_time'] = datetime.now().isoformat()
        
        checkpoint_path = Path(self.checkpoint_file)
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        
        try:
            with open(checkpoint_path, 'w', encoding='utf-8') as f:
                json.dump(self.checkpoint_data, f, indent=2)
            
            print(f"   💾 Checkpoint saved (Page {self.checkpoint_data['last_processed_page']})")
            
        except Exception as e:
            print(f"   ⚠️  Failed to save checkpoint: {e}")
    
    def _save_page_result(self, page_data: Dict, output_dir: str):
        """
        Save individual page result incrementally
        
        Args:
            page_data: Extracted page data
            output_dir: Output directory for page files
        """
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        
        page_num = page_data.get('page_number', 'unknown')
        page_file = output_path / f"page_{page_num:04d}.json"
        
        try:
            with open(page_file, 'w', encoding='utf-8') as f:
                json.dump(page_data, f, indent=2, ensure_ascii=False)
        except Exception as e:
            print(f"   ⚠️  Failed to save page {page_num}: {e}")
    
    def image_to_base64(self, image_path: str) -> str:
        """Convert image file to base64 data URL"""
        with open(image_path, 'rb') as image_file:
            encoded = base64.b64encode(image_file.read()).decode('utf-8')
            return f"data:image/png;base64,{encoded}"
    
    def create_extraction_prompt(self, page_number: int, previous_context: Optional[str] = None) -> str:
        """
        Create an enhanced prompt for automatic structure detection and extraction
        
        Args:
            page_number: Current page number
            previous_context: Context from previous page
            
        Returns:
            Formatted prompt string
        """
        context_info = ""
        if previous_context:
            context_info = f"""
PREVIOUS PAGE CONTEXT:
The previous page ended with: "{previous_context}"
If this page continues from the previous content, mark it appropriately.
"""

        prompt = f"""Analyze this PDF page (Page {page_number}) and extract ALL content with proper structure detection.

{context_info}

CRITICAL: EXTRACT EVERYTHING - START FROM THE VERY TOP
- Look at the TOP of the page first
- Check for headers, titles, or table headings at the very top
- Read from TOP to BOTTOM - do not skip any content
- For tables: extract ALL rows including headers
- For degree programs: extract the complete table with Program Requirements, Credits, etc.

STEP 1: AUTOMATIC LAYOUT DETECTION
First, identify the page layout:
- "single_column": Content flows top to bottom in one column
- "two_column": Content in LEFT column (top to bottom), then RIGHT column (top to bottom)
- "table_of_contents": List items with page numbers
- "mixed": Combination of layouts (describe the flow)

STEP 2: CONTENT STRUCTURE DETECTION
Identify all structural elements:
- **Main Headers**: Large, bold text that starts a new major section
- **Sub-headers**: Medium text under main headers
- **Sub-sub-headers**: Smaller headers under sub-headers
- **Paragraphs**: Body text content
- **Lists**: Bulleted or numbered items
- **Special formatting**: Italic text, bold emphasis, quotes

STEP 3: READING ORDER (CRITICAL FOR TWO-COLUMN LAYOUTS)
For two-column pages:
1. Read ENTIRE LEFT column from TOP to BOTTOM
2. Then read ENTIRE RIGHT column from TOP to BOTTOM
3. NEVER jump between columns mid-section

STEP 4: OUTPUT FORMAT

CRITICAL: Return VALID JSON with proper syntax:
- ALWAYS use commas after each key-value pair (except the last one in an object)
- ALWAYS use colons after keys
- Use double quotes for strings
- Example: {{"key1": "value1", "key2": "value2"}}

For GENERAL CONTENT (paragraphs, regular text):
{{
  "page_number": {page_number},
  "layout_type": "single_column",
  "reading_order": "top_to_bottom",
  "sections": [
    {{
      "level": 1,
      "header": "Main Section Header",
      "header_type": "main_header",
      "continues_from_previous": false,
      "continues_on_next": false,
      "subsections": [
        {{
          "level": 2,
          "header": "Sub-header (if any)",
          "header_type": "sub_header",
          "content": {{
            "type": "paragraph",
            "text": "Complete paragraph text...",
            "formatting": []
          }}
        }}
      ]
    }}
  ]
}}

Return ONLY valid JSON with NO comments, NO trailing commas, and proper structure."""
        return prompt

    def fix_json_syntax(self, json_text: str) -> str:
        """
        Fix common JSON syntax errors from LLM output
        
        Args:
            json_text: Raw JSON text with potential syntax errors
            
        Returns:
            Fixed JSON text
        """
        # Remove any markdown code blocks
        if "```json" in json_text:
            json_start = json_text.find("```json") + 7
            json_end = json_text.find("```", json_start)
            json_text = json_text[json_start:json_end].strip()
        elif "```" in json_text:
            json_start = json_text.find("```") + 3
            json_end = json_text.find("```", json_start)
            json_text = json_text[json_start:json_end].strip()
        
        # Remove comments (lines starting with # or //)
        lines = []
        for line in json_text.split('\n'):
            # Skip comment lines
            stripped = line.strip()
            if stripped.startswith('#') or stripped.startswith('//'):
                continue
            
            # Remove inline comments
            if '#' in line or '//' in line:
                in_string = False
                cleaned = []
                i = 0
                while i < len(line):
                    char = line[i]
                    
                    # Track if we're inside a string
                    if char == '"' and (i == 0 or line[i-1] != '\\'):
                        in_string = not in_string
                    
                    # Stop at comment if not in string
                    if not in_string:
                        if char == '#':
                            break
                        if char == '/' and i < len(line) - 1 and line[i+1] == '/':
                            break
                    
                    cleaned.append(char)
                    i += 1
                
                line = ''.join(cleaned).rstrip()
            
            if line.strip():
                lines.append(line)
        
        json_text = '\n'.join(lines)
        
        # Fix missing commas between key-value pairs
        json_text = re.sub(r'("\s*)\n(\s*"[^"]+"\s*:)', r'\1,\n\2', json_text)
        json_text = re.sub(r'(\})\n(\s*\{)', r'\1,\n\2', json_text)
        json_text = re.sub(r'(\])\n(\s*\{)', r'\1,\n\2', json_text)
        
        # Fix missing colons
        json_text = re.sub(r'"([^"]+)"\s+"', r'"\1": "', json_text)
        json_text = re.sub(r'"([^"]+)"\s+(\d+)', r'"\1": \2', json_text)
        json_text = re.sub(r'"([^"]+)"\s+(true|false|null)', r'"\1": \2', json_text)
        json_text = re.sub(r'"([^"]+)"\s+(\[|\{)', r'"\1": \2', json_text)
        
        # Remove trailing commas before closing braces/brackets
        json_text = re.sub(r',(\s*[\}\]])', r'\1', json_text)
        
        # Ensure closing braces if incomplete
        open_braces = json_text.count('{') - json_text.count('}')
        open_brackets = json_text.count('[') - json_text.count(']')
        
        if open_brackets > 0:
            for _ in range(open_brackets):
                json_text += '\n]'
        
        if open_braces > 0:
            for _ in range(open_braces):
                json_text += '\n}'
        
        return json_text
    
    def validate_and_parse_json(self, json_text: str, page_number: int, max_attempts: int = 3) -> Dict:
        """
        Attempt to parse JSON with multiple fix strategies
        
        Args:
            json_text: Raw JSON text
            page_number: Page number for error reporting
            max_attempts: Maximum number of fix attempts
            
        Returns:
            Parsed JSON dictionary or error structure
        """
        original_text = json_text
        
        for attempt in range(max_attempts):
            try:
                # Try to parse
                data = json.loads(json_text)
                return data
            
            except json.JSONDecodeError as e:
                print(f"   ⚠️  Attempt {attempt + 1}/{max_attempts}: JSON error at line {e.lineno}, col {e.colno}")
                print(f"       Error: {e.msg}")
                
                if attempt < max_attempts - 1:
                    # Try to fix the JSON
                    json_text = self.fix_json_syntax(json_text)
                else:
                    # Last attempt failed, return error structure
                    print(f"   ❌ Failed to parse JSON after {max_attempts} attempts")
                    print(f"   📄 Saving raw text for manual inspection")
                    
                    return {
                        "page_number": page_number,
                        "layout_type": "unknown",
                        "raw_text": original_text,
                        "extraction_status": "json_parse_failed",
                        "parse_error": f"{e.msg} at line {e.lineno}",
                        "fix_attempts": max_attempts
                    }

    def detect_structure_and_validate(self, extracted_data: Dict, page_number: int) -> Dict:
        """
        Validate and enhance the extracted structure
        
        Args:
            extracted_data: Raw extracted data from LLM
            page_number: Current page number
            
        Returns:
            Validated and enhanced data structure
        """
        if not extracted_data or 'sections' not in extracted_data:
            return extracted_data
        
        # Add metadata
        extracted_data['extraction_metadata'] = {
            'page_number': page_number,
            'validation_status': 'validated',
            'total_sections': len(extracted_data.get('sections', [])),
            'layout_detected': extracted_data.get('layout_type', 'unknown')
        }
        
        sections = extracted_data.get('sections', [])
        
        for section in sections:
            # Ensure level is set
            if 'level' not in section:
                section['level'] = 1
            
            # Ensure header is present
            if 'header' not in section:
                section['header'] = 'Untitled Section'
            
            # Validate content structure
            if 'content' in section:
                content = section['content']
                if isinstance(content, str):
                    # Convert simple string to structured format
                    section['content'] = {
                        'type': 'paragraph',
                        'text': content
                    }
            
            # Process subsections recursively
            if 'subsections' in section:
                for subsection in section['subsections']:
                    if 'level' not in subsection:
                        subsection['level'] = section.get('level', 1) + 1
        
        return extracted_data

    def _extract_from_image_internal(self, image_path: str, page_number: int) -> Dict:
        """
        Internal method to extract structured data from a single image
        
        Args:
            image_path: Path to the PNG image
            page_number: Page number for reference
            
        Returns:
            Dictionary containing extracted data with proper structure
        """
        # Convert image to base64
        image_data_url = self.image_to_base64(image_path)
        
        # Create enhanced prompt
        prompt = self.create_extraction_prompt(page_number, self.previous_page_context)
        
        # Call Groq API
        completion = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": prompt
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": image_data_url
                            }
                        }
                    ]
                }
            ],
            temperature=0.05,
            max_completion_tokens=8192,
            top_p=0.9,
            stream=False
        )
        
        # Extract response
        response_text = completion.choices[0].message.content
        
        # Parse JSON response with multiple fix attempts
        extracted_data = self.validate_and_parse_json(response_text, page_number, max_attempts=3)
        
        # Add usage info if available
        if hasattr(completion, 'usage') and completion.usage:
            extracted_data['usage'] = {
                'prompt_tokens': completion.usage.prompt_tokens,
                'completion_tokens': completion.usage.completion_tokens,
                'total_tokens': completion.usage.total_tokens
            }
        
        # Only validate if parsing succeeded
        if extracted_data.get('extraction_status') != 'json_parse_failed':
            # Validate and enhance structure
            extracted_data = self.detect_structure_and_validate(extracted_data, page_number)
            
            # Update context for next page
            sections = extracted_data.get('sections', [])
            if sections:
                last_section = sections[-1]
                self.previous_page_context = last_section.get('header', '')
            
            # Store in history
            self.page_history.append({
                'page_number': page_number,
                'last_section': last_section.get('header', '') if sections else '',
                'layout_type': extracted_data.get('layout_type', 'unknown')
            })
        
        return extracted_data

    def extract_from_image(self, image_path: str, page_number: int) -> Dict:
        """
        Extract structured data from a single image with retry logic
        
        Args:
            image_path: Path to the PNG image
            page_number: Page number for reference
            
        Returns:
            Dictionary containing extracted data with proper structure
        """
        result, success = self._extract_with_retry(image_path, page_number)
        return result
        
    def extract_page_range(
        self, 
        image_dir: str, 
        start_page: int = 1, 
        end_page: Optional[int] = None,
        output_file: Optional[str] = None,
        enable_checkpointing: bool = True,
        checkpoint_interval: int = 1,
        resume: bool = False
    ) -> List[Dict]:
        """
        Extract data from a range of pages with enhanced features
        
        Args:
            image_dir: Directory containing page images
            start_page: Starting page number (1-indexed)
            end_page: Ending page number (inclusive)
            output_file: Optional JSON file to save results
            enable_checkpointing: Enable checkpoint saving
            checkpoint_interval: Save checkpoint every N pages
            resume: Resume from checkpoint if available
            
        Returns:
            List of extracted data dictionaries
        """
        image_dir = Path(image_dir)
        all_extractions = []
        
        # Setup checkpointing
        if enable_checkpointing and output_file:
            output_path = Path(output_file)
            self.checkpoint_file = str(output_path.parent / f"{output_path.stem}_checkpoint.json")
            
            if resume:
                self._load_checkpoint(self.checkpoint_file)
        
        # Find all page images
        page_files = sorted(image_dir.glob("page_*.png"))
        
        if not page_files:
            print(f"❌ No page images found in {image_dir}")
            return []
        
        # Determine page range
        total_pages = len(page_files)
        end_page = end_page or total_pages
        
        # Validate range
        if start_page < 1 or start_page > total_pages:
            print(f"❌ Invalid start_page: {start_page}")
            return []
        
        if end_page > total_pages:
            print(f"⚠️  Using {total_pages} instead of {end_page}")
            end_page = total_pages
        
        # Adjust start page if resuming
        if resume and self.checkpoint_data['last_processed_page'] > 0:
            resume_page = self.checkpoint_data['last_processed_page'] + 1
            if resume_page > start_page and resume_page <= end_page:
                print(f"🔄 Resuming from page {resume_page}")
                start_page = resume_page
                
                # Load completed pages
                if output_file:
                    output_path = Path(output_file)
                    pages_dir = output_path.parent / f"{output_path.stem}_pages"
                    
                    for page_num in self.checkpoint_data['completed_pages']:
                        page_file = pages_dir / f"page_{page_num:04d}.json"
                        if page_file.exists():
                            with open(page_file, 'r', encoding='utf-8') as f:
                                all_extractions.append(json.load(f))
        
        print(f"\n{'='*60}")
        print(f"📄 Processing pages {start_page} to {end_page}")
        print(f"⏱️  Rate limit delay: {self.delay_between_calls}s between calls")
        print(f"💾 Checkpointing: {'Enabled' if enable_checkpointing else 'Disabled'}")
        print(f"📊 Token budget: {self.total_tokens_used:,} / {self.daily_token_limit:,} used")
        print(f"{'='*60}\n")
        
        # Reset context if not resuming
        if not resume or start_page == 1:
            self.previous_page_context = None
            self.page_history = []
        
        previous_data = None
        pages_processed = 0
        
        # Setup page output directory
        if output_file:
            output_path = Path(output_file)
            pages_dir = output_path.parent / f"{output_path.stem}_pages"
        else:
            pages_dir = None
        
        # Process each page
        for i in range(start_page - 1, end_page):
            page_file = page_files[i]
            page_num = i + 1
            
            # Skip if already completed
            if page_num in self.checkpoint_data['completed_pages']:
                print(f"⏭️  Skipping page {page_num} (already completed)")
                continue
            
            print(f"📖 Processing page {page_num}/{end_page}...")
            
            extracted_data, success = self._extract_with_retry(str(page_file), page_num)
            
            all_extractions.append(extracted_data)
            
            # Update checkpoint data
            if success:
                self.checkpoint_data['completed_pages'].append(page_num)
                status = extracted_data.get('extraction_status', 'success')
                
                if status == 'json_parse_failed':
                    print(f"   ❌ JSON parsing failed - raw text saved")
                    self.checkpoint_data['failed_pages'].append(page_num)
                else:
                    layout = extracted_data.get('layout_type', 'unknown')
                    num_sections = len(extracted_data.get('sections', []))
                    print(f"   ✅ Success ({layout}, {num_sections} sections)")
            else:
                self.checkpoint_data['failed_pages'].append(page_num)
                print(f"   ❌ Extraction failed")
            
            self.checkpoint_data['last_processed_page'] = page_num
            
            # Save individual page result
            if pages_dir:
                self._save_page_result(extracted_data, str(pages_dir))
            
            previous_data = extracted_data
            pages_processed += 1
            
            # Save checkpoint periodically
            if enable_checkpointing and pages_processed % checkpoint_interval == 0:
                self._save_checkpoint()
            
            # Check if we should stop due to token budget
            if not self._check_token_budget():
                print(f"\n⚠️  Stopping at page {page_num} due to token budget limit")
                print(f"   You can resume later using resume=True")
                break
        
        # Final checkpoint save
        if enable_checkpointing:
            self._save_checkpoint()
        
        # Save complete results
        if output_file:
            output_path = Path(output_file)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(all_extractions, f, indent=2, ensure_ascii=False)
            print(f"\n✅ Complete results saved to {output_file}")
        
        # Print summary
        self._print_extraction_summary(all_extractions, start_page, end_page)
        
        return all_extractions
    
    def _print_extraction_summary(self, extractions: List[Dict], start_page: int, end_page: int):
        """
        Print a summary of the extraction process
        
        Args:
            extractions: List of extracted page data
            start_page: Starting page number
            end_page: Ending page number
        """
        print(f"\n{'='*60}")
        print(f"📊 EXTRACTION SUMMARY")
        print(f"{'='*60}")
        print(f"Pages processed: {len(extractions)}")
        print(f"Page range: {start_page} to {end_page}")
        
        # Count success/failures
        successful = 0
        json_failed = 0
        rate_limited = 0
        other_failed = 0
        skipped_budget = 0
        
        for extraction in extractions:
            status = extraction.get('extraction_status', 'success')
            if status == 'success' or 'sections' in extraction:
                successful += 1
            elif status == 'json_parse_failed':
                json_failed += 1
            elif status == 'failed_rate_limit':
                rate_limited += 1
            elif status == 'skipped_budget_limit':
                skipped_budget += 1
            else:
                other_failed += 1
        
        print(f"\n✅ Successful: {successful}")
        if json_failed > 0:
            print(f"⚠️  JSON parse failed: {json_failed}")
        if rate_limited > 0:
            print(f"⏱️  Rate limited: {rate_limited}")
        if skipped_budget > 0:
            print(f"💰 Skipped (budget): {skipped_budget}")
        if other_failed > 0:
            print(f"❌ Other failures: {other_failed}")
        
        print(f"\n📊 Total tokens used: {self.total_tokens_used:,}")
        print(f"💾 Completed pages: {len(self.checkpoint_data['completed_pages'])}")
        print(f"❌ Failed pages: {self.checkpoint_data['failed_pages']}")
        print(f"{'='*60}\n")


# Example usage function
def main():
    """
    Example usage of the Enhanced PDF Image Extractor
    """
    # Initialize extractor
    extractor = EnhancedPDFImageExtractor(
        delay_between_calls=6.0  # 6 seconds between API calls
    )
    
    # Extract from page range
    results = extractor.extract_page_range(
        image_dir="C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\chtn_test_docs\\imgs",
        start_page=1,
        end_page=101,
        output_file="extracted_data_enhanced.json",
        enable_checkpointing=True,
        checkpoint_interval=5,  # Save checkpoint every 5 pages
        resume=True  # Resume from checkpoint if available
    )
    
    print(f"\n✅ Extraction complete! Processed {len(results)} pages.")


if __name__ == "__main__":
    main()

✅ Checkpoint loaded from extracted_data_enhanced_checkpoint.json
   Last processed page: 89
   Completed pages: 89
   Failed pages: 0
   Tokens used: 443,643
🔄 Resuming from page 90

📄 Processing pages 90 to 101
⏱️  Rate limit delay: 6.0s between calls
💾 Checkpointing: Enabled
📊 Token budget: 443,643 / 500,000 used

📖 Processing page 90/101...

⚠️  Approaching token limit!
   Used: 443,643 / 500,000
   Safety limit: 450,000.0
   ❌ Extraction failed

⚠️  Approaching token limit!
   Used: 443,643 / 500,000
   Safety limit: 450,000.0

⚠️  Stopping at page 90 due to token budget limit
   You can resume later using resume=True
   💾 Checkpoint saved (Page 90)

✅ Complete results saved to extracted_data_enhanced.json

📊 EXTRACTION SUMMARY
Pages processed: 90
Page range: 90 to 101

✅ Successful: 89
💰 Skipped (budget): 1

📊 Total tokens used: 443,643
💾 Completed pages: 89
❌ Failed pages: [90]


✅ Extraction complete! Processed 90 pages.


##### Chunking steps. Part 1 , some of the data below code was skipping thats why i commented that.

In [ ]:
# import json
# import re
# from typing import List, Dict, Any, Optional
# from datetime import datetime

# class CourseDataProcessor:
#     """
#     Processes structured course data JSON into embedding-ready chunks
#     with rich metadata for hybrid search capabilities.
#     """
    
#     def __init__(self):
#         self.chunks = []
#         self.chunk_id_counter = 0
        
#     def generate_chunk_id(self) -> str:
#         """Generate unique chunk ID"""
#         self.chunk_id_counter += 1
#         return f"chunk_{self.chunk_id_counter:06d}"
    
#     def clean_text(self, text: str) -> str:
#         """Clean and normalize text"""
#         if not text:
#             return ""
#         # Remove extra whitespace
#         text = re.sub(r'\s+', ' ', text)
#         # Remove special characters but keep basic punctuation
#         text = re.sub(r'[^\w\s.,;:()\-/&]', '', text)
#         return text.strip()
    
#     def extract_prerequisites(self, description: str) -> List[str]:
#         """Extract prerequisite courses from description"""
#         prereqs = []
#         # Match patterns like "Prerequisite: ACC 100" or "Prerequisites: ACC 200, BUS 265"
#         prereq_pattern = r'Prerequisite[s]?:\s*([^.]+)'
#         matches = re.findall(prereq_pattern, description, re.IGNORECASE)
        
#         for match in matches:
#             # Extract course codes (e.g., ACC 100, BIO 501)
#             courses = re.findall(r'[A-Z]{2,4}\s+\d{3}', match)
#             prereqs.extend(courses)
        
#         return list(set(prereqs))  # Remove duplicates
    
#     def determine_course_level(self, course_code: str) -> str:
#         """Determine course level from code"""
#         if not course_code:
#             return "unknown"
        
#         match = re.search(r'(\d{3})', course_code)
#         if match:
#             level_num = int(match.group(1)[0])
#             if level_num <= 1:
#                 return "introductory"
#             elif level_num <= 2:
#                 return "intermediate"
#             elif level_num <= 4:
#                 return "advanced"
#             else:
#                 return "graduate"
#         return "unknown"
    
#     def create_course_chunk(self, course: Dict, section_header: str, 
#                            page_number: int, column: str = "") -> Dict[str, Any]:
#         """Create a chunk from course description with rich metadata"""
        
#         code = course.get('code', '')
#         title = course.get('title', '')
#         credits = course.get('credits', '')
#         description = course.get('description', '')
        
#         # Extract prerequisites
#         prerequisites = self.extract_prerequisites(description)
        
#         # Determine course level
#         course_level = self.determine_course_level(code)
        
#         # Extract course prefix (e.g., ACC from ACC 100)
#         course_prefix = code.split()[0] if code else ""
        
#         # Create rich, searchable text
#         text_parts = []
        
#         # Add section context
#         if section_header:
#             text_parts.append(f"Department: {section_header}")
        
#         # Add course information
#         text_parts.append(f"Course Code: {code}")
#         text_parts.append(f"Course Title: {title}")
#         text_parts.append(f"Credits: {credits}")
        
#         # Add prerequisites if any
#         if prerequisites:
#             text_parts.append(f"Prerequisites: {', '.join(prerequisites)}")
#         else:
#             text_parts.append("Prerequisites: None")
        
#         # Add description
#         text_parts.append(f"Description: {description}")
        
#         # Combine into searchable text
#         searchable_text = " | ".join(text_parts)
        
#         # Create metadata
#         metadata = {
#             "course_code": code,
#             "course_title": title,
#             "course_prefix": course_prefix,
#             "credits": credits,
#             "prerequisites": prerequisites,
#             "has_prerequisites": len(prerequisites) > 0,
#             "course_level": course_level,
#             "department": section_header,
#             "page_number": page_number,
#             "column": column,
#             "chunk_type": "course_description",
#             "keywords": self.extract_keywords(f"{title} {description}"),
#         }
        
#         return {
#             "chunk_id": self.generate_chunk_id(),
#             "text": self.clean_text(searchable_text),
#             "metadata": metadata,
#             "timestamp": datetime.now().isoformat()
#         }
    
#     def create_program_chunk(self, header: str, content: str, 
#                             page_number: int, column: str = "") -> Dict[str, Any]:
#         """Create a chunk from program description"""
        
#         # Determine if it's a degree program
#         is_degree = any(word in header.lower() for word in 
#                        ['bachelor', 'master', 'degree', 'b.s.', 'm.s.'])
        
#         # Create searchable text
#         text_parts = [
#             f"Program: {header}",
#             f"Description: {content}"
#         ]
        
#         searchable_text = " | ".join(text_parts)
        
#         metadata = {
#             "program_name": header,
#             "is_degree_program": is_degree,
#             "page_number": page_number,
#             "column": column,
#             "chunk_type": "program_description",
#             "keywords": self.extract_keywords(f"{header} {content}"),
#         }
        
#         return {
#             "chunk_id": self.generate_chunk_id(),
#             "text": self.clean_text(searchable_text),
#             "metadata": metadata,
#             "timestamp": datetime.now().isoformat()
#         }
    
#     def create_semester_chunk(self, semester: str, courses: List[Dict], 
#                              program_context: str, page_number: int) -> Dict[str, Any]:
#         """Create a chunk from semester requirements"""
        
#         # Extract course information
#         course_list = []
#         total_credits = 0
        
#         for row in courses:
#             course = row.get('course', '')
#             credits = row.get('credits', '0')
            
#             if course and 'total' not in course.lower():
#                 course_list.append(f"{course} ({credits} credits)")
#                 try:
#                     total_credits += int(credits)
#                 except:
#                     pass
        
#         # Create searchable text
#         text_parts = [
#             f"Program: {program_context}" if program_context else "",
#             f"Semester: {semester}",
#             f"Required Courses: {'; '.join(course_list)}",
#             f"Total Credits: {total_credits}"
#         ]
        
#         searchable_text = " | ".join([p for p in text_parts if p])
        
#         # Extract course codes
#         course_codes = re.findall(r'[A-Z]{2,4}\s+\d{3}', ' '.join(course_list))
        
#         metadata = {
#             "semester": semester,
#             "program": program_context,
#             "courses": course_codes,
#             "total_credits": total_credits,
#             "course_count": len(course_list),
#             "page_number": page_number,
#             "chunk_type": "semester_requirements",
#             "keywords": self.extract_keywords(searchable_text),
#         }
        
#         return {
#             "chunk_id": self.generate_chunk_id(),
#             "text": self.clean_text(searchable_text),
#             "metadata": metadata,
#             "timestamp": datetime.now().isoformat()
#         }
    
#     def create_course_prefix_chunk(self, course_title: str, course_code: str, 
#                                    page_number: int) -> Dict[str, Any]:
#         """Create a chunk for course prefix mapping (e.g., ACC = Accounting)"""
        
#         text = f"Course Prefix: {course_code} | Department: {course_title} | {course_code} courses are in the {course_title} department"
        
#         metadata = {
#             "course_prefix": course_code,
#             "department_name": course_title,
#             "page_number": page_number,
#             "chunk_type": "course_prefix_mapping",
#             "keywords": [course_code.lower(), course_title.lower()],
#         }
        
#         return {
#             "chunk_id": self.generate_chunk_id(),
#             "text": self.clean_text(text),
#             "metadata": metadata,
#             "timestamp": datetime.now().isoformat()
#         }
    
#     def extract_keywords(self, text: str, max_keywords: int = 10) -> List[str]:
#         """Extract important keywords from text"""
#         # Remove common words
#         stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 
#                      'to', 'for', 'of', 'with', 'by', 'from', 'this', 'that',
#                      'is', 'are', 'was', 'were', 'be', 'been', 'being'}
        
#         # Extract words
#         words = re.findall(r'\b[a-z]{3,}\b', text.lower())
        
#         # Filter and count
#         keywords = [w for w in words if w not in stop_words]
        
#         # Get unique keywords (preserve order, limit count)
#         seen = set()
#         unique_keywords = []
#         for k in keywords:
#             if k not in seen and len(unique_keywords) < max_keywords:
#                 seen.add(k)
#                 unique_keywords.append(k)
        
#         return unique_keywords
    
#     def process_section(self, section: Dict, page_number: int, 
#                        parent_header: str = "", column: str = "") -> None:
#         """Process a section and its subsections recursively"""
        
#         header = section.get('header', '')
#         current_context = header if header else parent_header
        
#         # Process content
#         content = section.get('content', {})
#         content_type = content.get('type', '')
        
#         # Handle paragraph content (program descriptions)
#         if content_type == 'paragraph':
#             text = content.get('text', '')
#             if text and len(text) > 50:  # Only create chunk if substantial
#                 chunk = self.create_program_chunk(
#                     current_context, text, page_number, column
#                 )
#                 self.chunks.append(chunk)
        
#         # Handle course descriptions
#         elif content_type == 'multi_course_description':
#             courses = content.get('courses', [])
#             for course in courses:
#                 chunk = self.create_course_chunk(
#                     course, current_context, page_number, column
#                 )
#                 self.chunks.append(chunk)
        
#         # Handle tables (semester requirements, course prefix mappings)
#         elif content_type == 'table':
#             headers = content.get('headers', [])
#             rows = content.get('rows', [])
            
#             # Check if it's a course prefix table
#             if 'Course Title' in headers and 'Course Code' in headers:
#                 for row in rows:
#                     course_title = row.get('course_title', '')
#                     course_code = row.get('course_code', '')
#                     if course_title and course_code:
#                         chunk = self.create_course_prefix_chunk(
#                             course_title, course_code, page_number
#                         )
#                         self.chunks.append(chunk)
            
#             # Check if it's a semester requirements table
#             elif 'semester' in header.lower() and rows:
#                 chunk = self.create_semester_chunk(
#                     header, rows, parent_header, page_number
#                 )
#                 self.chunks.append(chunk)
        
#         # Process subsections recursively
#         subsections = section.get('subsections', [])
#         for subsection in subsections:
#             self.process_section(subsection, page_number, current_context, column)
    
#     def process_page(self, page: Dict) -> None:
#         """Process a single page"""
#         page_number = page.get('page_number', 0)
#         sections = page.get('sections', [])
        
#         for section in sections:
#             column = section.get('column', '')
#             self.process_section(section, page_number, column=column)
    
#     def process_json(self, json_data: Dict) -> List[Dict[str, Any]]:
#         """Main processing function"""
#         self.chunks = []
#         self.chunk_id_counter = 0
        
#         pages = json_data.get('pages', [])
        
#         for page in pages:
#             self.process_page(page)
        
#         return self.chunks
    
#     def save_chunks(self, chunks: List[Dict], output_file: str = 'embedding_chunks.json'):
#         """Save processed chunks to JSON file"""
#         output_data = {
#             "total_chunks": len(chunks),
#             "processed_at": datetime.now().isoformat(),
#             "chunks": chunks,
#             "statistics": self.get_statistics(chunks)
#         }
        
#         with open(output_file, 'w', encoding='utf-8') as f:
#             json.dump(output_data, f, indent=2, ensure_ascii=False)
        
#         print(f"✓ Saved {len(chunks)} chunks to {output_file}")
    
#     def get_statistics(self, chunks: List[Dict]) -> Dict[str, Any]:
#         """Generate statistics about the chunks"""
#         stats = {
#             "chunk_types": {},
#             "courses_by_level": {},
#             "courses_with_prerequisites": 0,
#             "total_programs": 0,
#             "pages_processed": set()
#         }
        
#         for chunk in chunks:
#             chunk_type = chunk['metadata'].get('chunk_type', 'unknown')
#             stats['chunk_types'][chunk_type] = stats['chunk_types'].get(chunk_type, 0) + 1
            
#             if chunk_type == 'course_description':
#                 level = chunk['metadata'].get('course_level', 'unknown')
#                 stats['courses_by_level'][level] = stats['courses_by_level'].get(level, 0) + 1
                
#                 if chunk['metadata'].get('has_prerequisites', False):
#                     stats['courses_with_prerequisites'] += 1
            
#             if chunk_type == 'program_description':
#                 stats['total_programs'] += 1
            
#             stats['pages_processed'].add(chunk['metadata'].get('page_number', 0))
        
#         stats['pages_processed'] = len(stats['pages_processed'])
        
#         return stats
    
#     def print_summary(self, chunks: List[Dict]) -> None:
#         """Print processing summary"""
#         stats = self.get_statistics(chunks)
        
#         print("\n" + "="*60)
#         print("PROCESSING SUMMARY")
#         print("="*60)
#         print(f"Total Chunks Created: {len(chunks)}")
#         print(f"Pages Processed: {stats['pages_processed']}")
#         print(f"\nChunk Types:")
#         for chunk_type, count in stats['chunk_types'].items():
#             print(f"  - {chunk_type}: {count}")
        
#         if stats['courses_by_level']:
#             print(f"\nCourses by Level:")
#             for level, count in stats['courses_by_level'].items():
#                 print(f"  - {level}: {count}")
        
#         print(f"\nCourses with Prerequisites: {stats['courses_with_prerequisites']}")
#         print(f"Total Programs: {stats['total_programs']}")
#         print("="*60)


# # Usage Example
# def main():
#     """Example usage of the processor"""
    
#     input_file = r'C:\chtn\gen_ai\hitesh\zenza_chatbot\project_xa001\chtn_test_docs\extracted_data_enhanced_merged.json'
#     # Load your JSON data
#     with open(input_file, 'r', encoding='utf-8') as f:
#         json_data = json.load(f)
    
#     # Create processor
#     processor = CourseDataProcessor()
    
#     # Process the data
#     print("Processing course data...")
#     chunks = processor.process_json(json_data)
    
#     # Print summary
#     processor.print_summary(chunks)
    
#     # Save chunks
#     processor.save_chunks(chunks, 'chtn_test_docs\\embedding_chunks.json')
    
#     # Print sample chunks
#     print("\n" + "="*60)
#     print("SAMPLE CHUNKS")
#     print("="*60)
    
#     for i, chunk in enumerate(chunks[:3], 1):
#         print(f"\nChunk {i}:")
#         print(f"ID: {chunk['chunk_id']}")
#         print(f"Type: {chunk['metadata']['chunk_type']}")
#         print(f"Text: {chunk['text'][:200]}...")
#         print(f"Metadata: {json.dumps(chunk['metadata'], indent=2)}")
    
#     return chunks


# if __name__ == "__main__":
#     chunks = main()

Processing course data...

PROCESSING SUMMARY
Total Chunks Created: 522
Pages Processed: 55

Chunk Types:
  - program_description: 360
  - semester_requirements: 94
  - course_prefix_mapping: 49
  - course_description: 19

Courses by Level:
  - introductory: 4
  - intermediate: 4
  - advanced: 11

Courses with Prerequisites: 8
Total Programs: 360
✓ Saved 522 chunks to chtn_test_docs\embedding_chunks.json

SAMPLE CHUNKS

Chunk 1:
ID: chunk_000001
Type: program_description
Text: Program: The Original Tribally-Controlled College  Established In 1968  Description: The Original Tribally-Controlled College  Established In 1968...
Metadata: {
  "program_name": "The Original Tribally-Controlled College \u2022 Established In 1968",
  "is_degree_program": false,
  "page_number": 1,
  "column": "",
  "chunk_type": "program_description",
  "keywords": [
    "original",
    "tribally",
    "controlled",
    "college",
    "established"
  ]
}

Chunk 2:
ID: chunk_000002
Type: program_description
Text

#### Chunking part 2 :- 

In [15]:
import json
import re
from typing import List, Dict, Any, Optional
from datetime import datetime

class CourseDataProcessor:
    """
    Processes structured course data JSON into embedding-ready chunks
    with rich metadata for hybrid search capabilities.
    """
    
    def __init__(self):
        self.chunks = []
        self.chunk_id_counter = 0
        self.processing_errors = []
        self.processed_pages = set()
        
    def generate_chunk_id(self) -> str:
        """Generate unique chunk ID"""
        self.chunk_id_counter += 1
        return f"chunk_{self.chunk_id_counter:06d}"
    
    def clean_text(self, text: str) -> str:
        """Clean and normalize text"""
        if not text:
            return ""
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        # Remove special characters but keep basic punctuation
        text = re.sub(r'[^\w\s.,;:()\-/&]', '', text)
        return text.strip()
    
    def extract_prerequisites(self, description: str, prerequisite_field: str = "") -> List[str]:
        """Extract prerequisite courses from description or prerequisite field"""
        prereqs = []
        
        # First check the prerequisite field if provided
        if prerequisite_field and prerequisite_field.strip():
            courses = re.findall(r'[A-Z]{2,4}\s+\d{3}', prerequisite_field)
            prereqs.extend(courses)
        
        # Then check in description
        if description:
            prereq_pattern = r'Prerequisite[s]?:\s*([^.]+)'
            matches = re.findall(prereq_pattern, description, re.IGNORECASE)
            
            for match in matches:
                courses = re.findall(r'[A-Z]{2,4}\s+\d{3}', match)
                prereqs.extend(courses)
        
        return list(set(prereqs))  # Remove duplicates
    
    def determine_course_level(self, course_code: str) -> str:
        """Determine course level from code"""
        if not course_code:
            return "unknown"
        
        match = re.search(r'(\d{3})', course_code)
        if match:
            level_num = int(match.group(1)[0])
            if level_num <= 1:
                return "introductory"
            elif level_num <= 2:
                return "intermediate"
            elif level_num <= 4:
                return "advanced"
            else:
                return "graduate"
        return "unknown"
    
    def parse_course_header(self, header: str) -> Dict[str, str]:
        """Parse course header to extract code, title, and credits"""
        try:
            # Pattern: "PREFIX NUM Title (Credits)"
            pattern = r'^([A-Z]{2,4})\s+(\d{3})\s+(.+?)\s*\((\d+)\)\s*$'
            match = re.match(pattern, header.strip())
            
            if match:
                prefix, number, title, credits = match.groups()
                return {
                    'code': f"{prefix} {number}",
                    'title': title.strip(),
                    'credits': credits,
                    'prefix': prefix
                }
            
            # Fallback: try to extract what we can
            code_match = re.search(r'([A-Z]{2,4}\s+\d{3})', header)
            credit_match = re.search(r'\((\d+)\)', header)
            
            if code_match:
                code = code_match.group(1)
                title = header.replace(code, '').strip()
                if credit_match:
                    title = title.replace(f"({credit_match.group(1)})", '').strip()
                
                return {
                    'code': code,
                    'title': title,
                    'credits': credit_match.group(1) if credit_match else '',
                    'prefix': code.split()[0] if code else ''
                }
            
        except Exception as e:
            self.processing_errors.append(f"Error parsing course header '{header}': {str(e)}")
        
        return {
            'code': '',
            'title': header,
            'credits': '',
            'prefix': ''
        }
    
    def create_course_chunk(self, course_data: Dict, section_header: str, 
                           page_number: int, column: str = "") -> Optional[Dict[str, Any]]:
        """Create a chunk from course description with rich metadata"""
        try:
            # Handle both old and new formats
            if 'header' in course_data:
                # New format: parse from header
                parsed = self.parse_course_header(course_data.get('header', ''))
                code = parsed['code']
                title = parsed['title']
                credits = parsed['credits']
                
                # Get content
                content = course_data.get('content', {})
                if isinstance(content, dict):
                    description = content.get('text', '') or content.get('description', '')
                    prerequisite_field = content.get('prerequisite', '')
                else:
                    description = str(content) if content else ''
                    prerequisite_field = ''
            else:
                # Old format: use existing fields
                code = course_data.get('code', '') or course_data.get('course_code', '')
                title = course_data.get('title', '') or course_data.get('course_name', '')
                credits = course_data.get('credits', '')
                description = course_data.get('description', '')
                prerequisite_field = course_data.get('prerequisite', '')
            
            # Skip if no valid course code and no title/description
            if not code and not title and not description:
                self.processing_errors.append(f"Skipping empty course entry on page {page_number}")
                return None
            
            # Extract prerequisites
            prerequisites = self.extract_prerequisites(description, prerequisite_field)
            
            # Determine course level
            course_level = self.determine_course_level(code)
            
            # Extract course prefix
            course_prefix = code.split()[0] if code else ""
            
            # Create rich, searchable text
            text_parts = []
            
            # Add section context
            if section_header:
                text_parts.append(f"Department: {section_header}")
            
            # Add course information
            if code:
                text_parts.append(f"Course Code: {code}")
            if title:
                text_parts.append(f"Course Title: {title}")
            if credits:
                text_parts.append(f"Credits: {credits}")
            
            # Add prerequisites if any
            if prerequisites:
                text_parts.append(f"Prerequisites: {', '.join(prerequisites)}")
            else:
                text_parts.append("Prerequisites: None")
            
            # Add description if available
            if description and description.strip():
                text_parts.append(f"Description: {description}")
            elif prerequisite_field and prerequisite_field.strip():
                text_parts.append(f"Requirements: {prerequisite_field}")
            
            # Combine into searchable text
            searchable_text = " | ".join(text_parts)
            
            # Skip if no meaningful content
            if not searchable_text.strip():
                return None
            
            # Create metadata
            metadata = {
                "course_code": code,
                "course_title": title,
                "course_prefix": course_prefix,
                "credits": credits,
                "prerequisites": prerequisites,
                "has_prerequisites": len(prerequisites) > 0,
                "course_level": course_level,
                "department": section_header,
                "page_number": page_number,
                "column": column,
                "chunk_type": "course_description",
                "keywords": self.extract_keywords(f"{title} {description}"),
                "has_description": bool(description and description.strip())
            }
            
            return {
                "chunk_id": self.generate_chunk_id(),
                "text": self.clean_text(searchable_text),
                "metadata": metadata,
                "timestamp": datetime.now().isoformat()
            }
            
        except Exception as e:
            self.processing_errors.append(f"Error creating course chunk on page {page_number}: {str(e)}")
            return None
    
    def create_program_chunk(self, header: str, content: str, 
                            page_number: int, column: str = "") -> Optional[Dict[str, Any]]:
        """Create a chunk from program description"""
        try:
            # Determine if it's a degree program
            is_degree = any(word in header.lower() for word in 
                           ['bachelor', 'master', 'degree', 'b.s.', 'm.s.'])
            
            # Create searchable text
            text_parts = [
                f"Program: {header}",
            ]
            
            if content and content.strip():
                text_parts.append(f"Description: {content}")
            
            searchable_text = " | ".join(text_parts)
            
            # Skip if no content
            if not searchable_text.strip():
                return None
            
            metadata = {
                "program_name": header,
                "is_degree_program": is_degree,
                "page_number": page_number,
                "column": column,
                "chunk_type": "program_description",
                "keywords": self.extract_keywords(f"{header} {content}"),
            }
            
            return {
                "chunk_id": self.generate_chunk_id(),
                "text": self.clean_text(searchable_text),
                "metadata": metadata,
                "timestamp": datetime.now().isoformat()
            }
        except Exception as e:
            self.processing_errors.append(f"Error creating program chunk on page {page_number}: {str(e)}")
            return None
    
    def create_semester_chunk(self, semester: str, courses: List[Dict], 
                             program_context: str, page_number: int) -> Optional[Dict[str, Any]]:
        """Create a chunk from semester requirements"""
        try:
            # Extract course information
            course_list = []
            total_credits = 0
            semester_name = semester
            additional_info = []
            
            # If no explicit semester name, try to extract from data
            if not semester_name:
                for row in courses:
                    if isinstance(row, dict):
                        for key in row.keys():
                            if 'semester' in str(key).lower():
                                semester_name = str(key)
                                break
                    if semester_name:
                        break
            
            for row in courses:
                if not isinstance(row, dict):
                    continue
                
                # Skip header row
                if row.get('Course') == 'Credits' or row.get('Course Title') == 'Course Code':
                    continue
                
                # Try multiple possible field names for course and credits
                course = (row.get('course', '') or row.get('Course', '') or 
                         row.get('Senior Semester IV', '') or row.get('Semester IV', ''))
                credits = (row.get('credits', '') or row.get('Credits', '') or 
                          row.get('credit', ''))
                
                # Check all keys in the row for semester-related or important info
                for key, value in row.items():
                    if not value or not str(value).strip():
                        continue
                    
                    key_lower = str(key).lower()
                    value_str = str(value).strip()
                    
                    # Skip empty or redundant values
                    if value_str in ['', 'Credits', 'Course', '']:
                        continue
                    
                    # Extract semester info from key names
                    if 'semester' in key_lower and value_str and not course:
                        course = value_str
                    elif key_lower == 'credits' and value_str:
                        credits = value_str
                    # Capture degree requirements info
                    elif any(x in key_lower for x in ['degree', 'division', 'requirement', 'earned', 'total']):
                        if value_str and value_str not in additional_info:
                            additional_info.append(f"{key}: {value_str}")
                    # Capture program credits info
                    elif 'program credits' in key_lower:
                        if value_str and value_str not in additional_info:
                            additional_info.append(f"{key}: {value_str}")
                
                # Add course to list if valid
                if course and course.strip() and course.lower() not in ['total', 'credits', 'course']:
                    credit_str = str(credits) if credits else '0'
                    course_list.append(f"{course} ({credit_str} credits)")
                    try:
                        # Try to extract numeric credits (handle ranges like "8-9")
                        credit_nums = re.findall(r'\d+', str(credits))
                        if credit_nums:
                            # Use the first number in a range
                            total_credits += int(credit_nums[0])
                    except:
                        pass
            
            # Create searchable text
            text_parts = []
            if program_context:
                text_parts.append(f"Program: {program_context}")
            text_parts.append(f"Semester: {semester_name or semester}")
            if course_list:
                text_parts.append(f"Required Courses: {'; '.join(course_list)}")
            if additional_info:
                text_parts.append(f"Additional Info: {'; '.join(additional_info)}")
            text_parts.append(f"Total Credits: {total_credits}")
            
            searchable_text = " | ".join(text_parts)
            
            # Skip if no content
            if not searchable_text.strip():
                return None
            
            # Extract course codes
            course_codes = re.findall(r'[A-Z]{2,4}\s+\d{3}', ' '.join(course_list))
            
            metadata = {
                "semester": semester_name or semester,
                "program": program_context,
                "courses": course_codes,
                "total_credits": total_credits,
                "course_count": len(course_list),
                "additional_requirements": additional_info,
                "page_number": page_number,
                "chunk_type": "semester_requirements",
                "keywords": self.extract_keywords(searchable_text),
            }
            
            return {
                "chunk_id": self.generate_chunk_id(),
                "text": self.clean_text(searchable_text),
                "metadata": metadata,
                "timestamp": datetime.now().isoformat()
            }
        except Exception as e:
            self.processing_errors.append(f"Error creating semester chunk on page {page_number}: {str(e)}")
            return None
        
    def create_summary_table_chunk(self, header: str, table_data: List[Dict], 
                                   parent_header: str, page_number: int) -> Optional[Dict[str, Any]]:
        """Create a chunk from summary/degree requirement tables or any general tables"""
        try:
            # Extract all key-value pairs from the table
            requirements = []
            
            for row in table_data:
                if not isinstance(row, dict):
                    # Handle if row is string or other - convert to dict if possible
                    if isinstance(row, str) and row.strip():
                        requirements.append(row)
                    continue
                
                for key, value in row.items():
                    if not key or not value:
                        continue
                    
                    key_str = str(key).strip()
                    value_str = str(value).strip()
                    
                    # Skip empty or header-like values
                    if not value_str or value_str in ['Credits', 'Course', '']:
                        continue
                    
                    # Skip duplicate keys
                    if key_str in ['', 'Credits', 'Course']:
                        continue
                    
                    # Create requirement entry
                    if key_str and value_str:
                        requirements.append(f"{key_str}: {value_str}")
            
            if not requirements:
                return None
            
            # Create searchable text
            text_parts = []
            if parent_header:
                text_parts.append(f"Section: {parent_header}")
            if header:
                text_parts.append(f"Table: {header}")
            text_parts.append(f"Requirements: {' | '.join(requirements)}")
            
            searchable_text = " | ".join(text_parts)
            
            metadata = {
                "section": parent_header,
                "table_name": header,
                "requirements": requirements,
                "page_number": page_number,
                "chunk_type": "requirements_table",
                "keywords": self.extract_keywords(searchable_text),
            }
            
            return {
                "chunk_id": self.generate_chunk_id(),
                "text": self.clean_text(searchable_text),
                "metadata": metadata,
                "timestamp": datetime.now().isoformat()
            }
        except Exception as e:
            self.processing_errors.append(f"Error creating summary table chunk on page {page_number}: {str(e)}")
            return None
        
    def create_generic_content_chunk(self, header: str, content: Dict, 
                                     parent_header: str, page_number: int, 
                                     column: str = "") -> Optional[Dict[str, Any]]:
        """Create a chunk from generic content (lists, paragraphs, text, images with desc)"""
        try:
            content_type = content.get('type', '')
            text_content = ""
            
            # Extract text based on content type
            text = content.get('text', '')
            if isinstance(text, list):
                text_content = ' | '.join([str(t).strip() for t in text if str(t).strip()])
            elif isinstance(text, str):
                text_content = text.strip()
            
            items = content.get('items', [])
            if isinstance(items, list):
                item_text = ' | '.join([str(i).strip() for i in items if str(i).strip()])
                if item_text:
                    text_content = (text_content + ' | ' + item_text).strip(' | ')
            
            data = content.get('data', [])
            if isinstance(data, list):
                data_text = ' | '.join([str(d).strip() for d in data if str(d).strip()])
                if data_text:
                    text_content = (text_content + ' | ' + data_text).strip(' | ')
            
            # For images, use description as content
            if content_type == 'image' and text_content:
                text_content = f"Image Description: {text_content}"
            
            # Skip if no meaningful content
            if not text_content.strip():
                return None
            
            # Create searchable text
            text_parts = []
            if parent_header and parent_header != header:
                text_parts.append(f"Section: {parent_header}")
            if header:
                text_parts.append(f"Title: {header}")
            text_parts.append(f"Content: {text_content}")
            
            searchable_text = " | ".join(text_parts)
            
            # Determine content category
            header_lower = (header or '').lower()
            parent_lower = (parent_header or '').lower()
            combined = f"{header_lower} {parent_lower}"
            
            if any(x in combined for x in ['calendar', 'schedule', 'semester', 'session']):
                category = 'academic_calendar'
            elif any(x in combined for x in ['administration', 'dean', 'president', 'board', 'regent']):
                category = 'administration'
            elif any(x in combined for x in ['table of contents', 'contents']):
                category = 'table_of_contents'
            elif any(x in combined for x in ['mission', 'vision', 'principle', 'history']):
                category = 'college_information'
            elif any(x in combined for x in ['admission', 'enrollment', 'registration', 'policy', 'procedure']):
                category = 'policies_procedures'
            else:
                category = 'general_information'
            
            metadata = {
                "title": header or parent_header,
                "section": parent_header,
                "content_type": content_type,
                "category": category,
                "page_number": page_number,
                "column": column,
                "chunk_type": "informational_content",
                "keywords": self.extract_keywords(searchable_text),
            }
            
            return {
                "chunk_id": self.generate_chunk_id(),
                "text": self.clean_text(searchable_text),
                "metadata": metadata,
                "timestamp": datetime.now().isoformat()
            }
        except Exception as e:
            self.processing_errors.append(f"Error creating generic content chunk on page {page_number}: {str(e)}")
            return None
    
    def process_course_list(self, items: List[Dict], section_header: str, 
                           page_number: int, column: str = "") -> None:
        """Process a list of courses from content.items"""
        try:
            courses_found = False
            for item in items:
                if not isinstance(item, dict):
                    continue
                
                course_code = item.get('course_code', '') or item.get('code', '')
                course_title = item.get('course_name', '') or item.get('title', '')
                description = item.get('description', '')
                
                if course_code or re.search(r'[A-Z]{2,4}\s+\d{3}', str(item)):
                    courses_found = True
                
                if not course_code:
                    continue
                
                # Parse course info
                code_match = re.search(r'([A-Z]{2,4}\s+\d{3})', course_code)
                title_match = re.search(r'\((\d+)\)', course_title)
                
                code = code_match.group(1) if code_match else course_code
                credits = title_match.group(1) if title_match else ''
                
                # Clean title - remove credits part
                title = re.sub(r'\s*\(\d+\)\s*$', '', course_title)
                
                # Create course data dict
                course_data = {
                    'code': code,
                    'title': title,
                    'credits': credits,
                    'description': description,
                    'prerequisite': ''
                }
                
                chunk = self.create_course_chunk(
                    course_data, section_header, page_number, column
                )
                if chunk:
                    self.chunks.append(chunk)
            
            # If no courses found, treat as generic list
            if not courses_found:
                text_content = ' | '.join([f"{str(item).strip()}" for item in items if str(item).strip()])
                if text_content:
                    generic_content = {'type': 'list', 'text': text_content}
                    chunk = self.create_generic_content_chunk(
                        section_header, generic_content, section_header, page_number, column
                    )
                    if chunk:
                        self.chunks.append(chunk)
                    
        except Exception as e:
            self.processing_errors.append(f"Error processing course list on page {page_number}: {str(e)}")
            
    def create_course_prefix_chunk(self, course_title: str, course_code: str, 
                                   page_number: int) -> Optional[Dict[str, Any]]:
        """Create a chunk for course prefix mapping"""
        try:
            text = f"Course Prefix: {course_code} | Department: {course_title} | {course_code} courses are in the {course_title} department"
            
            metadata = {
                "course_prefix": course_code,
                "department_name": course_title,
                "page_number": page_number,
                "chunk_type": "course_prefix_mapping",
                "keywords": [course_code.lower(), course_title.lower()],
            }
            
            return {
                "chunk_id": self.generate_chunk_id(),
                "text": self.clean_text(text),
                "metadata": metadata,
                "timestamp": datetime.now().isoformat()
            }
        except Exception as e:
            self.processing_errors.append(f"Error creating prefix chunk on page {page_number}: {str(e)}")
            return None
    
    def extract_keywords(self, text: str, max_keywords: int = 10) -> List[str]:
        """Extract important keywords from text"""
        if not text:
            return []
        
        # Remove common words
        stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 
                     'to', 'for', 'of', 'with', 'by', 'from', 'this', 'that',
                     'is', 'are', 'was', 'were', 'be', 'been', 'being'}
        
        # Extract words
        words = re.findall(r'\b[a-z]{3,}\b', text.lower())
        
        # Filter and count
        keywords = [w for w in words if w not in stop_words]
        
        # Get unique keywords (preserve order, limit count)
        seen = set()
        unique_keywords = []
        for k in keywords:
            if k not in seen and len(unique_keywords) < max_keywords:
                seen.add(k)
                unique_keywords.append(k)
        
        return unique_keywords
    
    def process_section(self, section: Dict, page_number: int, 
                       parent_header: str = "", column: str = "") -> None:
        """Process a section and its subsections recursively"""
        try:
            header = section.get('header', '')
            header_type = section.get('header_type', '')
            current_context = header if header else parent_header
            
            # Process content
            content = section.get('content', {})
            
            # Ensure content is a dict
            if not isinstance(content, dict):
                content = {'type': 'unknown', 'text': str(content)}
            
            content_type = content.get('type', '')
            
            # Handle course headers (new format)
            if header_type == 'course_header':
                chunk = self.create_course_chunk(
                    section, parent_header, page_number, column
                )
                if chunk:
                    self.chunks.append(chunk)
            
            # Handle department headers
            elif header_type == 'department_header':
                # Create a department chunk if there's description
                text = content.get('text', '')
                if text and text.strip():
                    chunk = self.create_program_chunk(
                        header, text, page_number, column
                    )
                    if chunk:
                        self.chunks.append(chunk)
            
            # Handle paragraph content (program descriptions)
            elif content_type in ['paragraph', 'department_description']:
                text = content.get('text', '')
                if text and text.strip():
                    chunk = self.create_program_chunk(
                        current_context, text, page_number, column
                    )
                    if chunk:
                        self.chunks.append(chunk)
            
            # Handle course descriptions (old format)
            elif content_type == 'multi_course_description':
                courses = content.get('courses', [])
                for course in courses:
                    chunk = self.create_course_chunk(
                        course, current_context, page_number, column
                    )
                    if chunk:
                        self.chunks.append(chunk)
            
            # Handle list-type content
            elif content_type == 'list':
                items = content.get('items', []) or content.get('text', []) if isinstance(content.get('text'), list) else []
                if items and isinstance(items, list) and len(items) > 0:
                    # Check if this looks like a course list
                    is_course_list = False
                    if isinstance(items[0], dict):
                        first_item = items[0]
                        if any(key in first_item for key in ['course_code', 'course_name', 'prerequisite', 'code', 'title']):
                            is_course_list = True
                    elif re.search(r'[A-Z]{2,4}\s+\d{3}', str(items[0])):
                        is_course_list = True
                    
                    if is_course_list:
                        self.process_course_list(items, current_context, page_number, column)
                    else:
                        chunk = self.create_generic_content_chunk(header, content, parent_header, page_number, column)
                        if chunk:
                            self.chunks.append(chunk)
            
            # Handle single course description
            elif content_type == 'course_description':
                chunk = self.create_course_chunk(
                    section, parent_header, page_number, column
                )
                if chunk:
                    self.chunks.append(chunk)
            
            # Handle tables
            elif content_type == 'table':
                data = content.get('data', [])
                headers = content.get('headers', [])
                rows = content.get('rows', [])
                
                # Use 'data' if available (new format), otherwise use 'rows' (old format)
                table_data = data if data else rows
                
                # Handle if table_data elements are not dicts
                processed_data = []
                for item in table_data:
                    if isinstance(item, dict):
                        processed_data.append(item)
                    elif isinstance(item, str) and item.strip():
                        processed_data.append({"Entry": item})
                    elif isinstance(item, (list, tuple)) and len(item) >= 2:
                        processed_data.append({str(item[0]): str(item[1])})
                
                table_data = processed_data
                
                # Skip if no data
                if not table_data:
                    pass
                
                # Check if it's a course prefix table
                if 'Course Title' in headers and 'Course Code' in headers:
                    for row in table_data:
                        if isinstance(row, dict):
                            course_title = row.get('course_title', '') or row.get('Course Title', '')
                            course_code = row.get('course_code', '') or row.get('Course Code', '')
                            if course_title and course_code and course_title != 'Course Title':
                                chunk = self.create_course_prefix_chunk(
                                    course_title, course_code, page_number
                                )
                                if chunk:
                                    self.chunks.append(chunk)
                
                # Check for semester table
                else:
                    # Check if this is a semester-related table
                    is_semester_table = False
                    has_degree_info = False
                    
                    for row in table_data:
                        if isinstance(row, dict):
                            row_keys = [str(k).lower() for k in row.keys()]
                            row_values = [str(v).lower() for v in row.values() if v]
                            
                            # Check for semester indicators
                            if any('semester' in k for k in row_keys):
                                is_semester_table = True
                            
                            # Check for degree requirement indicators
                            if any(x in ' '.join(row_keys + row_values) for x in 
                                  ['degree earned', 'division requirements', 'total credits earned']):
                                has_degree_info = True
                    
                    # Create appropriate chunk
                    if is_semester_table:
                        chunk = self.create_semester_chunk(
                            header or parent_header, table_data, parent_header, page_number
                        )
                        if chunk:
                            self.chunks.append(chunk)
                    else:
                        # Create summary chunk for all other tables if data exists
                        if len(table_data) > 0:
                            chunk = self.create_summary_table_chunk(
                                header, table_data, parent_header, page_number
                            )
                            if chunk:
                                self.chunks.append(chunk)
            
            # Handle images with descriptions or any unprocessed content
            else:
                # Check if this content has text or items
                has_content = (content.get('text') or content.get('items') or 
                               content.get('data') or content.get('rows'))
                
                # Create generic chunk for informational content
                if has_content:
                    chunk = self.create_generic_content_chunk(
                        header, content, parent_header, page_number, column
                    )
                    if chunk:
                        self.chunks.append(chunk)
            
            # Process subsections recursively
            subsections = section.get('subsections', [])
            for subsection in subsections:
                self.process_section(subsection, page_number, current_context, column)
                
        except Exception as e:
            self.processing_errors.append(f"Error processing section '{header}' on page {page_number}: {str(e)}")
    
    def process_page(self, page: Dict) -> None:
        """Process a single page"""
        try:
            page_number = page.get('page_number', 0)
            if not page_number:
                return  # Skip pages without number
            
            self.processed_pages.add(page_number)
            
            sections = page.get('sections', [])
            
            if not sections:
                self.processing_errors.append(f"Warning: Page {page_number} has no sections")
            
            for section in sections:
                column = section.get('column', '')
                self.process_section(section, page_number, column=column)
                
        except Exception as e:
            page_num = page.get('page_number', 'unknown')
            self.processing_errors.append(f"Error processing page {page_num}: {str(e)}")
    
    def process_json(self, json_data) -> List[Dict[str, Any]]:
        """Main processing function"""
        self.chunks = []
        self.chunk_id_counter = 0
        self.processing_errors = []
        self.processed_pages = set()
        
        # Handle both list format and dict format
        if isinstance(json_data, list):
            pages = json_data
        else:
            pages = json_data.get('pages', [])
        
        if not pages:
            self.processing_errors.append("Error: No pages found in JSON data")
            return []
        
        print(f"Processing {len(pages)} pages...")
        
        # Track which page numbers we expect
        expected_pages = set()
        for page in pages:
            page_num = page.get('page_number', 0)
            if page_num:
                expected_pages.add(page_num)
        
        for idx, page in enumerate(pages):
            try:
                self.process_page(page)
                if (idx + 1) % 10 == 0:
                    print(f"  Processed {idx + 1}/{len(pages)} pages...")
            except Exception as e:
                page_num = page.get('page_number', 'unknown')
                self.processing_errors.append(f"Critical error on page {page_num} (index {idx}): {str(e)}")
        
        # Check for missing pages
        missing_pages = expected_pages - self.processed_pages
        if missing_pages:
            print(f"\n⚠ WARNING: {len(missing_pages)} pages were not processed correctly:")
            for page_num in sorted(missing_pages):
                print(f"  - Page {page_num}")
                self.processing_errors.append(f"Page {page_num} was not processed")
        
        return self.chunks
    
    def save_chunks(self, chunks: List[Dict], output_file: str = 'embedding_chunks.json'):
        """Save processed chunks to JSON file"""
        output_data = {
            "total_chunks": len(chunks),
            "processed_at": datetime.now().isoformat(),
            "chunks": chunks,
            "statistics": self.get_statistics(chunks),
            "processing_errors": self.processing_errors
        }
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(output_data, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Saved {len(chunks)} chunks to {output_file}")
        
        if self.processing_errors:
            print(f"⚠ Encountered {len(self.processing_errors)} errors during processing")
            print("  Check 'processing_errors' field in output file for details")
    
    def get_statistics(self, chunks: List[Dict]) -> Dict[str, Any]:
        """Generate statistics about the chunks"""
        stats = {
            "chunk_types": {},
            "courses_by_level": {},
            "courses_with_prerequisites": 0,
            "courses_without_description": 0,
            "total_programs": 0,
            "pages_processed": sorted(list(self.processed_pages)),
            "page_count": len(self.processed_pages),
            "missing_pages": []
        }
        
        # Check for missing pages
        if self.processed_pages:
            min_page = min(self.processed_pages)
            max_page = max(self.processed_pages)
            all_pages = set(range(min_page, max_page + 1))
            stats['missing_pages'] = sorted(list(all_pages - self.processed_pages))
        
        for chunk in chunks:
            chunk_type = chunk['metadata'].get('chunk_type', 'unknown')
            stats['chunk_types'][chunk_type] = stats['chunk_types'].get(chunk_type, 0) + 1
            
            if chunk_type == 'course_description':
                level = chunk['metadata'].get('course_level', 'unknown')
                stats['courses_by_level'][level] = stats['courses_by_level'].get(level, 0) + 1
                
                if chunk['metadata'].get('has_prerequisites', False):
                    stats['courses_with_prerequisites'] += 1
                
                if not chunk['metadata'].get('has_description', True):
                    stats['courses_without_description'] += 1
            
            if chunk_type == 'program_description':
                stats['total_programs'] += 1
            if chunk_type == 'informational_content':
                category = chunk['metadata'].get('category', 'unknown')
                if 'info_categories' not in stats:
                    stats['info_categories'] = {}
                stats['info_categories'][category] = stats['info_categories'].get(category, 0) + 1
        
        return stats
    
    def print_summary(self, chunks: List[Dict]) -> None:
        """Print processing summary"""
        stats = self.get_statistics(chunks)
        
        print("\n" + "="*60)
        print("PROCESSING SUMMARY")
        print("="*60)
        print(f"Total Chunks Created: {len(chunks)}")
        print(f"Pages Processed: {stats['page_count']}")
        print(f"Page Range: {min(stats['pages_processed'])} - {max(stats['pages_processed'])}")
        
        if stats['missing_pages']:
            print(f"\n⚠ Missing Pages: {stats['missing_pages'][:10]}")
            if len(stats['missing_pages']) > 10:
                print(f"  ... and {len(stats['missing_pages']) - 10} more")
        
        print(f"\nChunk Types:")
        for chunk_type, count in stats['chunk_types'].items():
            print(f"  - {chunk_type}: {count}")
        
        if stats['courses_by_level']:
            print(f"\nCourses by Level:")
            for level, count in stats['courses_by_level'].items():
                print(f"  - {level}: {count}")
                
        if 'info_categories' in stats:
            print(f"\nInformational Content Categories:")
            for category, count in stats['info_categories'].items():
                print(f"  - {category}: {count}")
        
        print(f"\nCourses with Prerequisites: {stats['courses_with_prerequisites']}")
        print(f"Courses without Description: {stats['courses_without_description']}")
        print(f"Total Programs: {stats['total_programs']}")
        
        if self.processing_errors:
            print(f"\n⚠ Processing Errors: {len(self.processing_errors)}")
            print("  (See output file for details)")
        
        print("="*60)


# Usage Example
def main():
    """Example usage of the processor"""
    
    input_file = r'C:\chtn\gen_ai\hitesh\zenza_chatbot\project_xa001\chtn_test_docs\extracted_data_enhanced.json'
    
    try:
        # Load your JSON data
        with open(input_file, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        
        # Create processor
        processor = CourseDataProcessor()
        
        # Process the data
        print("Processing course data...")
        chunks = processor.process_json(json_data)
        
        # Print summary
        processor.print_summary(chunks)
        
        # Save chunks
        processor.save_chunks(chunks, 'chtn_test_docs\\embedding_chunks_robust.json')
        
        # Print sample chunks
        print("\n" + "="*60)
        print("SAMPLE CHUNKS")
        print("="*60)
        
        for i, chunk in enumerate(chunks[:3], 1):
            print(f"\nChunk {i}:")
            print(f"ID: {chunk['chunk_id']}")
            print(f"Type: {chunk['metadata']['chunk_type']}")
            print(f"Text: {chunk['text'][:200]}...")
            print(f"Metadata Keys: {list(chunk['metadata'].keys())}")
        
        return chunks
        
    except FileNotFoundError:
        print(f"Error: Could not find file {input_file}")
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON in file - {str(e)}")
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    chunks = main()

Processing course data...
Processing 90 pages...
  Processed 10/90 pages...
  Processed 20/90 pages...
  Processed 30/90 pages...
  Processed 40/90 pages...
  Processed 50/90 pages...
  Processed 60/90 pages...
  Processed 70/90 pages...
  Processed 80/90 pages...
  Processed 90/90 pages...

PROCESSING SUMMARY
Total Chunks Created: 793
Pages Processed: 90
Page Range: 1 - 90

Chunk Types:
  - informational_content: 50
  - program_description: 398
  - requirements_table: 169
  - semester_requirements: 13
  - course_description: 163

Courses by Level:
  - advanced: 92
  - introductory: 35
  - intermediate: 36

Informational Content Categories:
  - general_information: 17
  - administration: 11
  - table_of_contents: 13
  - academic_calendar: 3
  - college_information: 6

Courses with Prerequisites: 122
Courses without Description: 1
Total Programs: 398

⚠ Processing Errors: 3
  (See output file for details)
✓ Saved 793 chunks to chtn_test_docs\embedding_chunks_robust.json
⚠ Encountered 3 

#### Hybrid Search.

In [16]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from typing import List, Dict, Any, Optional, Tuple
import re
from datetime import datetime
import pickle
import os


class HybridSearchSystem:
    """
    A hybrid search system combining semantic search (embeddings) 
    with metadata filtering and query intent classification.
    """
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initialize the search system with a SentenceTransformer model.
        
        Args:
            model_name: Name of the SentenceTransformer model to use
        """
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()
        
        self.chunks = []
        self.embeddings = None
        self.index = None
        
        print(f"✓ Model loaded. Embedding dimension: {self.embedding_dim}")
    
    def load_chunks(self, chunks_file: str = 'embedding_chunks.json') -> None:
        """Load processed chunks from JSON file"""
        print(f"\nLoading chunks from {chunks_file}...")
        
        with open(chunks_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        self.chunks = data.get('chunks', [])
        print(f"✓ Loaded {len(self.chunks)} chunks")
        
        if 'statistics' in data:
            print("\nDataset Statistics:")
            stats = data['statistics']
            for key, value in stats.items():
                if key != 'pages_processed':
                    print(f"  - {key}: {value}")
    
    def create_embeddings(self, batch_size: int = 32, show_progress: bool = True) -> None:
        """Create embeddings for all chunks using SentenceTransformer."""
        print("\nCreating embeddings...")
        
        texts = [chunk['text'] for chunk in self.chunks]
        
        self.embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=show_progress,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        
        print(f"✓ Created embeddings with shape: {self.embeddings.shape}")
    
    def build_index(self, index_type: str = 'flat') -> None:
        """Build FAISS index for fast similarity search."""
        if self.embeddings is None:
            raise ValueError("Embeddings not created. Call create_embeddings() first.")
        
        print(f"\nBuilding FAISS index (type: {index_type})...")
        
        if index_type == 'flat':
            self.index = faiss.IndexFlatIP(self.embedding_dim)
            self.index.add(self.embeddings)
        elif index_type == 'ivf':
            nlist = min(100, len(self.chunks) // 10)
            quantizer = faiss.IndexFlatIP(self.embedding_dim)
            self.index = faiss.IndexIVFFlat(quantizer, self.embedding_dim, nlist)
            self.index.train(self.embeddings)
            self.index.add(self.embeddings)
            self.index.nprobe = 10
        
        print(f"✓ Index built with {self.index.ntotal} vectors")
    
    def save_system(self, save_dir: str = 'search_system') -> None:
        """Save the search system to disk"""
        os.makedirs(save_dir, exist_ok=True)
        
        print(f"\nSaving search system to {save_dir}...")
        
        np.save(os.path.join(save_dir, 'embeddings.npy'), self.embeddings)
        faiss.write_index(self.index, os.path.join(save_dir, 'faiss_index.bin'))
        
        with open(os.path.join(save_dir, 'chunks.pkl'), 'wb') as f:
            pickle.dump(self.chunks, f)
        
        metadata = {
            'embedding_dim': self.embedding_dim,
            'num_chunks': len(self.chunks),
            'created_at': datetime.now().isoformat()
        }
        with open(os.path.join(save_dir, 'metadata.json'), 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print("✓ System saved successfully")
    
    def load_system(self, save_dir: str = 'search_system') -> None:
        """Load the search system from disk"""
        print(f"\nLoading search system from {save_dir}...")
        
        self.embeddings = np.load(os.path.join(save_dir, 'embeddings.npy'))
        self.index = faiss.read_index(os.path.join(save_dir, 'faiss_index.bin'))
        
        with open(os.path.join(save_dir, 'chunks.pkl'), 'rb') as f:
            self.chunks = pickle.load(f)
        
        print(f"✓ System loaded with {len(self.chunks)} chunks")
    
    def classify_query_intent(self, query: str) -> Dict[str, Any]:
        """
        Classify the intent of the user's query.
        
        Returns:
            Dictionary with intent type and extracted entities
        """
        query_lower = query.lower()
        
        intent = {
            'type': 'specific',  # Default
            'requires_all': False,
            'entities': {}
        }
        
        # Pattern 1: List/All queries
        list_patterns = [
            r'\ball\b.*\bcourse',
            r'\blist\b.*\bcourse',
            r'\bshow\b.*\ball',
            r'\bgive\b.*\ball',
            r'\bname\b.*\ball',
            r'\bwhat\b.*\ball',
            r'\bhow many\b',
            r'\btotal\b.*\bcourse',
        ]
        
        for pattern in list_patterns:
            if re.search(pattern, query_lower):
                intent['type'] = 'list_all'
                intent['requires_all'] = True
                break
        
        # Pattern 2: Extract department/prefix
        dept_patterns = [
            r'\b([A-Z]{2,4})\b',  # Course prefix like AGR, BIO
            r'under\s+([A-Z][a-z]+)',  # "under Agriculture"
            r'in\s+([A-Z][a-z]+)',  # "in Biology"
            r'from\s+([A-Z][a-z]+)',  # "from Accounting"
        ]
        
        for pattern in dept_patterns:
            matches = re.findall(pattern, query)
            if matches:
                intent['entities']['department'] = matches[0]
                break
        
        # Pattern 3: Course level queries
        level_patterns = {
            'introductory': [r'\bintro', r'\b100\b', r'\b1\d{2}\b', r'\bbeginning', r'\bbasic'],
            'intermediate': [r'\bintermediate', r'\b200\b', r'\b2\d{2}\b'],
            'advanced': [r'\badvanced', r'\b300\b', r'\b400\b', r'\b[34]\d{2}\b'],
            'graduate': [r'\bgraduate', r'\bmaster', r'\b500\b', r'\b600\b', r'\b[56]\d{2}\b', r'\bgrad\b'],
        }
        
        for level, patterns in level_patterns.items():
            for pattern in patterns:
                if re.search(pattern, query_lower):
                    intent['entities']['level'] = level
                    break
        
        # Pattern 4: Prerequisite queries
        if re.search(r'\bprerequisite|\bprereq\b', query_lower):
            intent['type'] = 'prerequisite'
            if re.search(r'\bno\b|\bwithout\b|\bnone\b', query_lower):
                intent['entities']['has_prerequisites'] = False
            else:
                intent['entities']['has_prerequisites'] = True
        
        # Pattern 5: Credits queries
        credit_match = re.search(r'(\d+)\s*credit', query_lower)
        if credit_match:
            intent['entities']['credits'] = credit_match.group(1)
        
        # Pattern 6: Specific course code
        course_code_match = re.search(r'\b([A-Z]{2,4})\s*(\d{3})\b', query)
        if course_code_match:
            intent['type'] = 'specific_course'
            intent['entities']['course_code'] = f"{course_code_match.group(1)} {course_code_match.group(2)}"
        
        return intent
    
    def execute_list_all_query(self, intent: Dict[str, Any]) -> List[Dict[str, Any]]:
        """
        Execute queries that require ALL matching results.
        
        Args:
            intent: Query intent with entities
            
        Returns:
            List of all matching chunks
        """
        filters = {}
        
        # Always filter for course descriptions when listing
        filters['chunk_type'] = 'course_description'
        
        # Add department filter
        if 'department' in intent['entities']:
            dept = intent['entities']['department'].upper()
            # Filter by course_prefix
            matching_chunks = [
                chunk for chunk in self.chunks
                if chunk.get('metadata', {}).get('chunk_type') == 'course_description' and
                   chunk.get('metadata', {}).get('course_prefix', '') == dept
            ]
        else:
            matching_chunks = [
                chunk for chunk in self.chunks
                if chunk.get('metadata', {}).get('chunk_type') == 'course_description'
            ]
        
        # Add level filter if specified
        if 'level' in intent['entities']:
            level = intent['entities']['level']
            matching_chunks = [
                chunk for chunk in matching_chunks
                if chunk.get('metadata', {}).get('course_level') == level
            ]
        
        # Add prerequisite filter if specified
        if 'has_prerequisites' in intent['entities']:
            has_prereq = intent['entities']['has_prerequisites']
            matching_chunks = [
                chunk for chunk in matching_chunks
                if chunk.get('metadata', {}).get('has_prerequisites') == has_prereq
            ]
        
        # Add credits filter if specified
        if 'credits' in intent['entities']:
            credits = intent['entities']['credits']
            matching_chunks = [
                chunk for chunk in matching_chunks
                if chunk.get('metadata', {}).get('credits') == credits
            ]
        
        # Sort by course code for consistent ordering
        matching_chunks.sort(
            key=lambda x: x.get('metadata', {}).get('course_code', '')
        )
        
        return matching_chunks
    
    def semantic_search(self, query: str, top_k: int = 5) -> List[Tuple[int, float]]:
        """Perform semantic search using embeddings."""
        query_embedding = self.model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        
        scores, indices = self.index.search(query_embedding, top_k)
        results = [(int(idx), float(score)) for idx, score in zip(indices[0], scores[0])]
        return results
    
    def keyword_filter(self, query: str, chunks: List[Dict]) -> List[Dict]:
        """Filter chunks based on keyword matching for hybrid search."""
        course_codes = re.findall(r'\b[A-Z]{2,4}\s*\d{3}\b', query.upper())
        
        stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 
                     'to', 'for', 'of', 'with', 'by', 'from', 'is', 'are',
                     'what', 'which', 'who', 'when', 'where', 'how', 'tell',
                     'me', 'about', 'show', 'find', 'list', 'all', 'name'}
        
        query_words = set(re.findall(r'\b\w{3,}\b', query.lower()))
        query_keywords = query_words - stop_words
        
        filtered_chunks = []
        
        for chunk in chunks:
            keyword_score = 0
            metadata = chunk.get('metadata', {})
            
            # Boost for exact course code match
            if course_codes:
                chunk_code = metadata.get('course_code', '')
                for code in course_codes:
                    if code.replace(' ', '') in chunk_code.replace(' ', ''):
                        keyword_score += 10
            
            # Check keyword matches
            chunk_keywords = set(metadata.get('keywords', []))
            matching_keywords = query_keywords & chunk_keywords
            keyword_score += len(matching_keywords) * 2
            
            chunk['keyword_score'] = keyword_score
            filtered_chunks.append(chunk)
        
        return filtered_chunks
    
    def metadata_filter(self, chunks: List[Dict], filters: Dict[str, Any]) -> List[Dict]:
        """Filter chunks based on metadata criteria."""
        if not filters:
            return chunks
        
        filtered = []
        
        for chunk in chunks:
            metadata = chunk.get('metadata', {})
            match = True
            
            for key, value in filters.items():
                if isinstance(value, list):
                    if metadata.get(key) not in value:
                        match = False
                        break
                else:
                    if metadata.get(key) != value:
                        match = False
                        break
            
            if match:
                filtered.append(chunk)
        
        return filtered
    
    def hybrid_search(self, 
                     query: str, 
                     top_k: int = 10,
                     semantic_weight: float = 0.7,
                     keyword_weight: float = 0.3,
                     metadata_filters: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        """Perform hybrid search combining semantic and keyword search."""
        semantic_results = self.semantic_search(query, top_k * 3)
        
        candidate_chunks = []
        for idx, semantic_score in semantic_results:
            chunk = self.chunks[idx].copy()
            chunk['index'] = idx
            chunk['semantic_score'] = semantic_score
            candidate_chunks.append(chunk)
        
        candidate_chunks = self.keyword_filter(query, candidate_chunks)
        
        if metadata_filters:
            candidate_chunks = self.metadata_filter(candidate_chunks, metadata_filters)
        
        max_keyword_score = max([c.get('keyword_score', 0) for c in candidate_chunks]) or 1
        
        for chunk in candidate_chunks:
            norm_keyword_score = chunk.get('keyword_score', 0) / max_keyword_score
            chunk['hybrid_score'] = (
                semantic_weight * chunk['semantic_score'] + 
                keyword_weight * norm_keyword_score
            )
        
        candidate_chunks.sort(key=lambda x: x['hybrid_score'], reverse=True)
        
        return candidate_chunks[:top_k]
    
    def search(self,
               query: str,
               top_k: int = 5,
               search_type: str = 'auto',
               metadata_filters: Optional[Dict[str, Any]] = None,
               verbose: bool = True) -> List[Dict[str, Any]]:
        """
        Main search interface with automatic query intent detection.
        
        Args:
            query: Search query
            top_k: Number of results (ignored for 'list_all' queries)
            search_type: 'auto', 'semantic', 'hybrid', 'filtered', or 'list_all'
            metadata_filters: Optional metadata filters
            verbose: Print search results
            
        Returns:
            List of search results
        """
        if verbose:
            print(f"\n{'='*80}")
            print(f"QUERY: {query}")
            print(f"{'='*80}")
        
        # Automatic intent detection
        if search_type == 'auto':
            intent = self.classify_query_intent(query)
            
            if verbose:
                print(f"Detected Intent: {intent['type']}")
                if intent['entities']:
                    print(f"Extracted Entities: {intent['entities']}")
                print(f"{'-'*80}")
            
            # Execute based on intent
            if intent['requires_all'] or intent['type'] == 'list_all':
                results = self.execute_list_all_query(intent)
                
                if verbose:
                    print(f"\n✓ Found {len(results)} matching courses")
                    self.print_results(results, show_full=False)
                
                return results
            
            elif intent['type'] == 'specific_course':
                # Direct lookup for specific course
                course_code = intent['entities'].get('course_code', '')
                results = [
                    chunk for chunk in self.chunks
                    if chunk.get('metadata', {}).get('course_code', '').replace(' ', '') == 
                       course_code.replace(' ', '')
                ]
                
                if not results:
                    # Fallback to semantic search
                    results = self.hybrid_search(query, top_k=3)
                
                if verbose:
                    self.print_results(results)
                
                return results
            
            else:
                # Use hybrid search for other queries
                search_type = 'hybrid'
        
        # Execute search based on type
        if search_type == 'semantic':
            semantic_results = self.semantic_search(query, top_k)
            results = []
            for idx, score in semantic_results:
                chunk = self.chunks[idx].copy()
                chunk['score'] = score
                results.append(chunk)
        
        elif search_type == 'hybrid':
            results = self.hybrid_search(
                query, 
                top_k=top_k,
                metadata_filters=metadata_filters
            )
            for r in results:
                r['score'] = r.get('hybrid_score', 0)
        
        elif search_type == 'filtered':
            if not metadata_filters:
                raise ValueError("metadata_filters required for 'filtered' search type")
            
            filtered_chunks = self.metadata_filter(self.chunks, metadata_filters)
            
            if not filtered_chunks:
                return []
            
            filtered_indices = [self.chunks.index(c) for c in filtered_chunks]
            filtered_embeddings = self.embeddings[filtered_indices]
            
            temp_index = faiss.IndexFlatIP(self.embedding_dim)
            temp_index.add(filtered_embeddings)
            
            query_embedding = self.model.encode(
                [query],
                convert_to_numpy=True,
                normalize_embeddings=True
            )
            
            scores, indices = temp_index.search(query_embedding, min(top_k, len(filtered_chunks)))
            
            results = []
            for idx, score in zip(indices[0], scores[0]):
                chunk = filtered_chunks[idx].copy()
                chunk['score'] = float(score)
                results.append(chunk)
        
        elif search_type == 'list_all':
            # Manual list_all execution
            intent = self.classify_query_intent(query)
            results = self.execute_list_all_query(intent)
        
        else:
            raise ValueError(f"Unknown search_type: {search_type}")
        
        if verbose:
            self.print_results(results)
        
        return results
    
    def print_results(self, results: List[Dict[str, Any]], show_full: bool = True) -> None:
        """Pretty print search results"""
        if not results:
            print("No results found.")
            return
        
        # For list_all queries, show compact format
        if not show_full and len(results) > 10:
            print("\nCourse Codes Found:")
            print("-" * 80)
            
            for i, result in enumerate(results, 1):
                metadata = result.get('metadata', {})
                code = metadata.get('course_code', 'N/A')
                title = metadata.get('course_title', 'N/A')
                credits = metadata.get('credits', 'N/A')
                
                print(f"{i:3d}. {code:12s} - {title:50s} ({credits} credits)")
            
            print(f"\n{'='*80}")
            print(f"Total Results: {len(results)}")
            print(f"{'='*80}")
            return
        
        # Full format for detailed results
        for i, result in enumerate(results, 1):
            print(f"\n{'-'*80}")
            score = result.get('score', result.get('hybrid_score', 0))
            if score > 0:
                print(f"RESULT {i} | Score: {score:.4f}")
            else:
                print(f"RESULT {i}")
            print(f"{'-'*80}")
            
            metadata = result.get('metadata', {})
            chunk_type = metadata.get('chunk_type', 'unknown')
            
            if chunk_type == 'course_description':
                print(f"Course: {metadata.get('course_code', 'N/A')} - {metadata.get('course_title', 'N/A')}")
                print(f"Credits: {metadata.get('credits', 'N/A')}")
                print(f"Level: {metadata.get('course_level', 'N/A')}")
                
                prereqs = metadata.get('prerequisites', [])
                if prereqs:
                    print(f"Prerequisites: {', '.join(prereqs)}")
                else:
                    print("Prerequisites: None")
            
            elif chunk_type == 'program_description':
                print(f"Program: {metadata.get('program_name', 'N/A')}")
            
            elif chunk_type == 'semester_requirements':
                print(f"Semester: {metadata.get('semester', 'N/A')}")
                print(f"Program: {metadata.get('program', 'N/A')}")
                print(f"Total Credits: {metadata.get('total_credits', 'N/A')}")
            
            elif chunk_type == 'course_prefix_mapping':
                print(f"Department: {metadata.get('department_name', 'N/A')} ({metadata.get('course_prefix', 'N/A')})")
            
            if show_full:
                print(f"\nText Preview:")
                text = result.get('text', '')
                print(f"{text[:300]}..." if len(text) > 300 else text)
                
                print(f"\nPage: {metadata.get('page_number', 'N/A')}")
    
    def interactive_search(self) -> None:
        """Interactive search interface"""
        print("\n" + "="*80)
        print("INTERACTIVE SEARCH MODE")
        print("="*80)
        print("Commands:")
        print("  - Type your query to search")
        print("  - Query will be automatically classified")
        print("  - 'quit' or 'exit' to quit")
        print("="*80)
        
        while True:
            try:
                query = input("\nEnter query: ").strip()
                
                if not query:
                    continue
                
                if query.lower() in ['quit', 'exit']:
                    print("Goodbye!")
                    break
                
                # Perform auto search
                self.search(query, top_k=5, search_type='auto', verbose=True)
                
            except KeyboardInterrupt:
                print("\n\nGoodbye!")
                break
            except Exception as e:
                print(f"Error: {e}")


# def main():
#     """Main function demonstrating the complete workflow"""
    
#     # Initialize the system
#     system = HybridSearchSystem(model_name='all-MiniLM-L6-v2')
    
#     # Load chunks
#     system.load_chunks('embedding_chunks.json')
    
#     # Create embeddings
#     system.create_embeddings(batch_size=32, show_progress=True)
    
#     # Build index
#     system.build_index(index_type='flat')
    
#     # Save the system
#     system.save_system('search_system')
    
#     print("\n" + "="*80)
#     print("SYSTEM READY!")
#     print("="*80)
    
#     # Example searches
#     print("\n" + "="*80)
#     print("EXAMPLE SEARCHES")
#     print("="*80)
    
#     # Example 1: List all courses in a department
#     print("\n### Example 1: List all courses in Agriculture department")
#     system.search("list all course codes in AGRICULTURE (AGR)", search_type='auto')
    
#     # Example 2: List all graduate courses
#     print("\n### Example 2: List all graduate level courses")
#     system.search("show me all graduate courses", search_type='auto')
    
#     # Example 3: Specific course lookup
#     print("\n### Example 3: Specific course")
#     system.search("Tell me about AGR 105", search_type='auto')
    
#     # Example 4: Courses with no prerequisites
#     print("\n### Example 4: Courses with no prerequisites in AGR")
#     system.search("agriculture courses with no prerequisites", search_type='auto')
    
#     # Start interactive mode
#     print("\n" + "="*80)
#     print("Starting interactive mode...")
#     print("="*80)
#     system.interactive_search()
    
#     return system


# if __name__ == "__main__":
#     system = main()

In [17]:
def main():
    """Main function demonstrating the complete workflow"""
    
    # Initialize the system
    system = HybridSearchSystem(model_name='all-MiniLM-L6-v2')
    
    embeddings_path = r'C:\chtn\gen_ai\hitesh\zenza_chatbot\project_xa001\chtn_test_docs\chtn_test_docs\embedding_chunks_robust.json'
    # # Load chunks
    system.load_chunks(embeddings_path)
    
    # Create embeddings
    system.create_embeddings(batch_size=32, show_progress=True)
    
    # Build index
    system.build_index(index_type='flat')
    
    # Save the system for future use
    system.save_system('search_system')
    
    print("\n" + "="*80)
    print("SYSTEM READY!")
    print("="*80)
    
    # Example searches
    print("\n" + "="*80)
    print("EXAMPLE SEARCHES")
    print("="*80)
    
    # # Example 1: Simple course search
    # print("\n### Example 1: Search for accounting courses")
    # system.search("give me details about ACC 100 Fundamentals of Accounting.", top_k=3, search_type='hybrid')

    # Example 2: Simple course search
    print("\n### Example 2: Search for mushroom and molds course")
    system.search("give me details about AGR 323 Mushroom and Molds.", top_k=3, search_type='hybrid')
    
    # Example 2: Search with course code
    # print("\n### Example 2: Search for specific course")
    # system.search("i want name of all the course code that are coming under AGRICULTURE (AGR).", search_type='auto')

    # # Example 2: Search with course code
    # print("\n### Example 3: Search for specific course")
    # system.search("i want name of all the course code that are coming under ENVIRONMENTAL SCIENCE AND TECHNOLOGY.", search_type='auto')

    # # Example 3: Simple course search
    # print("\n### Example 3: Search forENV 105 Climate Change for Tribal Peoples")
    # system.search("give me details about ENV 105 Climate Change for Tribal Peoples.", top_k=3, search_type='hybrid')
    
    
    
    return system


if __name__ == "__main__":
    system = main()

Loading embedding model: all-MiniLM-L6-v2...
✓ Model loaded. Embedding dimension: 384

Loading chunks from C:\chtn\gen_ai\hitesh\zenza_chatbot\project_xa001\chtn_test_docs\chtn_test_docs\embedding_chunks_robust.json...
✓ Loaded 793 chunks

Dataset Statistics:
  - chunk_types: {'informational_content': 50, 'program_description': 398, 'requirements_table': 169, 'semester_requirements': 13, 'course_description': 163}
  - courses_by_level: {'advanced': 92, 'introductory': 35, 'intermediate': 36}
  - courses_with_prerequisites: 122
  - courses_without_description: 1
  - total_programs: 398
  - page_count: 90
  - missing_pages: []
  - info_categories: {'general_information': 17, 'administration': 11, 'table_of_contents': 13, 'academic_calendar': 3, 'college_information': 6}

Creating embeddings...


Batches: 100%|██████████| 25/25 [00:05<00:00,  4.85it/s]

✓ Created embeddings with shape: (793, 384)

Building FAISS index (type: flat)...
✓ Index built with 793 vectors

Saving search system to search_system...
✓ System saved successfully

SYSTEM READY!

EXAMPLE SEARCHES

### Example 2: Search for mushroom and molds course

QUERY: give me details about AGR 323 Mushroom and Molds.

--------------------------------------------------------------------------------
RESULT 1 | Score: 0.8000
--------------------------------------------------------------------------------
Course: AGR 323 - Mushroom and Molds
Credits: 3
Level: advanced
Prerequisites: ENG 101

Text Preview:
Department: Course Descriptions  Course Code: AGR 323  Course Title: Mushroom and Molds  Credits: 3  Prerequisites: ENG 101  Description: Prerequisite: ENG 101 An overview of how organisms in the Kingdom Fungi (mushrooms, molds, yeasts, rusts, mildews) impact individuals and society. Content will in...

Page: 62

--------------------------------------------------------------------